# Phase 12 — Controlled Model Challenger Benchmark

- **12A.** Experiment Contract & Safety
- **12B.** Champion Reproduction
- **12C.** Benchmark Utilities
- **12D.** Historical Controls
- **12E.** Focused 13–72 XGBoost
- **12F.** Focused XGBoost Continuation Gate
- **12G.** HistGradientBoosting
- **12H.** Random Forest
- **12I.** Stage-1 Leaderboard
- **12J.** External Challenger Gate
- **12K.** Robustness Analysis
- **12L.** Validation Selection Freeze
- **12M.** One-Time Final Test
- **12N.** Final Model Decision
- **12O.** Production Impact Plan
- **12P.** Save Benchmark Artifacts

## **12A.** Experiment Contract & Safety Validation

Before training any challenger, this phase verifies that the benchmark is
operating on the exact artifacts and data contract used by the existing
production model.

This phase validates:

- required experiment artifacts
- Phase 2 and production feature-contract agreement
- ordered model features
- target and identifier columns
- train, validation, and test schemas
- frozen split row counts and timestamp boundaries
- complete 1–72 hour forecast-horizon coverage
- missing-value constraints
- duplicate reference/horizon constraints
- chronological and purge-boundary integrity
- production model metadata consistency
- local software environment and artifact fingerprints

The test split is accessed here only for structural contract validation.
No test predictions, target-based test metrics, or model-selection decisions
are performed in this phase.

No model is trained and no production artifact is modified.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
def find_project_root(
    start: Path | None = None,
) -> Path:
    """Locate the repository root from the current notebook environment."""

    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "data" / "training").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    raise RuntimeError(
        "Could not locate the Pearls AQI Predictor project root."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "training"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

print("Project root:", PROJECT_ROOT)
print("Training data:", DATA_DIR)
print("Models:", MODEL_DIR)
print("Reports:", REPORT_DIR)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Training data: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/training
Models: /home/riyan/Riyan/projects/pearls-aqi-predictor/models
Reports: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports


In [3]:
PHASE_2_FEATURE_CONTRACT_PATH = (
    DATA_DIR / "feature_columns.json"
)

PHASE_2_VALIDATION_REPORT_PATH = (
    DATA_DIR / "phase_2_validation_report.json"
)

TRAIN_DATASET_PATH = (
    DATA_DIR / "train_dataset.parquet"
)

VALIDATION_DATASET_PATH = (
    DATA_DIR / "validation_dataset.parquet"
)

TEST_DATASET_PATH = (
    DATA_DIR / "test_dataset.parquet"
)

PRODUCTION_MODEL_PATH = (
    MODEL_DIR / "best_model.joblib"
)

PRODUCTION_FEATURE_CONTRACT_PATH = (
    MODEL_DIR / "model_feature_columns.json"
)

PRODUCTION_METADATA_PATH = (
    MODEL_DIR / "model_metadata.json"
)

MODEL_SELECTION_REPORT_PATH = (
    MODEL_DIR / "model_selection_report.json"
)


REQUIRED_ARTIFACTS = {
    "phase_2_feature_contract": PHASE_2_FEATURE_CONTRACT_PATH,
    "phase_2_validation_report": PHASE_2_VALIDATION_REPORT_PATH,
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "production_model": PRODUCTION_MODEL_PATH,
    "production_feature_contract": PRODUCTION_FEATURE_CONTRACT_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
    "model_selection_report": MODEL_SELECTION_REPORT_PATH,
}

In [4]:
artifact_status_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
        }
        for name, path in REQUIRED_ARTIFACTS.items()
    ]
)

display(artifact_status_df)

missing_artifacts = [
    name
    for name, path in REQUIRED_ARTIFACTS.items()
    if not path.exists()
]

assert not missing_artifacts, (
    "Required benchmark artifacts are missing: "
    f"{missing_artifacts}"
)

print("All required benchmark artifacts exist.")

,artifact,path,exists
0,phase_2_feature_contract,data/training/feature_columns.json,True
1,phase_2_validation_report,data/training/phase_2_validation_report.json,True
2,train_dataset,data/training/train_dataset.parquet,True
3,validation_dataset,data/training/validation_dataset.parquet,True
4,test_dataset,data/training/test_dataset.parquet,True
5,production_model,models/best_model.joblib,True
6,production_feature_contract,models/model_feature_columns.json,True
7,production_metadata,models/model_metadata.json,True
8,model_selection_report,models/model_selection_report.json,True


All required benchmark artifacts exist.


In [5]:
def load_json_object(
    path: Path,
) -> dict[str, Any]:
    """Load one JSON artifact and require a JSON object."""

    payload = json.loads(
        path.read_text(encoding="utf-8")
    )

    if not isinstance(payload, dict):
        raise ValueError(
            f"{path} must contain a JSON object."
        )

    return payload


phase_2_feature_contract = load_json_object(
    PHASE_2_FEATURE_CONTRACT_PATH
)

phase_2_validation_report = load_json_object(
    PHASE_2_VALIDATION_REPORT_PATH
)

production_feature_contract = load_json_object(
    PRODUCTION_FEATURE_CONTRACT_PATH
)

production_metadata = load_json_object(
    PRODUCTION_METADATA_PATH
)

model_selection_report = load_json_object(
    MODEL_SELECTION_REPORT_PATH
)

print("Experiment metadata loaded successfully.")

Experiment metadata loaded successfully.


In [6]:
MODEL_FEATURE_COLUMNS = list(
    phase_2_feature_contract["feature_columns"]
)

TARGET_COLUMN = str(
    phase_2_feature_contract["target_column"]
)

IDENTIFIER_COLUMNS = list(
    phase_2_feature_contract["identifier_columns"]
)

FORECAST_HORIZON_COLUMN = (
    "forecast_horizon_hours"
)

EXPECTED_HORIZONS = list(
    range(
        int(
            phase_2_feature_contract[
                "forecast_horizon_min"
            ]
        ),
        int(
            phase_2_feature_contract[
                "forecast_horizon_max"
            ]
        )
        + 1,
    )
)


print("Model features:", len(MODEL_FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)
print("Identifiers:", IDENTIFIER_COLUMNS)
print(
    "Forecast horizons:",
    EXPECTED_HORIZONS[0],
    "to",
    EXPECTED_HORIZONS[-1],
)

Model features: 56
Target: target_pm25_ug_m3
Identifiers: ['reference_time', 'target_time']
Forecast horizons: 1 to 72


In [7]:
production_feature_columns = list(
    production_feature_contract["feature_columns"]
)

metadata_feature_columns = list(
    production_metadata["ordered_feature_names"]
)

assert MODEL_FEATURE_COLUMNS == production_feature_columns, (
    "Phase 2 and production feature contracts differ."
)

assert MODEL_FEATURE_COLUMNS == metadata_feature_columns, (
    "Phase 2 features and production metadata features differ."
)

assert len(MODEL_FEATURE_COLUMNS) == 56

assert (
    production_feature_contract["feature_count"]
    == 56
)

assert (
    production_metadata["input_feature_count"]
    == 56
)

assert (
    TARGET_COLUMN
    == production_feature_contract["target_column"]
    == production_metadata["target_column"]
)

assert (
    IDENTIFIER_COLUMNS
    == production_feature_contract["identifier_columns"]
)

assert (
    FORECAST_HORIZON_COLUMN
    in MODEL_FEATURE_COLUMNS
)

assert TARGET_COLUMN not in MODEL_FEATURE_COLUMNS

for identifier_column in IDENTIFIER_COLUMNS:
    assert identifier_column not in MODEL_FEATURE_COLUMNS


future_pm25_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column.startswith("target_pm25")
]

assert not future_pm25_features, (
    "Future PM2.5-derived model inputs detected: "
    f"{future_pm25_features}"
)

print("Feature-contract agreement validated.")

Feature-contract agreement validated.


In [8]:
train_df = pd.read_parquet(
    TRAIN_DATASET_PATH
)

validation_df = pd.read_parquet(
    VALIDATION_DATASET_PATH
)

test_df = pd.read_parquet(
    TEST_DATASET_PATH
)


dataset_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "columns": len(train_df.columns),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "columns": len(validation_df.columns),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "columns": len(test_df.columns),
        },
    ]
)

display(dataset_summary_df)

,split,rows,columns
0,train,364798,59
1,validation,71256,59
2,test,76813,59


In [9]:
expected_splits = (
    phase_2_validation_report["splits"]
)

assert len(train_df) == int(
    expected_splits["train"]["rows"]
)

assert len(validation_df) == int(
    expected_splits["validation"]["rows"]
)

assert len(test_df) == int(
    expected_splits["test"]["rows"]
)


assert (
    train_df.columns.tolist()
    == validation_df.columns.tolist()
    == test_df.columns.tolist()
)

assert train_df.dtypes.equals(
    validation_df.dtypes
)

assert train_df.dtypes.equals(
    test_df.dtypes
)


required_columns = {
    *MODEL_FEATURE_COLUMNS,
    TARGET_COLUMN,
    *IDENTIFIER_COLUMNS,
}

missing_columns = sorted(
    required_columns.difference(
        train_df.columns
    )
)

assert not missing_columns, (
    f"Training schema is missing columns: {missing_columns}"
)

print("Frozen split sizes and schemas validated.")

Frozen split sizes and schemas validated.


In [10]:
def normalize_timestamp_columns(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    """Return a copy with identifier timestamps normalized to UTC."""

    result = dataframe.copy()

    for column in IDENTIFIER_COLUMNS:
        result[column] = pd.to_datetime(
            result[column],
            utc=True,
            errors="raise",
        )

    return result


train_checked_df = normalize_timestamp_columns(
    train_df
)

validation_checked_df = normalize_timestamp_columns(
    validation_df
)

test_checked_df = normalize_timestamp_columns(
    test_df
)

print("Timestamp columns normalized for contract checks.")

Timestamp columns normalized for contract checks.


In [11]:
split_frames = {
    "train": train_checked_df,
    "validation": validation_checked_df,
    "test": test_checked_df,
}


integrity_records = []

for split_name, dataframe in split_frames.items():
    missing_features = int(
        dataframe[
            MODEL_FEATURE_COLUMNS
        ]
        .isna()
        .sum()
        .sum()
    )

    missing_targets = int(
        dataframe[
            TARGET_COLUMN
        ].isna().sum()
    )

    duplicate_keys = int(
        dataframe.duplicated(
            subset=[
                "reference_time",
                FORECAST_HORIZON_COLUMN,
            ]
        ).sum()
    )

    horizons = sorted(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    integrity_records.append(
        {
            "split": split_name,
            "missing_features": missing_features,
            "missing_targets": missing_targets,
            "duplicate_reference_horizon_keys": duplicate_keys,
            "horizon_min": min(horizons),
            "horizon_max": max(horizons),
            "unique_horizons": len(horizons),
        }
    )

    assert missing_features == 0
    assert missing_targets == 0
    assert duplicate_keys == 0
    assert horizons == EXPECTED_HORIZONS


integrity_df = pd.DataFrame(
    integrity_records
)

display(integrity_df)

print("Dataset integrity validation passed.")

,split,missing_features,missing_targets,duplicate_reference_horizon_keys,horizon_min,horizon_max,unique_horizons
0,train,0,0,0,1,72,72
1,validation,0,0,0,1,72,72
2,test,0,0,0,1,72,72


Dataset integrity validation passed.


In [12]:
split_boundary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "reference_start": (
                train_checked_df["reference_time"].min()
            ),
            "reference_end": (
                train_checked_df["reference_time"].max()
            ),
            "target_start": (
                train_checked_df["target_time"].min()
            ),
            "target_end": (
                train_checked_df["target_time"].max()
            ),
        },
        {
            "split": "validation",
            "reference_start": (
                validation_checked_df["reference_time"].min()
            ),
            "reference_end": (
                validation_checked_df["reference_time"].max()
            ),
            "target_start": (
                validation_checked_df["target_time"].min()
            ),
            "target_end": (
                validation_checked_df["target_time"].max()
            ),
        },
        {
            "split": "test",
            "reference_start": (
                test_checked_df["reference_time"].min()
            ),
            "reference_end": (
                test_checked_df["reference_time"].max()
            ),
            "target_start": (
                test_checked_df["target_time"].min()
            ),
            "target_end": (
                test_checked_df["target_time"].max()
            ),
        },
    ]
)

display(split_boundary_df)

,split,reference_start,reference_end,target_start,target_end
0,train,2025-07-09 00:00:00+00:00,2026-03-18 11:00:00+00:00,2025-07-09 01:00:00+00:00,2026-03-21 11:00:00+00:00
1,validation,2026-03-21 12:00:00+00:00,2026-05-27 21:00:00+00:00,2026-03-21 13:00:00+00:00,2026-05-30 21:00:00+00:00
2,test,2026-05-30 22:00:00+00:00,2026-07-23 22:00:00+00:00,2026-05-30 23:00:00+00:00,2026-07-23 23:00:00+00:00


In [13]:
for split_name, dataframe in split_frames.items():
    expected = expected_splits[split_name]

    assert (
        dataframe["reference_time"].min()
        == pd.Timestamp(
            expected["reference_start"]
        )
    )

    assert (
        dataframe["reference_time"].max()
        == pd.Timestamp(
            expected["reference_end"]
        )
    )

    assert (
        dataframe["target_time"].min()
        == pd.Timestamp(
            expected["target_start"]
        )
    )

    assert (
        dataframe["target_time"].max()
        == pd.Timestamp(
            expected["target_end"]
        )
    )


assert (
    train_checked_df["target_time"].max()
    < validation_checked_df["reference_time"].min()
)

assert (
    validation_checked_df["target_time"].max()
    < test_checked_df["reference_time"].min()
)


for dataframe in split_frames.values():
    assert (
        dataframe["target_time"]
        > dataframe["reference_time"]
    ).all()


print("Chronological split and purge boundaries validated.")

Chronological split and purge boundaries validated.


In [14]:
EXPECTED_STRATEGY = (
    "hybrid_persistence_1_12_"
    "xgboost_shallower_13_72"
)

PERSISTENCE_MAX_HORIZON = 12


assert (
    production_metadata["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    model_selection_report["selected_strategy"]
    == EXPECTED_STRATEGY
)

assert (
    production_metadata["model_name"]
    == "xgboost_shallower"
)

assert (
    model_selection_report["selected_model"]
    == "xgboost_shallower"
)

assert (
    int(
        production_metadata[
            "routing"
        ]["persistence_max_horizon"]
    )
    == PERSISTENCE_MAX_HORIZON
)

assert (
    production_metadata[
        "routing"
    ]["horizons_1_to_12"]
    == "current_pm25_persistence"
)

assert (
    production_metadata[
        "routing"
    ]["horizons_13_to_72"]
    == "xgboost_shallower"
)


print("Production strategy metadata validated.")

Production strategy metadata validated.


In [15]:
current_environment = {
    "python": platform.python_version(),
    "pandas": version("pandas"),
    "numpy": version("numpy"),
    "scikit_learn": version("scikit-learn"),
    "xgboost": version("xgboost"),
    "joblib": version("joblib"),
}


production_environment = (
    production_metadata.get(
        "software_versions",
        {}
    )
)


environment_comparison_df = pd.DataFrame(
    [
        {
            "package": package,
            "current": current_environment.get(
                package
            ),
            "production_training": (
                production_environment.get(
                    package
                )
            ),
        }
        for package in current_environment
    ]
)

display(environment_comparison_df)

,package,current,production_training
0,python,3.12.3,3.12.3
1,pandas,2.3.3,3.0.5
2,numpy,2.2.6,2.5.1
3,scikit_learn,1.9.0,1.9.0
4,xgboost,3.3.0,3.3.0
5,joblib,1.5.3,None


In [16]:
def calculate_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    """Calculate the SHA-256 checksum of a local artifact."""

    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()


fingerprint_paths = {
    "train_dataset": TRAIN_DATASET_PATH,
    "validation_dataset": VALIDATION_DATASET_PATH,
    "test_dataset": TEST_DATASET_PATH,
    "phase_2_feature_contract": (
        PHASE_2_FEATURE_CONTRACT_PATH
    ),
    "production_feature_contract": (
        PRODUCTION_FEATURE_CONTRACT_PATH
    ),
    "production_model": PRODUCTION_MODEL_PATH,
    "production_metadata": PRODUCTION_METADATA_PATH,
}


artifact_fingerprints = {
    name: calculate_sha256(path)
    for name, path in fingerprint_paths.items()
}


fingerprint_df = pd.DataFrame(
    [
        {
            "artifact": name,
            "sha256": checksum,
        }
        for name, checksum
        in artifact_fingerprints.items()
    ]
)

display(fingerprint_df)

,artifact,sha256
0,train_dataset,aa8bd58e0a1dc3dbc7e35c1dd4216d5400da22ef26d656...
1,validation_dataset,b30b259d1daeec75386a0b9dfb81fa8d90deaaa22d86e3...
2,test_dataset,0359ff647d1bfa3f021ca8e0ae6a2313a4831addb09a5d...
3,phase_2_feature_contract,21260e0ab94d9196ff5fa4d782517cf113d61f8eb578c2...
4,production_feature_contract,613b785c93a2c4135416906cb4a1c09b402c53df68674e...
5,production_model,9d1cbb032a6f376892c2ad047bbf93bf485d18b21b47d1...
6,production_metadata,11a398dec9abdc619d25db46b3065ae400db4cfb9df49b...


In [17]:
contract_validation_summary = {
    "status": "PASSED",
    "feature_count": len(
        MODEL_FEATURE_COLUMNS
    ),
    "target_column": TARGET_COLUMN,
    "identifier_columns": (
        IDENTIFIER_COLUMNS
    ),
    "forecast_horizon_min": min(
        EXPECTED_HORIZONS
    ),
    "forecast_horizon_max": max(
        EXPECTED_HORIZONS
    ),
    "train_rows": len(train_df),
    "validation_rows": len(
        validation_df
    ),
    "test_rows": len(test_df),
    "selected_strategy": (
        production_metadata[
            "selected_strategy"
        ]
    ),
    "production_model_name": (
        production_metadata["model_name"]
    ),
    "persistence_max_horizon": (
        PERSISTENCE_MAX_HORIZON
    ),
}


display(
    pd.Series(
        contract_validation_summary,
        name="value",
    ).to_frame()
)

print(
    "Phase 12A PASSED — frozen experiment "
    "contract is internally consistent."
)

,value
status,PASSED
feature_count,56
target_column,target_pm25_ug_m3
identifier_columns,"[reference_time, target_time]"
forecast_horizon_min,1
forecast_horizon_max,72
train_rows,364798
validation_rows,71256
test_rows,76813
selected_strategy,hybrid_persistence_1_12_xgboost_shallower_13_72


Phase 12A PASSED — frozen experiment contract is internally consistent.


## **12B.** Production Champion Reproduction

Before introducing any challenger, this phase independently reproduces the
existing production champion on the frozen validation split.

The current production strategy is:

- horizons 1–12: current-value PM2.5 persistence
- horizons 13–72: `xgboost_shallower`

The saved production model is loaded without retraining. Predictions are
generated against the existing ordered feature contract, and the resulting
hybrid validation metrics are compared with the immutable metrics stored in
the production model metadata.

This is a hard benchmark gate.

If the saved validation metrics cannot be reproduced within a small numerical
tolerance, challenger development stops until the discrepancy is understood.

The historical test split remains excluded from performance evaluation.

In [18]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


HORIZON_GROUPS = {
    "1-6h": (1, 6),
    "7-12h": (7, 12),
    "13-24h": (13, 24),
    "25-48h": (25, 48),
    "49-72h": (49, 72),
}


def calculate_regression_metrics(
    y_true: pd.Series | np.ndarray,
    y_pred: pd.Series | np.ndarray,
) -> dict[str, float]:
    """Calculate standard PM2.5 regression metrics."""

    actual = np.asarray(
        y_true,
        dtype="float64",
    )

    predicted = np.asarray(
        y_pred,
        dtype="float64",
    )

    return {
        "mae": float(
            mean_absolute_error(
                actual,
                predicted,
            )
        ),
        "rmse": float(
            np.sqrt(
                mean_squared_error(
                    actual,
                    predicted,
                )
            )
        ),
        "r2": float(
            r2_score(
                actual,
                predicted,
            )
        ),
    }

In [19]:
production_model = joblib.load(
    PRODUCTION_MODEL_PATH
)


model_feature_count = getattr(
    production_model,
    "n_features_in_",
    None,
)


assert model_feature_count == len(
    MODEL_FEATURE_COLUMNS
), (
    "Saved production model feature count "
    "does not match the frozen contract."
)


loaded_model_type = type(
    production_model
).__name__


print(
    "Loaded model type:",
    loaded_model_type,
)

print(
    "Expected model type:",
    production_metadata["model_type"],
)

print(
    "Feature count:",
    model_feature_count,
)


assert (
    loaded_model_type
    == production_metadata["model_type"]
)

print("Saved production champion validated.")

Loaded model type: XGBRegressor
Expected model type: XGBRegressor
Feature count: 56
Saved production champion validated.


In [20]:
X_validation = validation_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_validation = validation_df[
    TARGET_COLUMN
].astype(float).copy()


assert (
    X_validation.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)

assert X_validation.isna().sum().sum() == 0

assert y_validation.isna().sum() == 0

assert np.isfinite(
    X_validation.to_numpy(
        dtype="float64"
    )
).all()

assert np.isfinite(
    y_validation.to_numpy(
        dtype="float64"
    )
).all()


validation_horizons = pd.to_numeric(
    validation_df[
        FORECAST_HORIZON_COLUMN
    ],
    errors="raise",
).astype(int)


print(
    "Validation rows:",
    len(validation_df),
)

print(
    "Validation horizon range:",
    validation_horizons.min(),
    "to",
    validation_horizons.max(),
)

Validation rows: 71256
Validation horizon range: 1 to 72


In [21]:
def generate_evaluation_hybrid_predictions(
    *,
    dataframe: pd.DataFrame,
    model: Any,
    feature_columns: list[str],
    persistence_max_horizon: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Generate production-equivalent hybrid predictions on a historical split.

    Returns both raw and operationally clipped predictions.
    """

    horizons = pd.to_numeric(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ],
        errors="raise",
    )

    raw_predictions = np.empty(
        len(dataframe),
        dtype="float64",
    )

    persistence_mask = (
        horizons
        .le(persistence_max_horizon)
        .to_numpy()
    )

    model_mask = ~persistence_mask

    raw_predictions[
        persistence_mask
    ] = (
        dataframe.loc[
            persistence_mask,
            "pm25_current",
        ]
        .astype(float)
        .to_numpy()
    )

    if model_mask.any():
        raw_predictions[
            model_mask
        ] = model.predict(
            dataframe.loc[
                model_mask,
                feature_columns,
            ]
        )

    if not np.isfinite(
        raw_predictions
    ).all():
        raise ValueError(
            "Hybrid predictions contain "
            "NaN or infinite values."
        )

    operational_predictions = np.clip(
        raw_predictions,
        a_min=0.0,
        a_max=None,
    )

    return (
        raw_predictions,
        operational_predictions,
    )


(
    champion_validation_predictions_raw,
    champion_validation_predictions,
) = generate_evaluation_hybrid_predictions(
    dataframe=validation_df,
    model=production_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [22]:
persistence_mask = (
    validation_horizons
    .le(PERSISTENCE_MAX_HORIZON)
    .to_numpy()
)

model_mask = ~persistence_mask


expected_persistence_values = (
    validation_df.loc[
        persistence_mask,
        "pm25_current",
    ]
    .astype(float)
    .to_numpy()
)


np.testing.assert_allclose(
    champion_validation_predictions_raw[
        persistence_mask
    ],
    expected_persistence_values,
    rtol=0.0,
    atol=0.0,
)


assert persistence_mask.sum() > 0
assert model_mask.sum() > 0


print(
    "Persistence rows:",
    int(persistence_mask.sum()),
)

print(
    "Learned-model rows:",
    int(model_mask.sum()),
)

print(
    "Negative raw predictions:",
    int(
        (
            champion_validation_predictions_raw
            < 0
        ).sum()
    ),
)

print("Hybrid routing validation passed.")

Persistence rows: 12319
Learned-model rows: 58937
Negative raw predictions: 0
Hybrid routing validation passed.


In [23]:
reproduced_validation_metrics = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            champion_validation_predictions
        ),
    )
)


expected_validation_metrics = {
    key: float(value)
    for key, value
    in production_metadata[
        "validation_metrics"
    ].items()
    if key in {
        "mae",
        "rmse",
        "r2",
    }
}


validation_metric_comparison_df = pd.DataFrame(
    [
        {
            "metric": metric,
            "expected": (
                expected_validation_metrics[
                    metric
                ]
            ),
            "reproduced": (
                reproduced_validation_metrics[
                    metric
                ]
            ),
            "absolute_difference": abs(
                reproduced_validation_metrics[
                    metric
                ]
                - expected_validation_metrics[
                    metric
                ]
            ),
        }
        for metric in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    validation_metric_comparison_df
)

,metric,expected,reproduced,absolute_difference
0,mae,6.695642,6.695642,2.664535e-15
1,rmse,9.419553,9.419553,0.000000e+00
2,r2,0.066611,0.066611,1.110223e-16


In [24]:
METRIC_ABSOLUTE_TOLERANCE = 1e-6


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        reproduced_validation_metrics[
            metric_name
        ],
        expected_validation_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
        err_msg=(
            "Production validation metric "
            f"could not be reproduced: "
            f"{metric_name}"
        ),
    )


print(
    "Production champion validation metrics "
    "reproduced successfully."
)

Production champion validation metrics reproduced successfully.


In [25]:
champion_validation_results_df = (
    validation_df[
        [
            "reference_time",
            "target_time",
            FORECAST_HORIZON_COLUMN,
            TARGET_COLUMN,
        ]
    ]
    .copy()
)


champion_validation_results_df[
    "prediction"
] = champion_validation_predictions


champion_horizon_group_records = []


for group_name, (
    minimum_horizon,
    maximum_horizon,
) in HORIZON_GROUPS.items():

    group_mask = (
        champion_validation_results_df[
            FORECAST_HORIZON_COLUMN
        ]
        .between(
            minimum_horizon,
            maximum_horizon,
        )
    )

    group_df = (
        champion_validation_results_df.loc[
            group_mask
        ]
    )

    group_metrics = (
        calculate_regression_metrics(
            y_true=group_df[
                TARGET_COLUMN
            ],
            y_pred=group_df[
                "prediction"
            ],
        )
    )

    champion_horizon_group_records.append(
        {
            "model": EXPECTED_STRATEGY,
            "horizon_group": group_name,
            "rows": len(group_df),
            **group_metrics,
        }
    )


champion_validation_by_horizon_group_df = (
    pd.DataFrame(
        champion_horizon_group_records
    )
)


display(
    champion_validation_by_horizon_group_df
)

,model,horizon_group,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1-6h,6190,3.648142,6.990142,0.516136
1,hybrid_persistence_1_12_xgboost_shallower_13_72,7-12h,6129,5.131832,8.653139,0.268671
2,hybrid_persistence_1_12_xgboost_shallower_13_72,13-24h,12163,7.117727,9.692964,0.091642
3,hybrid_persistence_1_12_xgboost_shallower_13_72,25-48h,23709,7.168849,9.669828,-0.050194
4,hybrid_persistence_1_12_xgboost_shallower_13_72,49-72h,23065,7.220054,9.769384,-0.031042


In [26]:
champion_per_horizon_records = []


for horizon in EXPECTED_HORIZONS:
    horizon_mask = (
        champion_validation_results_df[
            FORECAST_HORIZON_COLUMN
        ]
        .eq(horizon)
    )

    horizon_df = (
        champion_validation_results_df.loc[
            horizon_mask
        ]
    )

    horizon_metrics = (
        calculate_regression_metrics(
            y_true=horizon_df[
                TARGET_COLUMN
            ],
            y_pred=horizon_df[
                "prediction"
            ],
        )
    )

    champion_per_horizon_records.append(
        {
            "model": EXPECTED_STRATEGY,
            "forecast_horizon_hours": horizon,
            "rows": len(horizon_df),
            **horizon_metrics,
        }
    )


champion_validation_by_horizon_df = (
    pd.DataFrame(
        champion_per_horizon_records
    )
)


assert len(
    champion_validation_by_horizon_df
) == 72


display(
    champion_validation_by_horizon_df.head(
        12
    )
)

,model,forecast_horizon_hours,rows,mae,rmse,r2
0,hybrid_persistence_1_12_xgboost_shallower_13_72,1,1039,2.061886,4.209370,0.822803
1,hybrid_persistence_1_12_xgboost_shallower_13_72,2,1035,3.007246,5.920939,0.651224
2,hybrid_persistence_1_12_xgboost_shallower_13_72,3,1032,3.668605,7.005961,0.512925
3,hybrid_persistence_1_12_xgboost_shallower_13_72,4,1030,4.047670,7.627872,0.424880
4,hybrid_persistence_1_12_xgboost_shallower_13_72,5,1028,4.456031,8.173541,0.342357
5,hybrid_persistence_1_12_xgboost_shallower_13_72,6,1026,4.669883,8.170961,0.344593
6,hybrid_persistence_1_12_xgboost_shallower_13_72,7,1025,4.850439,8.232904,0.336435
7,hybrid_persistence_1_12_xgboost_shallower_13_72,8,1024,4.983105,8.450385,0.302254
8,hybrid_persistence_1_12_xgboost_shallower_13_72,9,1022,5.090802,8.522998,0.290635
9,hybrid_persistence_1_12_xgboost_shallower_13_72,10,1020,5.118431,8.521727,0.289935


In [27]:
learned_horizon_mask = (
    validation_horizons
    .gt(PERSISTENCE_MAX_HORIZON)
    .to_numpy()
)


champion_learned_horizon_metrics = (
    calculate_regression_metrics(
        y_true=(
            y_validation.to_numpy()[
                learned_horizon_mask
            ]
        ),
        y_pred=(
            champion_validation_predictions[
                learned_horizon_mask
            ]
        ),
    )
)


display(
    pd.Series(
        champion_learned_horizon_metrics,
        name="13-72h champion",
    ).to_frame()
)

,13-72h champion
mae,7.178338
rmse,9.713671
r2,-0.009030


In [28]:
champion_reproduction_summary = {
    "status": "PASSED",
    "model_type": (
        loaded_model_type
    ),
    "model_name": (
        production_metadata[
            "model_name"
        ]
    ),
    "strategy": (
        production_metadata[
            "selected_strategy"
        ]
    ),
    "persistence_max_horizon": (
        PERSISTENCE_MAX_HORIZON
    ),
    "validation_rows": len(
        validation_df
    ),
    "validation_mae": (
        reproduced_validation_metrics[
            "mae"
        ]
    ),
    "validation_rmse": (
        reproduced_validation_metrics[
            "rmse"
        ]
    ),
    "validation_r2": (
        reproduced_validation_metrics[
            "r2"
        ]
    ),
    "negative_raw_predictions": int(
        (
            champion_validation_predictions_raw
            < 0
        ).sum()
    ),
}


display(
    pd.Series(
        champion_reproduction_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12B PASSED — the existing "
    "production champion has been reproduced."
)

,value
status,PASSED
model_type,XGBRegressor
model_name,xgboost_shallower
strategy,hybrid_persistence_1_12_xgboost_shallower_13_72
persistence_max_horizon,12
validation_rows,71256
validation_mae,6.695642
validation_rmse,9.419553
validation_r2,0.066611
negative_raw_predictions,0


Phase 12B PASSED — the existing production champion has been reproduced.


## **12C.** Standardized Benchmark Utilities

The production champion is now reproducible on the frozen validation split.

Before training any challenger, this phase defines a single evaluation contract
that will be applied consistently to every candidate model.

The utilities created here standardize:

- hybrid persistence/model prediction generation
- overall validation metrics
- learned-horizon metrics for hours 13–72
- horizon-group metrics
- individual forecast-horizon metrics
- severe-PM2.5 performance
- prediction validity and clipping behavior
- training and prediction runtime reporting
- serialized model size
- candidate result packaging

The utilities are estimator-neutral. They operate on a fitted estimator through
the standard `.predict()` interface and do not contain model-family-specific
training logic.

No challenger model is trained in this phase.

The historical test split remains excluded from performance evaluation.

In [29]:
from time import perf_counter
from tempfile import TemporaryDirectory

In [30]:
LEARNED_HORIZON_MIN = (
    PERSISTENCE_MAX_HORIZON + 1
)

LEARNED_HORIZON_MAX = max(
    EXPECTED_HORIZONS
)

SEVERE_PM25_THRESHOLD = 55.5


print(
    "Learned-model horizon range:",
    LEARNED_HORIZON_MIN,
    "to",
    LEARNED_HORIZON_MAX,
)

print(
    "Severe PM2.5 threshold:",
    SEVERE_PM25_THRESHOLD,
    "µg/m³",
)

Learned-model horizon range: 13 to 72
Severe PM2.5 threshold: 55.5 µg/m³


In [31]:
def calculate_horizon_group_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
    model_name: str,
) -> pd.DataFrame:
    """Calculate regression metrics for each configured horizon group."""

    records: list[dict[str, Any]] = []

    for group_name, (
        minimum_horizon,
        maximum_horizon,
    ) in HORIZON_GROUPS.items():
        group_df = dataframe.loc[
            dataframe[
                FORECAST_HORIZON_COLUMN
            ].between(
                minimum_horizon,
                maximum_horizon,
            )
        ]

        if group_df.empty:
            raise ValueError(
                "No rows are available for "
                f"horizon group {group_name}."
            )

        metrics = calculate_regression_metrics(
            y_true=group_df[TARGET_COLUMN],
            y_pred=group_df[prediction_column],
        )

        records.append(
            {
                "model": model_name,
                "horizon_group": group_name,
                "horizon_min": minimum_horizon,
                "horizon_max": maximum_horizon,
                "rows": len(group_df),
                **metrics,
            }
        )

    result = pd.DataFrame(records)

    assert len(result) == len(
        HORIZON_GROUPS
    )

    return result

In [32]:
def calculate_per_horizon_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
    model_name: str,
) -> pd.DataFrame:
    """Calculate regression metrics independently for horizons 1 through 72."""

    records: list[dict[str, Any]] = []

    observed_horizons = sorted(
        dataframe[
            FORECAST_HORIZON_COLUMN
        ]
        .astype(int)
        .unique()
        .tolist()
    )

    if observed_horizons != EXPECTED_HORIZONS:
        raise ValueError(
            "Evaluation data does not contain "
            "the complete expected horizon range."
        )

    for horizon in EXPECTED_HORIZONS:
        horizon_df = dataframe.loc[
            dataframe[
                FORECAST_HORIZON_COLUMN
            ].eq(horizon)
        ]

        if horizon_df.empty:
            raise ValueError(
                f"No rows found for horizon {horizon}."
            )

        metrics = calculate_regression_metrics(
            y_true=horizon_df[TARGET_COLUMN],
            y_pred=horizon_df[prediction_column],
        )

        records.append(
            {
                "model": model_name,
                "forecast_horizon_hours": horizon,
                "rows": len(horizon_df),
                **metrics,
            }
        )

    result = pd.DataFrame(records)

    assert len(result) == 72

    return result

In [33]:
def calculate_learned_horizon_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
) -> dict[str, float]:
    """Calculate metrics only where the learned estimator is active."""

    learned_df = dataframe.loc[
        dataframe[
            FORECAST_HORIZON_COLUMN
        ].between(
            LEARNED_HORIZON_MIN,
            LEARNED_HORIZON_MAX,
        )
    ]

    if learned_df.empty:
        raise ValueError(
            "No learned-horizon evaluation rows are available."
        )

    return calculate_regression_metrics(
        y_true=learned_df[TARGET_COLUMN],
        y_pred=learned_df[prediction_column],
    )

In [34]:
def calculate_severe_pm25_metrics(
    *,
    dataframe: pd.DataFrame,
    prediction_column: str,
) -> dict[str, Any]:
    """Evaluate predictions for observations at or above the severe threshold."""

    severe_df = dataframe.loc[
        dataframe[
            TARGET_COLUMN
        ].ge(SEVERE_PM25_THRESHOLD)
    ]

    sample_count = len(
        severe_df
    )

    if sample_count == 0:
        return {
            "threshold_ug_m3": (
                SEVERE_PM25_THRESHOLD
            ),
            "sample_count": 0,
            "metrics": None,
        }

    metrics = calculate_regression_metrics(
        y_true=severe_df[TARGET_COLUMN],
        y_pred=severe_df[prediction_column],
    )

    return {
        "threshold_ug_m3": (
            SEVERE_PM25_THRESHOLD
        ),
        "sample_count": (
            sample_count
        ),
        "metrics": metrics,
    }

In [35]:
def summarize_prediction_validity(
    *,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
) -> dict[str, Any]:
    """Summarize numerical validity and PM2.5 clipping behavior."""

    raw = np.asarray(
        raw_predictions,
        dtype="float64",
    )

    operational = np.asarray(
        operational_predictions,
        dtype="float64",
    )

    if raw.shape != operational.shape:
        raise ValueError(
            "Raw and operational predictions "
            "have different shapes."
        )

    raw_nan_count = int(
        np.isnan(raw).sum()
    )

    raw_inf_count = int(
        np.isinf(raw).sum()
    )

    negative_raw_count = int(
        (raw < 0).sum()
    )

    clipped_count = int(
        (
            ~np.isclose(
                raw,
                operational,
                rtol=0.0,
                atol=0.0,
            )
        ).sum()
    )

    return {
        "prediction_count": int(
            len(raw)
        ),
        "raw_nan_count": (
            raw_nan_count
        ),
        "raw_inf_count": (
            raw_inf_count
        ),
        "negative_raw_count": (
            negative_raw_count
        ),
        "clipped_prediction_count": (
            clipped_count
        ),
        "raw_min": (
            float(np.nanmin(raw))
            if raw_nan_count < len(raw)
            else None
        ),
        "raw_max": (
            float(np.nanmax(raw))
            if raw_nan_count < len(raw)
            else None
        ),
        "operational_min": (
            float(
                np.nanmin(
                    operational
                )
            )
            if len(operational)
            else None
        ),
        "operational_max": (
            float(
                np.nanmax(
                    operational
                )
            )
            if len(operational)
            else None
        ),
    }

In [36]:
def measure_serialized_model_size_bytes(
    model: Any,
) -> int:
    """Measure joblib artifact size without keeping a permanent file."""

    with TemporaryDirectory(
        prefix="pearls-benchmark-"
    ) as temporary_directory:
        model_path = (
            Path(temporary_directory)
            / "model.joblib"
        )

        joblib.dump(
            model,
            model_path,
        )

        return int(
            model_path.stat().st_size
        )

In [37]:
def generate_timed_hybrid_predictions(
    *,
    dataframe: pd.DataFrame,
    model: Any,
    feature_columns: list[str],
    persistence_max_horizon: int,
) -> tuple[np.ndarray, np.ndarray, float]:
    """Generate hybrid predictions and record prediction runtime."""

    prediction_start = perf_counter()

    (
        raw_predictions,
        operational_predictions,
    ) = generate_evaluation_hybrid_predictions(
        dataframe=dataframe,
        model=model,
        feature_columns=feature_columns,
        persistence_max_horizon=(
            persistence_max_horizon
        ),
    )

    prediction_seconds = (
        perf_counter()
        - prediction_start
    )

    return (
        raw_predictions,
        operational_predictions,
        float(prediction_seconds),
    )

In [38]:
def build_evaluation_frame(
    *,
    dataframe: pd.DataFrame,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
) -> pd.DataFrame:
    """Build one aligned evaluation table from a frozen split."""

    raw = np.asarray(
        raw_predictions,
        dtype="float64",
    )

    operational = np.asarray(
        operational_predictions,
        dtype="float64",
    )

    if (
        len(dataframe) != len(raw)
        or len(dataframe) != len(
            operational
        )
    ):
        raise ValueError(
            "Prediction lengths do not match "
            "the evaluation dataframe."
        )

    result = dataframe[
        [
            "reference_time",
            "target_time",
            FORECAST_HORIZON_COLUMN,
            TARGET_COLUMN,
            "pm25_current",
        ]
    ].copy()

    result[
        "prediction_raw"
    ] = raw

    result[
        "prediction"
    ] = operational

    result[
        "prediction_was_clipped"
    ] = ~np.isclose(
        raw,
        operational,
        rtol=0.0,
        atol=0.0,
    )

    return result

In [39]:
def evaluate_candidate_predictions(
    *,
    candidate_name: str,
    model_family: str,
    training_scope: str,
    dataframe: pd.DataFrame,
    raw_predictions: np.ndarray,
    operational_predictions: np.ndarray,
    training_seconds: float | None,
    prediction_seconds: float,
    model_size_bytes: int | None,
    parameters: dict[str, Any] | None = None,
) -> dict[str, Any]:
    """Evaluate one fitted candidate under the standard benchmark contract."""

    evaluation_df = build_evaluation_frame(
        dataframe=dataframe,
        raw_predictions=raw_predictions,
        operational_predictions=(
            operational_predictions
        ),
    )

    overall_metrics = (
        calculate_regression_metrics(
            y_true=evaluation_df[
                TARGET_COLUMN
            ],
            y_pred=evaluation_df[
                "prediction"
            ],
        )
    )

    learned_metrics = (
        calculate_learned_horizon_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
        )
    )

    horizon_group_metrics = (
        calculate_horizon_group_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
            model_name=candidate_name,
        )
    )

    per_horizon_metrics = (
        calculate_per_horizon_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
            model_name=candidate_name,
        )
    )

    severe_pm25 = (
        calculate_severe_pm25_metrics(
            dataframe=evaluation_df,
            prediction_column="prediction",
        )
    )

    prediction_validity = (
        summarize_prediction_validity(
            raw_predictions=(
                raw_predictions
            ),
            operational_predictions=(
                operational_predictions
            ),
        )
    )

    return {
        "candidate_name": (
            candidate_name
        ),
        "model_family": (
            model_family
        ),
        "training_scope": (
            training_scope
        ),
        "parameters": (
            parameters or {}
        ),
        "training_seconds": (
            None
            if training_seconds is None
            else float(training_seconds)
        ),
        "prediction_seconds": float(
            prediction_seconds
        ),
        "model_size_bytes": (
            model_size_bytes
        ),
        "model_size_mib": (
            None
            if model_size_bytes is None
            else float(
                model_size_bytes
                / (1024 ** 2)
            )
        ),
        "overall_metrics": (
            overall_metrics
        ),
        "learned_horizon_metrics": (
            learned_metrics
        ),
        "horizon_group_metrics": (
            horizon_group_metrics
        ),
        "per_horizon_metrics": (
            per_horizon_metrics
        ),
        "severe_pm25": (
            severe_pm25
        ),
        "prediction_validity": (
            prediction_validity
        ),
        "evaluation_frame": (
            evaluation_df
        ),
    }

In [40]:
def build_leaderboard_row(
    result: dict[str, Any],
) -> dict[str, Any]:
    """Flatten headline candidate metrics into one leaderboard row."""

    overall = result[
        "overall_metrics"
    ]

    learned = result[
        "learned_horizon_metrics"
    ]

    validity = result[
        "prediction_validity"
    ]

    return {
        "candidate": (
            result["candidate_name"]
        ),
        "model_family": (
            result["model_family"]
        ),
        "training_scope": (
            result["training_scope"]
        ),
        "hybrid_mae": (
            overall["mae"]
        ),
        "hybrid_rmse": (
            overall["rmse"]
        ),
        "hybrid_r2": (
            overall["r2"]
        ),
        "learned_13_72_mae": (
            learned["mae"]
        ),
        "learned_13_72_rmse": (
            learned["rmse"]
        ),
        "learned_13_72_r2": (
            learned["r2"]
        ),
        "negative_raw_predictions": (
            validity[
                "negative_raw_count"
            ]
        ),
        "clipped_predictions": (
            validity[
                "clipped_prediction_count"
            ]
        ),
        "training_seconds": (
            result[
                "training_seconds"
            ]
        ),
        "prediction_seconds": (
            result[
                "prediction_seconds"
            ]
        ),
        "model_size_mib": (
            result[
                "model_size_mib"
            ]
        ),
    }

In [41]:
def calculate_improvement_percent(
    *,
    champion_value: float,
    candidate_value: float,
) -> float:
    """Return positive percentage when a lower-is-better metric improves."""

    if champion_value == 0:
        raise ValueError(
            "Champion metric must be non-zero "
            "for percentage comparison."
        )

    return float(
        (
            champion_value
            - candidate_value
        )
        / abs(champion_value)
        * 100.0
    )

In [42]:
benchmark_results: dict[
    str,
    dict[str, Any],
] = {}

In [43]:
(
    champion_raw_12c,
    champion_predictions_12c,
    champion_prediction_seconds_12c,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=production_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


champion_model_size_bytes = (
    measure_serialized_model_size_bytes(
        production_model
    )
)


champion_benchmark_result = (
    evaluate_candidate_predictions(
        candidate_name="production_champion_v1",
        model_family=loaded_model_type,
        training_scope=(
            "historical production artifact"
        ),
        dataframe=validation_df,
        raw_predictions=(
            champion_raw_12c
        ),
        operational_predictions=(
            champion_predictions_12c
        ),
        training_seconds=None,
        prediction_seconds=(
            champion_prediction_seconds_12c
        ),
        model_size_bytes=(
            champion_model_size_bytes
        ),
        parameters=(
            production_metadata.get(
                "model_parameters",
                {},
            )
        ),
    )
)


benchmark_results[
    "production_champion_v1"
] = champion_benchmark_result

In [44]:
utility_champion_metrics = (
    champion_benchmark_result[
        "overall_metrics"
    ]
)


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        utility_champion_metrics[
            metric_name
        ],
        reproduced_validation_metrics[
            metric_name
        ],
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
        err_msg=(
            "12C utility evaluation differs "
            "from the Phase 12B champion baseline."
        ),
    )


np.testing.assert_allclose(
    champion_raw_12c,
    champion_validation_predictions_raw,
    rtol=0.0,
    atol=1e-12,
)


np.testing.assert_allclose(
    champion_predictions_12c,
    champion_validation_predictions,
    rtol=0.0,
    atol=1e-12,
)


print(
    "12C benchmark utilities reproduce "
    "the Phase 12B champion baseline."
)

12C benchmark utilities reproduce the Phase 12B champion baseline.


In [45]:
champion_leaderboard_df = pd.DataFrame(
    [
        build_leaderboard_row(
            champion_benchmark_result
        )
    ]
)


display(
    champion_leaderboard_df
)

,candidate,model_family,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,learned_13_72_r2,negative_raw_predictions,clipped_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,-0.00903,0,0,None,0.394257,0.645863


In [46]:
display(
    champion_benchmark_result[
        "horizon_group_metrics"
    ]
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
4,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042


In [47]:
display(
    champion_benchmark_result[
        "per_horizon_metrics"
    ].head(12)
)

,model,forecast_horizon_hours,rows,mae,rmse,r2
0,production_champion_v1,1,1039,2.061886,4.209370,0.822803
1,production_champion_v1,2,1035,3.007246,5.920939,0.651224
2,production_champion_v1,3,1032,3.668605,7.005961,0.512925
3,production_champion_v1,4,1030,4.047670,7.627872,0.424880
4,production_champion_v1,5,1028,4.456031,8.173541,0.342357
5,production_champion_v1,6,1026,4.669883,8.170961,0.344593
6,production_champion_v1,7,1025,4.850439,8.232904,0.336435
7,production_champion_v1,8,1024,4.983105,8.450385,0.302254
8,production_champion_v1,9,1022,5.090802,8.522998,0.290635
9,production_champion_v1,10,1020,5.118431,8.521727,0.289935


In [48]:
display(
    pd.Series(
        champion_benchmark_result[
            "severe_pm25"
        ],
        name="value",
    ).to_frame()
)

,value
threshold_ug_m3,55.5
sample_count,181
metrics,"{'mae': 49.37182242461989, 'rmse': 50.15318038..."


In [49]:
champion_prediction_validity = (
    champion_benchmark_result[
        "prediction_validity"
    ]
)


display(
    pd.Series(
        champion_prediction_validity,
        name="value",
    ).to_frame()
)


assert (
    champion_prediction_validity[
        "raw_nan_count"
    ]
    == 0
)

assert (
    champion_prediction_validity[
        "raw_inf_count"
    ]
    == 0
)


print(
    "Champion prediction validity "
    "checks passed."
)

,value
prediction_count,71256.000000
raw_nan_count,0.000000
raw_inf_count,0.000000
negative_raw_count,0.000000
clipped_prediction_count,0.000000
raw_min,0.100000
raw_max,99.575546
operational_min,0.100000
operational_max,99.575546


Champion prediction validity checks passed.


In [50]:
benchmark_utility_summary = {
    "status": "PASSED",
    "registered_results": len(
        benchmark_results
    ),
    "champion_registered": (
        "production_champion_v1"
        in benchmark_results
    ),
    "overall_metrics_match_12b": True,
    "horizon_group_count": len(
        champion_benchmark_result[
            "horizon_group_metrics"
        ]
    ),
    "per_horizon_count": len(
        champion_benchmark_result[
            "per_horizon_metrics"
        ]
    ),
    "severe_threshold_ug_m3": (
        SEVERE_PM25_THRESHOLD
    ),
    "learned_horizon_min": (
        LEARNED_HORIZON_MIN
    ),
    "learned_horizon_max": (
        LEARNED_HORIZON_MAX
    ),
}


display(
    pd.Series(
        benchmark_utility_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12C PASSED — standardized "
    "benchmark utilities are ready."
)

,value
status,PASSED
registered_results,1
champion_registered,True
overall_metrics_match_12b,True
horizon_group_count,5
per_horizon_count,72
severe_threshold_ug_m3,55.5
learned_horizon_min,13
learned_horizon_max,72


Phase 12C PASSED — standardized benchmark utilities are ready.


## **12D.** Historical Control Reproduction

Before beginning new challenger experiments, this phase reproduces a focused
subset of the original Phase 3 model comparisons.

The purpose is not to reopen the original model-selection process or tune old
models. Instead, these controls verify that the current benchmark environment,
frozen datasets, feature contract, and metric utilities produce conclusions
consistent with the original experiment.

The historical controls are:

- current-value persistence
- previous-day persistence
- Ridge Regression with the original Phase 3 configuration
- Histogram Gradient Boosting with the original Phase 3 configuration
- the saved `xgboost_shallower` production estimator

Two evaluation views are kept separate:

1. **Historical raw-model reproduction**  
   Reproduces the way these baselines and estimators were originally measured
   across the full validation split.

2. **Current hybrid benchmark view**  
   For fitted learned estimators, applies the production routing contract:
   persistence for hours 1–12 and the estimator for hours 13–72.

No hyperparameter tuning is performed in this phase.

No candidate is selected for production, and the historical test split remains
excluded from performance evaluation.

In [51]:
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [52]:
X_train = train_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_train = train_df[
    TARGET_COLUMN
].astype(float).copy()


assert (
    X_train.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)

assert X_train.isna().sum().sum() == 0
assert y_train.isna().sum() == 0

assert np.isfinite(
    X_train.to_numpy(
        dtype="float64"
    )
).all()

assert np.isfinite(
    y_train.to_numpy(
        dtype="float64"
    )
).all()


print(
    "Training rows:",
    len(X_train),
)

print(
    "Training features:",
    X_train.shape[1],
)

print(
    "Validation rows:",
    len(X_validation),
)

Training rows: 364798
Training features: 56
Validation rows: 71256


In [53]:
historical_model_records = {
    str(record["model"]): record
    for record in model_selection_report[
        "models_evaluated"
    ]
}


historical_xgboost_records = {
    str(record["model"]): record
    for record in model_selection_report[
        "xgboost_configurations"
    ]
}


EXPECTED_HISTORICAL_METRICS = {
    "current_persistence": (
        historical_model_records[
            "current_persistence"
        ]
    ),
    "previous_day_persistence": (
        historical_model_records[
            "previous_day_persistence"
        ]
    ),
    "ridge": (
        historical_model_records[
            "ridge"
        ]
    ),
    "hist_gradient_boosting": (
        historical_model_records[
            "hist_gradient_boosting"
        ]
    ),
    "xgboost_shallower": (
        historical_xgboost_records[
            "xgboost_shallower"
        ]
    ),
}


expected_historical_metrics_df = (
    pd.DataFrame(
        [
            {
                "model": model_name,
                "mae": float(
                    metrics["mae"]
                ),
                "rmse": float(
                    metrics["rmse"]
                ),
                "r2": float(
                    metrics["r2"]
                ),
            }
            for model_name, metrics
            in EXPECTED_HISTORICAL_METRICS.items()
        ]
    )
)


display(
    expected_historical_metrics_df
)

,model,mae,rmse,r2
0,current_persistence,6.621121,10.506971,-0.161334
1,previous_day_persistence,7.942474,11.887120,-0.486467
2,ridge,9.779916,12.994571,-0.776340
3,hist_gradient_boosting,7.176154,9.827731,-0.016035
4,xgboost_shallower,7.080239,9.604153,0.029668


### **12D.1.** Persistence Controls

The two original persistence baselines are reproduced first.

Current-value persistence predicts every future PM2.5 value using the PM2.5
observed at the forecast reference time.

Previous-day persistence predicts using the PM2.5 value observed 24 hours
before the reference time.

These predictions require no model fitting and provide simple reference points
for the learned estimators.

In [54]:
current_persistence_predictions = (
    validation_df[
        "pm25_current"
    ]
    .astype(float)
    .to_numpy()
)

previous_day_persistence_predictions = (
    validation_df[
        "pm25_lag_24h"
    ]
    .astype(float)
    .to_numpy()
)


assert np.isfinite(
    current_persistence_predictions
).all()

assert np.isfinite(
    previous_day_persistence_predictions
).all()


current_persistence_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            current_persistence_predictions
        ),
    )
)

previous_day_persistence_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            previous_day_persistence_predictions
        ),
    )
)

In [55]:
persistence_reproduction_df = pd.DataFrame(
    [
        {
            "model": (
                "current_persistence"
            ),
            "expected_mae": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["mae"]
            ),
            "reproduced_mae": (
                current_persistence_metrics_12d[
                    "mae"
                ]
            ),
            "expected_rmse": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["rmse"]
            ),
            "reproduced_rmse": (
                current_persistence_metrics_12d[
                    "rmse"
                ]
            ),
            "expected_r2": float(
                EXPECTED_HISTORICAL_METRICS[
                    "current_persistence"
                ]["r2"]
            ),
            "reproduced_r2": (
                current_persistence_metrics_12d[
                    "r2"
                ]
            ),
        },
        {
            "model": (
                "previous_day_persistence"
            ),
            "expected_mae": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["mae"]
            ),
            "reproduced_mae": (
                previous_day_persistence_metrics_12d[
                    "mae"
                ]
            ),
            "expected_rmse": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["rmse"]
            ),
            "reproduced_rmse": (
                previous_day_persistence_metrics_12d[
                    "rmse"
                ]
            ),
            "expected_r2": float(
                EXPECTED_HISTORICAL_METRICS[
                    "previous_day_persistence"
                ]["r2"]
            ),
            "reproduced_r2": (
                previous_day_persistence_metrics_12d[
                    "r2"
                ]
            ),
        },
    ]
)


display(
    persistence_reproduction_df
)

,model,expected_mae,reproduced_mae,expected_rmse,reproduced_rmse,expected_r2,reproduced_r2
0,current_persistence,6.621121,6.621121,10.506971,10.506971,-0.161334,-0.161334
1,previous_day_persistence,7.942474,7.942474,11.887120,11.887120,-0.486467,-0.486467


In [56]:
for (
    model_name,
    reproduced_metrics,
) in (
    (
        "current_persistence",
        current_persistence_metrics_12d,
    ),
    (
        "previous_day_persistence",
        previous_day_persistence_metrics_12d,
    ),
):
    expected_metrics = (
        EXPECTED_HISTORICAL_METRICS[
            model_name
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        np.testing.assert_allclose(
            reproduced_metrics[
                metric_name
            ],
            float(
                expected_metrics[
                    metric_name
                ]
            ),
            rtol=0.0,
            atol=METRIC_ABSOLUTE_TOLERANCE,
            err_msg=(
                "Historical persistence "
                "metric did not reproduce: "
                f"{model_name}/{metric_name}"
            ),
        )


print(
    "Historical persistence controls "
    "reproduced successfully."
)

Historical persistence controls reproduced successfully.


### **12D.2.** Ridge Regression Control

The original Ridge Regression control is now reproduced using the same Phase 3
configuration.

The model is intentionally not tuned.

It uses:

1. `StandardScaler` fitted only on the training split.
2. `Ridge(alpha=1.0)` fitted on the scaled training features.

The full 1–72 hour training and validation datasets are used because the goal
of this subphase is to reproduce the historical Phase 3 experiment, not yet to
create a new horizon-specialized challenger.

In [57]:
HISTORICAL_RIDGE_ALPHA = 1.0


historical_ridge_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            Ridge(
                alpha=(
                    HISTORICAL_RIDGE_ALPHA
                ),
                solver="auto",
            ),
        ),
    ]
)


historical_ridge_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [58]:
ridge_training_start = perf_counter()

historical_ridge_model.fit(
    X_train,
    y_train,
)

ridge_training_seconds_12d = (
    perf_counter()
    - ridge_training_start
)


print(
    "Historical Ridge training time:",
    f"{ridge_training_seconds_12d:.3f}",
    "seconds",
)

Historical Ridge training time: 2.848 seconds


In [59]:
ridge_raw_prediction_start = (
    perf_counter()
)

ridge_raw_validation_predictions = (
    historical_ridge_model.predict(
        X_validation
    )
)

ridge_raw_prediction_seconds = (
    perf_counter()
    - ridge_raw_prediction_start
)


assert np.isfinite(
    ridge_raw_validation_predictions
).all()


ridge_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            ridge_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        ridge_historical_metrics_12d,
        name="reproduced_ridge",
    ).to_frame()
)

,reproduced_ridge
mae,9.779916
rmse,12.994571
r2,-0.776340


In [60]:
ridge_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "ridge"
    ]
)


ridge_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                ridge_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                ridge_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                ridge_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    ridge_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    ridge_reproduction_df
)

,metric,expected,reproduced,absolute_difference
0,mae,9.779916,9.779916,6.039613e-14
1,rmse,12.994571,12.994571,1.207923e-13
2,r2,-0.776340,-0.776340,3.330669e-14


In [61]:
for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        ridge_historical_metrics_12d[
            metric_name
        ],
        float(
            ridge_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=1e-5,
        err_msg=(
            "Historical Ridge metric "
            f"did not reproduce: {metric_name}"
        ),
    )


print(
    "Historical Ridge control "
    "reproduced successfully."
)

Historical Ridge control reproduced successfully.


### **12D.3.** Histogram Gradient Boosting Control

Histogram Gradient Boosting was the strongest non-XGBoost learned estimator in
the original Phase 3 comparison.

This subphase reproduces the original configuration without tuning it.

As with Ridge, the estimator is trained across the historical 1–72 hour
training rows because this is a reproduction control. Horizon-specialized
training will be introduced later as a genuinely new experiment.

In [62]:
HISTORICAL_RANDOM_SEED = 42


historical_histgb_model = (
    HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=False,
        random_state=(
            HISTORICAL_RANDOM_SEED
        ),
    )
)


historical_histgb_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",300
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",30
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",False
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0


In [63]:
histgb_training_start = (
    perf_counter()
)

historical_histgb_model.fit(
    X_train,
    y_train,
)

histgb_training_seconds_12d = (
    perf_counter()
    - histgb_training_start
)


print(
    "Historical HistGB training time:",
    f"{histgb_training_seconds_12d:.3f}",
    "seconds",
)

print(
    "Iterations completed:",
    historical_histgb_model.n_iter_,
)

Historical HistGB training time: 30.337 seconds
Iterations completed: 300


In [64]:
histgb_raw_prediction_start = (
    perf_counter()
)

histgb_raw_validation_predictions = (
    historical_histgb_model.predict(
        X_validation
    )
)

histgb_raw_prediction_seconds = (
    perf_counter()
    - histgb_raw_prediction_start
)


assert np.isfinite(
    histgb_raw_validation_predictions
).all()


histgb_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            histgb_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        histgb_historical_metrics_12d,
        name="reproduced_histgb",
    ).to_frame()
)

,reproduced_histgb
mae,7.176154
rmse,9.827731
r2,-0.016035


In [65]:
histgb_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "hist_gradient_boosting"
    ]
)


histgb_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                histgb_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                histgb_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                histgb_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    histgb_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    histgb_reproduction_df
)

,metric,expected,reproduced,absolute_difference
0,mae,7.176154,7.176154,1.243450e-14
1,rmse,9.827731,9.827731,8.881784e-15
2,r2,-0.016035,-0.016035,1.776357e-15


In [66]:
for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        histgb_historical_metrics_12d[
            metric_name
        ],
        float(
            histgb_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=1e-5,
        err_msg=(
            "Historical HistGB metric "
            f"did not reproduce: {metric_name}"
        ),
    )


print(
    "Historical HistGradientBoosting "
    "control reproduced successfully."
)

Historical HistGradientBoosting control reproduced successfully.


### **12D.4.** Saved XGBoost Raw-Model Control

The saved production estimator is now evaluated without hybrid persistence
routing.

This reproduces the historical learned-model view of
`xgboost_shallower` across the complete validation split.

The model is not retrained. The immutable production artifact loaded in Phase
12B is reused.

In [67]:
xgboost_raw_prediction_start = (
    perf_counter()
)

xgboost_raw_validation_predictions = (
    production_model.predict(
        X_validation
    )
)

xgboost_raw_prediction_seconds_12d = (
    perf_counter()
    - xgboost_raw_prediction_start
)


assert np.isfinite(
    xgboost_raw_validation_predictions
).all()


xgboost_historical_metrics_12d = (
    calculate_regression_metrics(
        y_true=y_validation,
        y_pred=(
            xgboost_raw_validation_predictions
        ),
    )
)


display(
    pd.Series(
        xgboost_historical_metrics_12d,
        name="saved_xgboost_shallower",
    ).to_frame()
)

,saved_xgboost_shallower
mae,7.080239
rmse,9.604153
r2,0.029668


In [68]:
xgboost_expected_metrics = (
    EXPECTED_HISTORICAL_METRICS[
        "xgboost_shallower"
    ]
)


xgboost_raw_reproduction_df = pd.DataFrame(
    [
        {
            "metric": metric_name,
            "expected": float(
                xgboost_expected_metrics[
                    metric_name
                ]
            ),
            "reproduced": (
                xgboost_historical_metrics_12d[
                    metric_name
                ]
            ),
            "absolute_difference": abs(
                xgboost_historical_metrics_12d[
                    metric_name
                ]
                - float(
                    xgboost_expected_metrics[
                        metric_name
                    ]
                )
            ),
        }
        for metric_name in (
            "mae",
            "rmse",
            "r2",
        )
    ]
)


display(
    xgboost_raw_reproduction_df
)


for metric_name in (
    "mae",
    "rmse",
    "r2",
):
    np.testing.assert_allclose(
        xgboost_historical_metrics_12d[
            metric_name
        ],
        float(
            xgboost_expected_metrics[
                metric_name
            ]
        ),
        rtol=0.0,
        atol=METRIC_ABSOLUTE_TOLERANCE,
    )


print(
    "Saved xgboost_shallower raw "
    "validation metrics reproduced."
)

,metric,expected,reproduced,absolute_difference
0,mae,7.080239,7.080239,0.000000e+00
1,rmse,9.604153,9.604153,0.000000e+00
2,r2,0.029668,0.029668,2.220446e-16


Saved xgboost_shallower raw validation metrics reproduced.


In [69]:
historical_control_comparison_df = (
    pd.DataFrame(
        [
            {
                "model": (
                    "xgboost_shallower"
                ),
                **xgboost_historical_metrics_12d,
            },
            {
                "model": (
                    "hist_gradient_boosting"
                ),
                **histgb_historical_metrics_12d,
            },
            {
                "model": (
                    "current_persistence"
                ),
                **current_persistence_metrics_12d,
            },
            {
                "model": (
                    "previous_day_persistence"
                ),
                **previous_day_persistence_metrics_12d,
            },
            {
                "model": "ridge",
                **ridge_historical_metrics_12d,
            },
        ]
    )
    .sort_values(
        "rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_control_comparison_df
)

,model,mae,rmse,r2
0,xgboost_shallower,7.080239,9.604153,0.029668
1,hist_gradient_boosting,7.176154,9.827731,-0.016035
2,current_persistence,6.621121,10.506971,-0.161334
3,previous_day_persistence,7.942474,11.887120,-0.486467
4,ridge,9.779916,12.994571,-0.776340


In [70]:
assert (
    historical_control_comparison_df[
        "model"
    ].tolist()
    == [
        "xgboost_shallower",
        "hist_gradient_boosting",
        "current_persistence",
        "previous_day_persistence",
        "ridge",
    ]
)


print(
    "Historical Phase 3 control ordering "
    "has been reproduced."
)

Historical Phase 3 control ordering has been reproduced.


### **12D.5.** Historical Models Under the Current Hybrid Strategy

The reproduced Ridge and Histogram Gradient Boosting estimators are now
evaluated through the current production routing architecture.

For each model:

- hours 1–12 use current-value persistence
- hours 13–72 use the fitted estimator

This does not make either estimator a new challenger. Their training remains
identical to the original Phase 3 configuration.

The purpose is to establish how those historical controls compare when judged
under the same hybrid contract that all later challengers will use.

In [71]:
(
    ridge_hybrid_raw_predictions,
    ridge_hybrid_predictions,
    ridge_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=historical_ridge_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


ridge_model_size_bytes = (
    measure_serialized_model_size_bytes(
        historical_ridge_model
    )
)


ridge_hybrid_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "historical_ridge_hybrid"
        ),
        model_family="Ridge",
        training_scope=(
            "historical 1-72h control"
        ),
        dataframe=validation_df,
        raw_predictions=(
            ridge_hybrid_raw_predictions
        ),
        operational_predictions=(
            ridge_hybrid_predictions
        ),
        training_seconds=(
            ridge_training_seconds_12d
        ),
        prediction_seconds=(
            ridge_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            ridge_model_size_bytes
        ),
        parameters={
            "alpha": (
                HISTORICAL_RIDGE_ALPHA
            ),
            "scaled": True,
        },
    )
)


benchmark_results[
    "historical_ridge_hybrid"
] = ridge_hybrid_result

In [72]:
(
    histgb_hybrid_raw_predictions,
    histgb_hybrid_predictions,
    histgb_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=historical_histgb_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


histgb_model_size_bytes = (
    measure_serialized_model_size_bytes(
        historical_histgb_model
    )
)


histgb_hybrid_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "historical_histgb_hybrid"
        ),
        model_family=(
            "HistGradientBoostingRegressor"
        ),
        training_scope=(
            "historical 1-72h control"
        ),
        dataframe=validation_df,
        raw_predictions=(
            histgb_hybrid_raw_predictions
        ),
        operational_predictions=(
            histgb_hybrid_predictions
        ),
        training_seconds=(
            histgb_training_seconds_12d
        ),
        prediction_seconds=(
            histgb_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            histgb_model_size_bytes
        ),
        parameters={
            "loss": "squared_error",
            "learning_rate": 0.05,
            "max_iter": 300,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 30,
            "l2_regularization": 1.0,
            "early_stopping": False,
            "random_state": 42,
        },
    )
)


benchmark_results[
    "historical_histgb_hybrid"
] = histgb_hybrid_result

In [73]:
historical_hybrid_control_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "historical_histgb_hybrid"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "historical_ridge_hybrid"
                ]
            ),
        ]
    )
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_hybrid_control_df
)

,candidate,model_family,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,learned_13_72_r2,negative_raw_predictions,clipped_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,-0.009030,0,0,NaN,0.394257,0.645863
1,historical_histgb_hybrid,HistGradientBoostingRegressor,historical 1-72h control,6.762337,9.579777,0.034588,7.258972,9.901305,-0.048389,0,0,30.336777,1.129033,1.096182
2,historical_ridge_hybrid,Ridge,historical 1-72h control,8.162262,11.310933,-0.345857,8.951510,11.906307,-0.515973,9848,9848,2.847719,0.077809,0.003968


In [74]:
champion_hybrid_rmse = (
    benchmark_results[
        "production_champion_v1"
    ][
        "overall_metrics"
    ]["rmse"]
)

champion_hybrid_mae = (
    benchmark_results[
        "production_champion_v1"
    ][
        "overall_metrics"
    ]["mae"]
)


historical_hybrid_control_df[
    "rmse_improvement_vs_champion_pct"
] = historical_hybrid_control_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


historical_hybrid_control_df[
    "mae_improvement_vs_champion_pct"
] = historical_hybrid_control_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


display(
    historical_hybrid_control_df[
        [
            "candidate",
            "model_family",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,model_family,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,XGBRegressor,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,0,NaN,0.394257,0.645863
1,historical_histgb_hybrid,HistGradientBoostingRegressor,6.762337,9.579777,0.034588,7.258972,9.901305,-1.700974,-0.996086,0,30.336777,1.129033,1.096182
2,historical_ridge_hybrid,Ridge,8.162262,11.310933,-0.345857,8.951510,11.906307,-20.079305,-21.904098,9848,2.847719,0.077809,0.003968


In [75]:
historical_horizon_group_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "production_champion_v1"
            ][
                "horizon_group_metrics"
            ],
            benchmark_results[
                "historical_histgb_hybrid"
            ][
                "horizon_group_metrics"
            ],
            benchmark_results[
                "historical_ridge_hybrid"
            ][
                "horizon_group_metrics"
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    historical_horizon_group_comparison_df
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,historical_histgb_hybrid,1-6h,1,6,6190,3.648142,6.990142,0.516136
2,historical_ridge_hybrid,1-6h,1,6,6190,3.648142,6.990142,0.516136
3,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
4,historical_histgb_hybrid,7-12h,7,12,6129,5.131832,8.653139,0.268671
5,historical_ridge_hybrid,7-12h,7,12,6129,5.131832,8.653139,0.268671
6,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
7,historical_histgb_hybrid,13-24h,13,24,12163,7.160805,9.813267,0.068954
8,historical_ridge_hybrid,13-24h,13,24,12163,8.831251,11.769248,-0.339187
9,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194


In [76]:
for group_name in (
    "1-6h",
    "7-12h",
):
    group_df = (
        historical_horizon_group_comparison_df.loc[
            historical_horizon_group_comparison_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        metric_values = (
            group_df[
                metric_name
            ]
            .astype(float)
            .to_numpy()
        )

        np.testing.assert_allclose(
            metric_values,
            np.repeat(
                metric_values[0],
                len(metric_values),
            ),
            rtol=0.0,
            atol=1e-12,
        )


print(
    "All hybrid controls are identical "
    "for persistence-routed horizons 1–12."
)

All hybrid controls are identical for persistence-routed horizons 1–12.


In [77]:
historical_control_summary = {
    "status": "PASSED",
    "historical_order_reproduced": True,
    "persistence_controls_reproduced": True,
    "ridge_control_reproduced": True,
    "histgb_control_reproduced": True,
    "xgboost_raw_control_reproduced": True,
    "hybrid_short_horizon_equivalence": True,
    "benchmark_result_count": len(
        benchmark_results
    ),
    "registered_hybrid_controls": [
        "production_champion_v1",
        "historical_histgb_hybrid",
        "historical_ridge_hybrid",
    ],
}


display(
    pd.Series(
        historical_control_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12D PASSED — historical "
    "controls are consistent with Phase 3."
)

,value
status,PASSED
historical_order_reproduced,True
persistence_controls_reproduced,True
ridge_control_reproduced,True
histgb_control_reproduced,True
xgboost_raw_control_reproduced,True
hybrid_short_horizon_equivalence,True
benchmark_result_count,3
registered_hybrid_controls,"[production_champion_v1, historical_histgb_hyb..."


Phase 12D PASSED — historical controls are consistent with Phase 3.


## **12E.** Focused 13–72 Hour XGBoost Challenger

The historical benchmark controls are now reproduced successfully.

This phase introduces the first genuinely new challenger.

The current production estimator was originally developed across the full
1–72 hour forecasting range, but the final production strategy uses the learned
model only for horizons 13–72.

This experiment therefore asks:

> Can the existing XGBoost model family perform better if training is aligned
> directly with the horizons it is responsible for in production?

To isolate that effect, this challenger keeps the current production
`xgboost_shallower` hyperparameters unchanged.

The only intentional change is the training scope:

- production champion: historically trained across horizons 1–72
- focused challenger: trained only across horizons 13–72

The validation split is filtered to the same learned-model horizon range for
XGBoost early stopping.

Final candidate evaluation still uses the complete production hybrid strategy:

- hours 1–12: current-value persistence
- hours 13–72: focused XGBoost challenger

No hyperparameter tuning is performed in this phase.

The historical test split remains excluded from performance evaluation.

In [78]:
from xgboost import XGBRegressor

In [79]:
focused_train_df = train_df.loc[
    train_df[
        FORECAST_HORIZON_COLUMN
    ].between(
        LEARNED_HORIZON_MIN,
        LEARNED_HORIZON_MAX,
    )
].copy()


focused_validation_df = validation_df.loc[
    validation_df[
        FORECAST_HORIZON_COLUMN
    ].between(
        LEARNED_HORIZON_MIN,
        LEARNED_HORIZON_MAX,
    )
].copy()


print(
    "Focused training rows:",
    len(focused_train_df),
)

print(
    "Focused validation rows:",
    len(focused_validation_df),
)

print(
    "Focused training horizon range:",
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].min(),
    "to",
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].max(),
)

print(
    "Focused validation horizon range:",
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].min(),
    "to",
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].max(),
)

Focused training rows: 303618
Focused validation rows: 58937
Focused training horizon range: 13 to 72
Focused validation horizon range: 13 to 72


In [80]:
assert not focused_train_df.empty
assert not focused_validation_df.empty

assert (
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].min()
    == LEARNED_HORIZON_MIN
)

assert (
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].max()
    == LEARNED_HORIZON_MAX
)

assert (
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].min()
    == LEARNED_HORIZON_MIN
)

assert (
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].max()
    == LEARNED_HORIZON_MAX
)

assert (
    focused_train_df[
        MODEL_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    focused_validation_df[
        MODEL_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    focused_train_df[
        TARGET_COLUMN
    ].isna().sum()
    == 0
)

assert (
    focused_validation_df[
        TARGET_COLUMN
    ].isna().sum()
    == 0
)


print(
    "Focused 13–72h training and "
    "validation subsets validated."
)

Focused 13–72h training and validation subsets validated.


In [81]:
X_train_focused = focused_train_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_train_focused = focused_train_df[
    TARGET_COLUMN
].astype(float).copy()


X_validation_focused = (
    focused_validation_df[
        MODEL_FEATURE_COLUMNS
    ].copy()
)

y_validation_focused = (
    focused_validation_df[
        TARGET_COLUMN
    ].astype(float).copy()
)


assert (
    X_train_focused.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)

assert (
    X_validation_focused.columns.tolist()
    == MODEL_FEATURE_COLUMNS
)


print(
    "Focused train matrix:",
    X_train_focused.shape,
)

print(
    "Focused validation matrix:",
    X_validation_focused.shape,
)

Focused train matrix: (303618, 56)
Focused validation matrix: (58937, 56)


In [82]:
production_xgb_parameters = dict(
    production_metadata[
        "model_parameters"
    ]
)

In [83]:
FOCUSED_XGB_E1_PARAMETERS = {
    "objective": "reg:squarederror",
    "n_estimators": 1800,
    "learning_rate": 0.04,
    "max_depth": 4,
    "min_child_weight": 15,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.2,
    "reg_lambda": 2.0,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "early_stopping_rounds": 75,
    "random_state": 42,
    "n_jobs": -1,
}

In [84]:
for parameter_name, expected_value in (
    FOCUSED_XGB_E1_PARAMETERS.items()
):
    metadata_value = (
        production_xgb_parameters.get(
            parameter_name
        )
    )

    assert metadata_value == expected_value, (
        "Focused E1 parameter differs "
        "from production metadata: "
        f"{parameter_name}. "
        f"metadata={metadata_value!r}, "
        f"benchmark={expected_value!r}"
    )


print(
    "Focused E1 parameters match "
    "the production champion configuration."
)

Focused E1 parameters match the production champion configuration.


In [85]:
focused_xgb_e1_model = XGBRegressor(
    **FOCUSED_XGB_E1_PARAMETERS
)


focused_xgb_e1_model

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.85
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",75
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'rmse'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [86]:
focused_xgb_e1_training_start = (
    perf_counter()
)


focused_xgb_e1_model.fit(
    X_train_focused,
    y_train_focused,
    eval_set=[
        (
            X_validation_focused,
            y_validation_focused,
        )
    ],
    verbose=False,
)


focused_xgb_e1_training_seconds = (
    perf_counter()
    - focused_xgb_e1_training_start
)


print(
    "Focused XGBoost E1 training time:",
    f"{focused_xgb_e1_training_seconds:.3f}",
    "seconds",
)

print(
    "Best iteration:",
    focused_xgb_e1_model.best_iteration,
)

print(
    "Best validation score:",
    focused_xgb_e1_model.best_score,
)

Focused XGBoost E1 training time: 22.857 seconds
Best iteration: 149
Best validation score: 10.003156368182772


In [87]:
assert (
    focused_xgb_e1_model.n_features_in_
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    focused_xgb_e1_model.best_iteration
    is not None
)

assert (
    focused_xgb_e1_model.best_iteration
    >= 0
)


print(
    "Focused XGBoost E1 fitted-model "
    "contract validated."
)

Focused XGBoost E1 fitted-model contract validated.


### **12E.1.** Learned-Region Comparison

The focused challenger is first compared directly with the existing production
estimator on the 13–72 hour validation region.

This isolates learned-model performance before persistence routing is
introduced.

Both estimators are evaluated on exactly the same 13–72 hour validation rows.

Any difference therefore reflects the change in training scope rather than
short-horizon persistence behavior.

In [88]:
production_xgb_focused_predictions = (
    production_model.predict(
        X_validation_focused
    )
)


assert np.isfinite(
    production_xgb_focused_predictions
).all()


production_xgb_focused_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            production_xgb_focused_predictions
        ),
    )
)

In [89]:
focused_xgb_e1_prediction_start = (
    perf_counter()
)


focused_xgb_e1_learned_predictions = (
    focused_xgb_e1_model.predict(
        X_validation_focused
    )
)


focused_xgb_e1_learned_prediction_seconds = (
    perf_counter()
    - focused_xgb_e1_prediction_start
)


assert np.isfinite(
    focused_xgb_e1_learned_predictions
).all()


focused_xgb_e1_learned_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            focused_xgb_e1_learned_predictions
        ),
    )
)

In [90]:
focused_xgb_learned_comparison_df = pd.DataFrame(
    [
        {
            "candidate": (
                "production_xgboost_shallower"
            ),
            **production_xgb_focused_metrics,
        },
        {
            "candidate": (
                "focused_xgb_e1"
            ),
            **focused_xgb_e1_learned_metrics,
        },
    ]
)

In [91]:
focused_xgb_learned_comparison_df[
    "rmse_improvement_vs_production_pct"
] = focused_xgb_learned_comparison_df[
    "rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_xgb_focused_metrics[
                    "rmse"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


focused_xgb_learned_comparison_df[
    "mae_improvement_vs_production_pct"
] = focused_xgb_learned_comparison_df[
    "mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_xgb_focused_metrics[
                    "mae"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


display(
    focused_xgb_learned_comparison_df
)

,candidate,mae,rmse,r2,rmse_improvement_vs_production_pct,mae_improvement_vs_production_pct
0,production_xgboost_shallower,7.178338,9.713671,-0.009030,0.000000,0.000000
1,focused_xgb_e1,7.690803,10.003156,-0.070068,-2.980189,-7.139054


In [92]:
focused_xgb_e1_negative_learned = int(
    (
        focused_xgb_e1_learned_predictions
        < 0
    ).sum()
)


production_xgb_negative_learned = int(
    (
        production_xgb_focused_predictions
        < 0
    ).sum()
)


learned_prediction_behavior_df = pd.DataFrame(
    [
        {
            "candidate": (
                "production_xgboost_shallower"
            ),
            "negative_predictions": (
                production_xgb_negative_learned
            ),
            "prediction_min": float(
                production_xgb_focused_predictions.min()
            ),
            "prediction_max": float(
                production_xgb_focused_predictions.max()
            ),
        },
        {
            "candidate": (
                "focused_xgb_e1"
            ),
            "negative_predictions": (
                focused_xgb_e1_negative_learned
            ),
            "prediction_min": float(
                focused_xgb_e1_learned_predictions.min()
            ),
            "prediction_max": float(
                focused_xgb_e1_learned_predictions.max()
            ),
        },
    ]
)


display(
    learned_prediction_behavior_df
)

,candidate,negative_predictions,prediction_min,prediction_max
0,production_xgboost_shallower,0,1.303076,99.575546
1,focused_xgb_e1,0,4.894518,89.964134


### **12E.2.** Full Hybrid Validation Evaluation

The focused estimator is now evaluated as a complete production-style
forecasting strategy.

The routing rule remains unchanged:

- hours 1–12: current-value persistence
- hours 13–72: focused XGBoost E1

This provides the direct validation comparison against production model
version 1.

Because the persistence component is identical, any change in full hybrid
performance originates from the learned 13–72 hour estimator.

In [93]:
(
    focused_xgb_e1_raw_predictions,
    focused_xgb_e1_hybrid_predictions,
    focused_xgb_e1_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=focused_xgb_e1_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [94]:
focused_xgb_e1_model_size_bytes = (
    measure_serialized_model_size_bytes(
        focused_xgb_e1_model
    )
)


print(
    "Focused XGBoost E1 artifact size:",
    f"{focused_xgb_e1_model_size_bytes / (1024 ** 2):.3f}",
    "MiB",
)

Focused XGBoost E1 artifact size: 0.370 MiB


In [95]:
focused_xgb_e1_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "focused_xgb_e1"
        ),
        model_family="XGBRegressor",
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            focused_xgb_e1_raw_predictions
        ),
        operational_predictions=(
            focused_xgb_e1_hybrid_predictions
        ),
        training_seconds=(
            focused_xgb_e1_training_seconds
        ),
        prediction_seconds=(
            focused_xgb_e1_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            focused_xgb_e1_model_size_bytes
        ),
        parameters=(
            FOCUSED_XGB_E1_PARAMETERS
        ),
    )
)


benchmark_results[
    "focused_xgb_e1"
] = focused_xgb_e1_result

In [96]:
focused_xgb_e1_leaderboard_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "focused_xgb_e1"
                ]
            ),
        ]
    )
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)

In [97]:
focused_xgb_e1_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = focused_xgb_e1_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


focused_xgb_e1_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = focused_xgb_e1_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


display(
    focused_xgb_e1_leaderboard_df[
        [
            "candidate",
            "training_scope",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,0,NaN,0.394257,0.645863
1,focused_xgb_e1,13-72h,7.119511,9.666899,0.016948,7.690803,10.003156,-2.625883,-6.330513,0,22.85657,0.259038,0.370402


### **12E.3.** Horizon Stability

Overall validation metrics can hide improvements or regressions at specific
forecast distances.

The production champion and focused E1 challenger are therefore compared across
the existing horizon groups.

The first two groups must remain identical because both strategies use the same
persistence predictions there.

The meaningful comparison is:

- 13–24 hours
- 25–48 hours
- 49–72 hours

In [98]:
focused_xgb_horizon_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "production_champion_v1"
            ][
                "horizon_group_metrics"
            ],
            benchmark_results[
                "focused_xgb_e1"
            ][
                "horizon_group_metrics"
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    focused_xgb_horizon_comparison_df
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,focused_xgb_e1,1-6h,1,6,6190,3.648142,6.990142,0.516136
2,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
3,focused_xgb_e1,7-12h,7,12,6129,5.131832,8.653139,0.268671
4,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
5,focused_xgb_e1,13-24h,13,24,12163,7.722965,10.071922,0.019227
6,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
7,focused_xgb_e1,25-48h,25,48,23709,7.688572,9.962901,-0.114818
8,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042
9,focused_xgb_e1,49-72h,49,72,23065,7.676136,10.008064,-0.082037


In [99]:
learned_group_comparison_records = []


for group_name in (
    "13-24h",
    "25-48h",
    "49-72h",
):
    champion_group = (
        benchmark_results[
            "production_champion_v1"
        ][
            "horizon_group_metrics"
        ]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    candidate_group = (
        benchmark_results[
            "focused_xgb_e1"
        ][
            "horizon_group_metrics"
        ]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    learned_group_comparison_records.append(
        {
            "horizon_group": (
                group_name
            ),
            "champion_mae": float(
                champion_group["mae"]
            ),
            "candidate_mae": float(
                candidate_group["mae"]
            ),
            "mae_improvement_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        champion_group[
                            "mae"
                        ]
                    ),
                    candidate_value=float(
                        candidate_group[
                            "mae"
                        ]
                    ),
                )
            ),
            "champion_rmse": float(
                champion_group["rmse"]
            ),
            "candidate_rmse": float(
                candidate_group["rmse"]
            ),
            "rmse_improvement_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        champion_group[
                            "rmse"
                        ]
                    ),
                    candidate_value=float(
                        candidate_group[
                            "rmse"
                        ]
                    ),
                )
            ),
        }
    )


focused_xgb_learned_group_improvement_df = (
    pd.DataFrame(
        learned_group_comparison_records
    )
)


display(
    focused_xgb_learned_group_improvement_df
)

,horizon_group,champion_mae,candidate_mae,mae_improvement_pct,champion_rmse,candidate_rmse,rmse_improvement_pct
0,13-24h,7.117727,7.722965,-8.503256,9.692964,10.071922,-3.909622
1,25-48h,7.168849,7.688572,-7.249746,9.669828,9.962901,-3.030804
2,49-72h,7.220054,7.676136,-6.316881,9.769384,10.008064,-2.443141


In [100]:
for group_name in (
    "1-6h",
    "7-12h",
):
    group_df = (
        focused_xgb_horizon_comparison_df.loc[
            focused_xgb_horizon_comparison_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        values = (
            group_df[
                metric_name
            ]
            .astype(float)
            .to_numpy()
        )

        np.testing.assert_allclose(
            values,
            np.repeat(
                values[0],
                len(values),
            ),
            rtol=0.0,
            atol=1e-12,
        )


print(
    "Champion and E1 remain identical "
    "for persistence-routed horizons."
)

Champion and E1 remain identical for persistence-routed horizons.


### **12E.4.** Severe-PM2.5 and Prediction Validity Check

The focused challenger must not be judged only by average validation error.

This subphase inspects:

- performance when observed PM2.5 is at least 55.5 µg/m³
- negative raw predictions
- operational clipping
- numerical validity
- prediction range

These checks are diagnostic in Phase 12E.

Formal robustness and promotion criteria will be applied later after all serious
challengers have been evaluated.

In [101]:
def flatten_severe_result(
    *,
    candidate_name: str,
    severe_result: dict[str, Any],
) -> dict[str, Any]:
    """Flatten one severe-PM2.5 result for display."""

    metrics = severe_result.get(
        "metrics"
    )

    return {
        "candidate": (
            candidate_name
        ),
        "threshold_ug_m3": (
            severe_result[
                "threshold_ug_m3"
            ]
        ),
        "sample_count": (
            severe_result[
                "sample_count"
            ]
        ),
        "mae": (
            None
            if metrics is None
            else metrics["mae"]
        ),
        "rmse": (
            None
            if metrics is None
            else metrics["rmse"]
        ),
        "r2": (
            None
            if metrics is None
            else metrics["r2"]
        ),
    }


focused_xgb_severe_comparison_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=(
                "production_champion_v1"
            ),
            severe_result=(
                benchmark_results[
                    "production_champion_v1"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                "focused_xgb_e1"
            ),
            severe_result=(
                benchmark_results[
                    "focused_xgb_e1"
                ]["severe_pm25"]
            ),
        ),
    ]
)


display(
    focused_xgb_severe_comparison_df
)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,production_champion_v1,55.5,181,49.371822,50.15318,-52.451700
1,focused_xgb_e1,55.5,181,48.691305,49.79469,-51.690295


In [102]:
focused_xgb_validity_df = pd.DataFrame(
    [
        {
            "candidate": (
                candidate_name
            ),
            **benchmark_results[
                candidate_name
            ][
                "prediction_validity"
            ],
        }
        for candidate_name in (
            "production_champion_v1",
            "focused_xgb_e1",
        )
    ]
)


display(
    focused_xgb_validity_df
)

,candidate,prediction_count,raw_nan_count,raw_inf_count,negative_raw_count,clipped_prediction_count,raw_min,raw_max,operational_min,operational_max
0,production_champion_v1,71256,0,0,0,0,0.1,99.575546,0.1,99.575546
1,focused_xgb_e1,71256,0,0,0,0,0.1,89.964134,0.1,89.964134


In [103]:
focused_xgb_e1_validity = (
    benchmark_results[
        "focused_xgb_e1"
    ][
        "prediction_validity"
    ]
)


assert (
    focused_xgb_e1_validity[
        "raw_nan_count"
    ]
    == 0
)

assert (
    focused_xgb_e1_validity[
        "raw_inf_count"
    ]
    == 0
)


print(
    "Focused XGBoost E1 prediction "
    "validity checks passed."
)

Focused XGBoost E1 prediction validity checks passed.


### **12E.5.** Focused-Training Diagnostic Decision

This phase does not select a production challenger.

Its purpose is to determine whether aligning XGBoost training with the learned
13–72 hour routing region provides enough validation evidence to justify
testing a small number of focused XGBoost variants in Phase 12F.

The diagnostic decision considers:

- full hybrid RMSE and MAE
- learned-region RMSE and MAE
- 13–24, 25–48, and 49–72 hour stability
- severe-PM2.5 behavior
- prediction validity
- training and inference cost

A small improvement may justify further controlled XGBoost experimentation,
but it is not sufficient by itself for production replacement.

In [104]:
focused_e1_overall = (
    benchmark_results[
        "focused_xgb_e1"
    ][
        "overall_metrics"
    ]
)

champion_overall = (
    benchmark_results[
        "production_champion_v1"
    ][
        "overall_metrics"
    ]
)


focused_e1_learned = (
    benchmark_results[
        "focused_xgb_e1"
    ][
        "learned_horizon_metrics"
    ]
)

champion_learned = (
    benchmark_results[
        "production_champion_v1"
    ][
        "learned_horizon_metrics"
    ]
)


focused_e1_hybrid_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["rmse"]
        ),
        candidate_value=(
            focused_e1_overall["rmse"]
        ),
    )
)


focused_e1_hybrid_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["mae"]
        ),
        candidate_value=(
            focused_e1_overall["mae"]
        ),
    )
)


focused_e1_learned_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_learned["rmse"]
        ),
        candidate_value=(
            focused_e1_learned["rmse"]
        ),
    )
)


focused_e1_learned_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_learned["mae"]
        ),
        candidate_value=(
            focused_e1_learned["mae"]
        ),
    )
)

In [105]:
focused_xgb_e1_summary = {
    "status": "COMPLETED",
    "candidate": (
        "focused_xgb_e1"
    ),
    "training_scope": "13-72h",
    "best_iteration": int(
        focused_xgb_e1_model.best_iteration
    ),
    "training_seconds": float(
        focused_xgb_e1_training_seconds
    ),
    "hybrid_mae": float(
        focused_e1_overall["mae"]
    ),
    "hybrid_rmse": float(
        focused_e1_overall["rmse"]
    ),
    "hybrid_r2": float(
        focused_e1_overall["r2"]
    ),
    "hybrid_rmse_improvement_pct": float(
        focused_e1_hybrid_rmse_improvement
    ),
    "hybrid_mae_improvement_pct": float(
        focused_e1_hybrid_mae_improvement
    ),
    "learned_13_72_rmse": float(
        focused_e1_learned["rmse"]
    ),
    "learned_13_72_mae": float(
        focused_e1_learned["mae"]
    ),
    "learned_rmse_improvement_pct": float(
        focused_e1_learned_rmse_improvement
    ),
    "learned_mae_improvement_pct": float(
        focused_e1_learned_mae_improvement
    ),
    "negative_raw_predictions": int(
        focused_xgb_e1_validity[
            "negative_raw_count"
        ]
    ),
    "model_size_mib": float(
        focused_xgb_e1_result[
            "model_size_mib"
        ]
    ),
}


display(
    pd.Series(
        focused_xgb_e1_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12E COMPLETED — focused "
    "13–72h XGBoost has been evaluated."
)

,value
status,COMPLETED
candidate,focused_xgb_e1
training_scope,13-72h
best_iteration,149
training_seconds,22.85657
hybrid_mae,7.119511
hybrid_rmse,9.666899
hybrid_r2,0.016948
hybrid_rmse_improvement_pct,-2.625883
hybrid_mae_improvement_pct,-6.330513


Phase 12E COMPLETED — focused 13–72h XGBoost has been evaluated.


## **12F.** Focused XGBoost Continuation Gate

Phase 12E tested whether aligning XGBoost training directly with the 13–72 hour
learned-model region could improve the existing production strategy.

The experiment deliberately changed only the training scope while preserving:

- the production estimator family
- the ordered 56-feature contract
- the production XGBoost hyperparameters
- chronological validation
- early stopping methodology
- the 1–12 hour persistence routing rule
- the 13–72 hour learned-model routing rule

The focused E1 challenger did not improve the production champion.

This phase therefore determines whether there is sufficient validation evidence
to justify further focused-XGBoost parameter variants.

The decision is made before running any additional XGBoost experiments so that
validation data is not repeatedly searched without a meaningful experimental
signal.

In [106]:
focused_xgb_gate_evidence = {
    "hybrid_rmse_improvement_pct": (
        focused_e1_hybrid_rmse_improvement
    ),
    "hybrid_mae_improvement_pct": (
        focused_e1_hybrid_mae_improvement
    ),
    "learned_rmse_improvement_pct": (
        focused_e1_learned_rmse_improvement
    ),
    "learned_mae_improvement_pct": (
        focused_e1_learned_mae_improvement
    ),
}

In [107]:
for _, row in (
    focused_xgb_learned_group_improvement_df
    .iterrows()
):
    group_key = (
        str(row["horizon_group"])
        .replace("-", "_")
    )

    focused_xgb_gate_evidence[
        f"{group_key}_rmse_improvement_pct"
    ] = float(
        row["rmse_improvement_pct"]
    )

    focused_xgb_gate_evidence[
        f"{group_key}_mae_improvement_pct"
    ] = float(
        row["mae_improvement_pct"]
    )

In [108]:
display(
    pd.Series(
        focused_xgb_gate_evidence,
        name="value",
    ).to_frame()
)

,value
hybrid_rmse_improvement_pct,-2.625883
hybrid_mae_improvement_pct,-6.330513
learned_rmse_improvement_pct,-2.980189
learned_mae_improvement_pct,-7.139054
13_24h_rmse_improvement_pct,-3.909622
13_24h_mae_improvement_pct,-8.503256
25_48h_rmse_improvement_pct,-3.030804
25_48h_mae_improvement_pct,-7.249746
49_72h_rmse_improvement_pct,-2.443141
49_72h_mae_improvement_pct,-6.316881


In [109]:
focused_xgb_has_primary_signal = any(
    [
        focused_e1_hybrid_rmse_improvement
        > 0.0,
        focused_e1_hybrid_mae_improvement
        > 0.0,
        focused_e1_learned_rmse_improvement
        > 0.0,
        focused_e1_learned_mae_improvement
        > 0.0,
    ]
)

In [110]:
learned_group_rmse_improvements = (
    focused_xgb_learned_group_improvement_df[
        "rmse_improvement_pct"
    ]
    .astype(float)
    .to_numpy()
)


focused_xgb_has_horizon_signal = bool(
    (
        learned_group_rmse_improvements
        > 0.0
    ).any()
)

In [111]:
focused_xgb_continue = bool(
    focused_xgb_has_primary_signal
    or focused_xgb_has_horizon_signal
)

In [112]:
focused_xgb_gate_df = pd.DataFrame(
    [
        {
            "criterion": (
                "Positive overall or learned metric"
            ),
            "passed": (
                focused_xgb_has_primary_signal
            ),
        },
        {
            "criterion": (
                "Positive learned horizon-group RMSE signal"
            ),
            "passed": (
                focused_xgb_has_horizon_signal
            ),
        },
        {
            "criterion": (
                "Continue focused XGBoost variants"
            ),
            "passed": (
                focused_xgb_continue
            ),
        },
    ]
)


display(
    focused_xgb_gate_df
)

,criterion,passed
0,Positive overall or learned metric,False
1,Positive learned horizon-group RMSE signal,False
2,Continue focused XGBoost variants,False


In [113]:
assert not focused_xgb_continue, (
    "Focused XGBoost produced a positive "
    "validation signal. Reconsider whether "
    "additional variants are justified."
)

In [114]:
print(
    "Focused XGBoost continuation gate: STOP"
)

print(
    "No E2/E3 focused XGBoost variants "
    "will be trained."
)

Focused XGBoost continuation gate: STOP
No E2/E3 focused XGBoost variants will be trained.


In [115]:
champion_severe_metrics = (
    benchmark_results[
        "production_champion_v1"
    ]["severe_pm25"]["metrics"]
)

focused_e1_severe_metrics = (
    benchmark_results[
        "focused_xgb_e1"
    ]["severe_pm25"]["metrics"]
)


focused_e1_severe_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_severe_metrics[
                "rmse"
            ]
        ),
        candidate_value=(
            focused_e1_severe_metrics[
                "rmse"
            ]
        ),
    )
)


focused_e1_severe_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_severe_metrics[
                "mae"
            ]
        ),
        candidate_value=(
            focused_e1_severe_metrics[
                "mae"
            ]
        ),
    )
)

In [116]:
severe_signal_df = pd.DataFrame(
    [
        {
            "metric": "MAE",
            "champion": (
                champion_severe_metrics[
                    "mae"
                ]
            ),
            "focused_e1": (
                focused_e1_severe_metrics[
                    "mae"
                ]
            ),
            "improvement_pct": (
                focused_e1_severe_mae_improvement
            ),
        },
        {
            "metric": "RMSE",
            "champion": (
                champion_severe_metrics[
                    "rmse"
                ]
            ),
            "focused_e1": (
                focused_e1_severe_metrics[
                    "rmse"
                ]
            ),
            "improvement_pct": (
                focused_e1_severe_rmse_improvement
            ),
        },
    ]
)


display(
    severe_signal_df
)

,metric,champion,focused_e1,improvement_pct
0,MAE,49.371822,48.691305,1.378351
1,RMSE,50.153180,49.794690,0.714790


In [117]:
focused_xgb_continuation_summary = {
    "status": "CLOSED",
    "experiment_family": (
        "focused_xgboost_13_72"
    ),
    "e1_best_iteration": int(
        focused_xgb_e1_model.best_iteration
    ),
    "hybrid_rmse_improvement_pct": float(
        focused_e1_hybrid_rmse_improvement
    ),
    "hybrid_mae_improvement_pct": float(
        focused_e1_hybrid_mae_improvement
    ),
    "learned_rmse_improvement_pct": float(
        focused_e1_learned_rmse_improvement
    ),
    "learned_mae_improvement_pct": float(
        focused_e1_learned_mae_improvement
    ),
    "all_learned_rmse_groups_improved": bool(
        (
            learned_group_rmse_improvements
            > 0.0
        ).all()
    ),
    "any_learned_rmse_group_improved": bool(
        (
            learned_group_rmse_improvements
            > 0.0
        ).any()
    ),
    "severe_rmse_improvement_pct": float(
        focused_e1_severe_rmse_improvement
    ),
    "negative_predictions": int(
        focused_xgb_e1_validity[
            "negative_raw_count"
        ]
    ),
    "continue_to_xgb_variants": (
        focused_xgb_continue
    ),
    "decision": (
        "STOP_FOCUSED_XGBOOST_VARIANTS"
    ),
}


display(
    pd.Series(
        focused_xgb_continuation_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12F PASSED — further focused "
    "XGBoost variants are not justified."
)

,value
status,CLOSED
experiment_family,focused_xgboost_13_72
e1_best_iteration,149
hybrid_rmse_improvement_pct,-2.625883
hybrid_mae_improvement_pct,-6.330513
learned_rmse_improvement_pct,-2.980189
learned_mae_improvement_pct,-7.139054
all_learned_rmse_groups_improved,False
any_learned_rmse_group_improved,False
severe_rmse_improvement_pct,0.71479


Phase 12F PASSED — further focused XGBoost variants are not justified.


## **12G.** Focused Histogram Gradient Boosting Challenger

The focused-XGBoost branch has been closed because restricting XGBoost training
to horizons 13–72 produced consistent validation regressions.

The benchmark now evaluates a different estimator family.

Histogram Gradient Boosting was the strongest non-XGBoost learned estimator in
the original Phase 3 experiment and requires no additional project dependency.

This phase begins with a controlled challenger named `focused_histgb_h1`.

The historical Histogram Gradient Boosting configuration is preserved exactly.
The only intentional experimental change is the training scope:

- historical HistGB control: trained across horizons 1–72
- focused HistGB H1: trained only across horizons 13–72

The production routing contract remains unchanged:

- hours 1–12: current-value persistence
- hours 13–72: Histogram Gradient Boosting

The historical configuration used `early_stopping=False`; that behavior is
preserved so that training scope remains the only changed variable.

No hyperparameter tuning is performed in H1.

The historical test split remains excluded from performance evaluation.

In [118]:
assert (
    X_train_focused.shape[1]
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    X_validation_focused.shape[1]
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].min()
    == LEARNED_HORIZON_MIN
)

assert (
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].min()
    == LEARNED_HORIZON_MIN
)

assert (
    focused_train_df[
        FORECAST_HORIZON_COLUMN
    ].max()
    == LEARNED_HORIZON_MAX
)

assert (
    focused_validation_df[
        FORECAST_HORIZON_COLUMN
    ].max()
    == LEARNED_HORIZON_MAX
)


print(
    "Focused 13–72h experiment matrices "
    "remain valid for HistGB."
)

Focused 13–72h experiment matrices remain valid for HistGB.


In [119]:
FOCUSED_HISTGB_H1_PARAMETERS = {
    "loss": "squared_error",
    "learning_rate": 0.05,
    "max_iter": 300,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 30,
    "l2_regularization": 1.0,
    "early_stopping": False,
    "random_state": 42,
}

In [120]:
historical_histgb_parameters_12d = (
    historical_histgb_model.get_params()
)


for parameter_name, expected_value in (
    FOCUSED_HISTGB_H1_PARAMETERS.items()
):
    historical_value = (
        historical_histgb_parameters_12d[
            parameter_name
        ]
    )

    assert historical_value == expected_value, (
        "Focused HistGB H1 parameter differs "
        "from the reproduced historical control: "
        f"{parameter_name}. "
        f"historical={historical_value!r}, "
        f"focused={expected_value!r}"
    )


print(
    "Focused HistGB H1 parameters match "
    "the historical Phase 3 configuration."
)

Focused HistGB H1 parameters match the historical Phase 3 configuration.


In [121]:
focused_histgb_h1_model = (
    HistGradientBoostingRegressor(
        **FOCUSED_HISTGB_H1_PARAMETERS
    )
)


focused_histgb_h1_model

,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.05
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",300
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",30
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",1.0
,"early_stopping early_stopping: 'auto' or bool, default='auto'If 'auto', early stopping is enabled if the sample size is larger than10000 or if `X_val` and `y_val` are passed to `fit`. If True, early stoppingis enabled, otherwise early stopping is disabled... versionadded:: 0.23",False
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0


In [122]:
focused_histgb_h1_training_start = (
    perf_counter()
)


focused_histgb_h1_model.fit(
    X_train_focused,
    y_train_focused,
)


focused_histgb_h1_training_seconds = (
    perf_counter()
    - focused_histgb_h1_training_start
)


print(
    "Focused HistGB H1 training time:",
    f"{focused_histgb_h1_training_seconds:.3f}",
    "seconds",
)

print(
    "Iterations completed:",
    focused_histgb_h1_model.n_iter_,
)

Focused HistGB H1 training time: 31.001 seconds
Iterations completed: 300


In [123]:
assert (
    focused_histgb_h1_model.n_features_in_
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    focused_histgb_h1_model.n_iter_
    == FOCUSED_HISTGB_H1_PARAMETERS[
        "max_iter"
    ]
)


print(
    "Focused HistGB H1 fitted-model "
    "contract validated."
)

Focused HistGB H1 fitted-model contract validated.


### **12G.1.** Learned-Region Comparison

The focused HistGB challenger is first evaluated directly on horizons 13–72.

Three learned-estimator views are compared:

1. the existing production XGBoost estimator
2. the historical 1–72 trained HistGB control
3. the new 13–72 trained HistGB H1 challenger

This separates two questions:

- Can HistGB compete with the production XGBoost estimator?
- Does horizon-specialized training improve HistGB relative to its own
  historical configuration?

Persistence routing is not involved in this direct learned-region comparison.

In [124]:
historical_histgb_focused_predictions = (
    historical_histgb_model.predict(
        X_validation_focused
    )
)


assert np.isfinite(
    historical_histgb_focused_predictions
).all()


historical_histgb_focused_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            historical_histgb_focused_predictions
        ),
    )
)

In [125]:
focused_histgb_h1_prediction_start = (
    perf_counter()
)


focused_histgb_h1_learned_predictions = (
    focused_histgb_h1_model.predict(
        X_validation_focused
    )
)


focused_histgb_h1_learned_prediction_seconds = (
    perf_counter()
    - focused_histgb_h1_prediction_start
)


assert np.isfinite(
    focused_histgb_h1_learned_predictions
).all()


focused_histgb_h1_learned_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            focused_histgb_h1_learned_predictions
        ),
    )
)

In [126]:
histgb_learned_comparison_df = (
    pd.DataFrame(
        [
            {
                "candidate": (
                    "production_xgboost_shallower"
                ),
                "training_scope": (
                    "historical 1-72h"
                ),
                **production_xgb_focused_metrics,
            },
            {
                "candidate": (
                    "historical_histgb"
                ),
                "training_scope": (
                    "historical 1-72h"
                ),
                **historical_histgb_focused_metrics,
            },
            {
                "candidate": (
                    "focused_histgb_h1"
                ),
                "training_scope": "13-72h",
                **focused_histgb_h1_learned_metrics,
            },
        ]
    )
)

In [127]:
histgb_learned_comparison_df[
    "rmse_improvement_vs_production_pct"
] = histgb_learned_comparison_df[
    "rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_xgb_focused_metrics[
                    "rmse"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


histgb_learned_comparison_df[
    "mae_improvement_vs_production_pct"
] = histgb_learned_comparison_df[
    "mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_xgb_focused_metrics[
                    "mae"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)

In [128]:
histgb_learned_comparison_df[
    "rmse_improvement_vs_historical_histgb_pct"
] = histgb_learned_comparison_df[
    "rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                historical_histgb_focused_metrics[
                    "rmse"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


histgb_learned_comparison_df[
    "mae_improvement_vs_historical_histgb_pct"
] = histgb_learned_comparison_df[
    "mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                historical_histgb_focused_metrics[
                    "mae"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


display(
    histgb_learned_comparison_df
)

,candidate,training_scope,mae,rmse,r2,rmse_improvement_vs_production_pct,mae_improvement_vs_production_pct,rmse_improvement_vs_historical_histgb_pct,mae_improvement_vs_historical_histgb_pct
0,production_xgboost_shallower,historical 1-72h,7.178338,9.713671,-0.009030,0.000000,0.000000,1.895045,1.110829
1,historical_histgb,historical 1-72h,7.258972,9.901305,-0.048389,-1.931651,-1.123307,0.000000,0.000000
2,focused_histgb_h1,13-72h,7.405951,9.716116,-0.009538,-0.025177,-3.170839,1.870345,-2.024787


In [129]:
histgb_learned_prediction_behavior_df = (
    pd.DataFrame(
        [
            {
                "candidate": (
                    "production_xgboost_shallower"
                ),
                "negative_predictions": int(
                    (
                        production_xgb_focused_predictions
                        < 0
                    ).sum()
                ),
                "prediction_min": float(
                    production_xgb_focused_predictions.min()
                ),
                "prediction_max": float(
                    production_xgb_focused_predictions.max()
                ),
            },
            {
                "candidate": (
                    "historical_histgb"
                ),
                "negative_predictions": int(
                    (
                        historical_histgb_focused_predictions
                        < 0
                    ).sum()
                ),
                "prediction_min": float(
                    historical_histgb_focused_predictions.min()
                ),
                "prediction_max": float(
                    historical_histgb_focused_predictions.max()
                ),
            },
            {
                "candidate": (
                    "focused_histgb_h1"
                ),
                "negative_predictions": int(
                    (
                        focused_histgb_h1_learned_predictions
                        < 0
                    ).sum()
                ),
                "prediction_min": float(
                    focused_histgb_h1_learned_predictions.min()
                ),
                "prediction_max": float(
                    focused_histgb_h1_learned_predictions.max()
                ),
            },
        ]
    )
)


display(
    histgb_learned_prediction_behavior_df
)

,candidate,negative_predictions,prediction_min,prediction_max
0,production_xgboost_shallower,0,1.303076,99.575546
1,historical_histgb,0,4.437495,101.705413
2,focused_histgb_h1,0,5.587126,81.374006


### **12G.2.** Full Hybrid Validation Evaluation

The focused HistGB estimator is now evaluated as a complete production-style
hybrid strategy.

Routing remains identical to production:

- hours 1–12: current-value persistence
- hours 13–72: focused HistGB H1

The resulting hybrid is compared against:

- production champion version 1
- the historical HistGB hybrid control from Phase 12D

This determines both whether focused HistGB improves upon its historical
configuration and whether it can challenge the current production strategy.

In [130]:
(
    focused_histgb_h1_raw_predictions,
    focused_histgb_h1_hybrid_predictions,
    focused_histgb_h1_hybrid_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=focused_histgb_h1_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [131]:
focused_histgb_h1_model_size_bytes = (
    measure_serialized_model_size_bytes(
        focused_histgb_h1_model
    )
)


print(
    "Focused HistGB H1 artifact size:",
    f"{focused_histgb_h1_model_size_bytes / (1024 ** 2):.3f}",
    "MiB",
)

Focused HistGB H1 artifact size: 1.096 MiB


In [132]:
focused_histgb_h1_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "focused_histgb_h1"
        ),
        model_family=(
            "HistGradientBoostingRegressor"
        ),
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            focused_histgb_h1_raw_predictions
        ),
        operational_predictions=(
            focused_histgb_h1_hybrid_predictions
        ),
        training_seconds=(
            focused_histgb_h1_training_seconds
        ),
        prediction_seconds=(
            focused_histgb_h1_hybrid_prediction_seconds
        ),
        model_size_bytes=(
            focused_histgb_h1_model_size_bytes
        ),
        parameters=(
            FOCUSED_HISTGB_H1_PARAMETERS
        ),
    )
)


benchmark_results[
    "focused_histgb_h1"
] = focused_histgb_h1_result

In [133]:
histgb_hybrid_leaderboard_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "historical_histgb_hybrid"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "focused_histgb_h1"
                ]
            ),
        ]
    )
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)

In [134]:
histgb_hybrid_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = histgb_hybrid_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


histgb_hybrid_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = histgb_hybrid_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)

In [135]:
historical_histgb_hybrid_metrics = (
    benchmark_results[
        "historical_histgb_hybrid"
    ]["overall_metrics"]
)


histgb_hybrid_leaderboard_df[
    "rmse_improvement_vs_historical_histgb_pct"
] = histgb_hybrid_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                historical_histgb_hybrid_metrics[
                    "rmse"
                ]
            ),
            candidate_value=float(
                value
            ),
        )
    )
)

In [136]:
display(
    histgb_hybrid_leaderboard_df[
        [
            "candidate",
            "training_scope",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "rmse_improvement_vs_historical_histgb_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,rmse_improvement_vs_historical_histgb_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,production_champion_v1,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,1.672525,0,NaN,0.394257,0.645863
1,focused_histgb_h1,13-72h,6.883905,9.421639,0.066198,7.405951,9.716116,-0.022145,-2.811722,1.650750,0,31.001019,0.888559,1.096167
2,historical_histgb_hybrid,historical 1-72h control,6.762337,9.579777,0.034588,7.258972,9.901305,-1.700974,-0.996086,0.000000,0,30.336777,1.129033,1.096182


### **12G.3.** Horizon Stability

The HistGB challenger is now compared across the standard forecast-horizon
groups.

As with every hybrid candidate, hours 1–12 must remain identical to production
because those horizons use persistence.

The meaningful learned-model comparison is therefore:

- 13–24 hours
- 25–48 hours
- 49–72 hours

Focused H1 is compared with both the production champion and the historical
HistGB hybrid so that specialization effects can be distinguished from
model-family effects.

In [137]:
histgb_horizon_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "production_champion_v1"
            ]["horizon_group_metrics"],
            benchmark_results[
                "historical_histgb_hybrid"
            ]["horizon_group_metrics"],
            benchmark_results[
                "focused_histgb_h1"
            ]["horizon_group_metrics"],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
    .reset_index(
        drop=True
    )
)


display(
    histgb_horizon_comparison_df
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
0,production_champion_v1,1-6h,1,6,6190,3.648142,6.990142,0.516136
1,historical_histgb_hybrid,1-6h,1,6,6190,3.648142,6.990142,0.516136
2,focused_histgb_h1,1-6h,1,6,6190,3.648142,6.990142,0.516136
3,production_champion_v1,7-12h,7,12,6129,5.131832,8.653139,0.268671
4,historical_histgb_hybrid,7-12h,7,12,6129,5.131832,8.653139,0.268671
5,focused_histgb_h1,7-12h,7,12,6129,5.131832,8.653139,0.268671
6,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
7,focused_histgb_h1,13-24h,13,24,12163,7.381548,9.730627,0.084569
8,historical_histgb_hybrid,13-24h,13,24,12163,7.160805,9.813267,0.068954
9,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194


In [138]:
for group_name in (
    "1-6h",
    "7-12h",
):
    group_df = (
        histgb_horizon_comparison_df.loc[
            histgb_horizon_comparison_df[
                "horizon_group"
            ].eq(group_name)
        ]
    )

    for metric_name in (
        "mae",
        "rmse",
        "r2",
    ):
        values = (
            group_df[
                metric_name
            ]
            .astype(float)
            .to_numpy()
        )

        np.testing.assert_allclose(
            values,
            np.repeat(
                values[0],
                len(values),
            ),
            rtol=0.0,
            atol=1e-12,
        )


print(
    "All HistGB hybrid comparisons are "
    "identical for horizons 1–12."
)

All HistGB hybrid comparisons are identical for horizons 1–12.


In [139]:
histgb_learned_group_records = []


for group_name in (
    "13-24h",
    "25-48h",
    "49-72h",
):
    production_group = (
        benchmark_results[
            "production_champion_v1"
        ][
            "horizon_group_metrics"
        ]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    historical_group = (
        benchmark_results[
            "historical_histgb_hybrid"
        ][
            "horizon_group_metrics"
        ]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    focused_group = (
        benchmark_results[
            "focused_histgb_h1"
        ][
            "horizon_group_metrics"
        ]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    histgb_learned_group_records.append(
        {
            "horizon_group": (
                group_name
            ),
            "production_rmse": float(
                production_group["rmse"]
            ),
            "historical_histgb_rmse": float(
                historical_group["rmse"]
            ),
            "focused_h1_rmse": float(
                focused_group["rmse"]
            ),
            "h1_rmse_improvement_vs_production_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        production_group[
                            "rmse"
                        ]
                    ),
                    candidate_value=float(
                        focused_group[
                            "rmse"
                        ]
                    ),
                )
            ),
            "h1_rmse_improvement_vs_historical_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        historical_group[
                            "rmse"
                        ]
                    ),
                    candidate_value=float(
                        focused_group[
                            "rmse"
                        ]
                    ),
                )
            ),
            "production_mae": float(
                production_group["mae"]
            ),
            "historical_histgb_mae": float(
                historical_group["mae"]
            ),
            "focused_h1_mae": float(
                focused_group["mae"]
            ),
            "h1_mae_improvement_vs_production_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        production_group[
                            "mae"
                        ]
                    ),
                    candidate_value=float(
                        focused_group[
                            "mae"
                        ]
                    ),
                )
            ),
            "h1_mae_improvement_vs_historical_pct": (
                calculate_improvement_percent(
                    champion_value=float(
                        historical_group[
                            "mae"
                        ]
                    ),
                    candidate_value=float(
                        focused_group[
                            "mae"
                        ]
                    ),
                )
            ),
        }
    )


histgb_learned_group_improvement_df = (
    pd.DataFrame(
        histgb_learned_group_records
    )
)


display(
    histgb_learned_group_improvement_df
)

,horizon_group,production_rmse,historical_histgb_rmse,focused_h1_rmse,h1_rmse_improvement_vs_production_pct,h1_rmse_improvement_vs_historical_pct,production_mae,historical_histgb_mae,focused_h1_mae,h1_mae_improvement_vs_production_pct,h1_mae_improvement_vs_historical_pct
0,13-24h,9.692964,9.813267,9.730627,-0.388557,0.842133,7.117727,7.160805,7.381548,-3.706537,-3.082646
1,25-48h,9.669828,9.934614,9.754224,-0.872775,1.815776,7.168849,7.315578,7.475623,-4.279261,-2.187718
2,49-72h,9.769384,9.913220,9.669097,1.026546,2.462602,7.220054,7.252553,7.347203,-1.761061,-1.305061


### **12G.4.** Severe-PM2.5 and Prediction Validity

The focused HistGB challenger is also inspected for behavior that may not be
visible in headline MAE and RMSE.

The comparison includes:

- observations with PM2.5 at or above 55.5 µg/m³
- negative raw predictions
- operational clipping
- NaN or infinite predictions
- prediction range

These remain diagnostic measures. Final robustness requirements will be applied
later after all serious model families have been evaluated.

In [140]:
histgb_severe_comparison_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=(
                "production_champion_v1"
            ),
            severe_result=(
                benchmark_results[
                    "production_champion_v1"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                "historical_histgb_hybrid"
            ),
            severe_result=(
                benchmark_results[
                    "historical_histgb_hybrid"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                "focused_histgb_h1"
            ),
            severe_result=(
                benchmark_results[
                    "focused_histgb_h1"
                ]["severe_pm25"]
            ),
        ),
    ]
)


display(
    histgb_severe_comparison_df
)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,production_champion_v1,55.5,181,49.371822,50.153180,-52.451700
1,historical_histgb_hybrid,55.5,181,48.171581,49.429039,-50.919308
2,focused_histgb_h1,55.5,181,48.309084,49.820887,-51.745749


In [141]:
histgb_validity_df = pd.DataFrame(
    [
        {
            "candidate": candidate_name,
            **benchmark_results[
                candidate_name
            ]["prediction_validity"],
        }
        for candidate_name in (
            "production_champion_v1",
            "historical_histgb_hybrid",
            "focused_histgb_h1",
        )
    ]
)


display(
    histgb_validity_df
)

,candidate,prediction_count,raw_nan_count,raw_inf_count,negative_raw_count,clipped_prediction_count,raw_min,raw_max,operational_min,operational_max
0,production_champion_v1,71256,0,0,0,0,0.1,99.575546,0.1,99.575546
1,historical_histgb_hybrid,71256,0,0,0,0,0.1,101.705413,0.1,101.705413
2,focused_histgb_h1,71256,0,0,0,0,0.1,88.900000,0.1,88.900000


In [142]:
focused_histgb_h1_validity = (
    benchmark_results[
        "focused_histgb_h1"
    ]["prediction_validity"]
)


assert (
    focused_histgb_h1_validity[
        "raw_nan_count"
    ]
    == 0
)

assert (
    focused_histgb_h1_validity[
        "raw_inf_count"
    ]
    == 0
)


print(
    "Focused HistGB H1 numerical "
    "validity checks passed."
)

Focused HistGB H1 numerical validity checks passed.


### **12G.5.** HistGB Continuation Diagnostic

`focused_histgb_h1` is the controlled training-scope experiment for the
Histogram Gradient Boosting family.

Before testing alternative HistGB hyperparameters, the benchmark checks whether
H1 produces a meaningful positive validation signal.

Further HistGB variants will be considered only if there is evidence that:

- focused HistGB improves upon its historical HistGB control, or
- HistGB is sufficiently competitive with the production champion to justify
  limited parameter exploration.

This prevents unnecessary validation-driven tuning when the base experiment
shows no useful signal.

In [143]:
focused_histgb_h1_overall = (
    benchmark_results[
        "focused_histgb_h1"
    ]["overall_metrics"]
)


historical_histgb_overall = (
    benchmark_results[
        "historical_histgb_hybrid"
    ]["overall_metrics"]
)


focused_histgb_h1_learned = (
    benchmark_results[
        "focused_histgb_h1"
    ]["learned_horizon_metrics"]
)


historical_histgb_learned = (
    benchmark_results[
        "historical_histgb_hybrid"
    ]["learned_horizon_metrics"]
)

In [144]:
focused_histgb_h1_rmse_vs_production = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["rmse"]
        ),
        candidate_value=(
            focused_histgb_h1_overall[
                "rmse"
            ]
        ),
    )
)


focused_histgb_h1_mae_vs_production = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["mae"]
        ),
        candidate_value=(
            focused_histgb_h1_overall[
                "mae"
            ]
        ),
    )
)

In [145]:
focused_histgb_h1_rmse_vs_historical = (
    calculate_improvement_percent(
        champion_value=(
            historical_histgb_overall[
                "rmse"
            ]
        ),
        candidate_value=(
            focused_histgb_h1_overall[
                "rmse"
            ]
        ),
    )
)


focused_histgb_h1_mae_vs_historical = (
    calculate_improvement_percent(
        champion_value=(
            historical_histgb_overall[
                "mae"
            ]
        ),
        candidate_value=(
            focused_histgb_h1_overall[
                "mae"
            ]
        ),
    )
)

In [146]:
focused_histgb_h1_learned_rmse_vs_historical = (
    calculate_improvement_percent(
        champion_value=(
            historical_histgb_learned[
                "rmse"
            ]
        ),
        candidate_value=(
            focused_histgb_h1_learned[
                "rmse"
            ]
        ),
    )
)

In [147]:
focused_histgb_h1_summary = {
    "status": "COMPLETED",
    "candidate": (
        "focused_histgb_h1"
    ),
    "training_scope": "13-72h",
    "iterations": int(
        focused_histgb_h1_model.n_iter_
    ),
    "training_seconds": float(
        focused_histgb_h1_training_seconds
    ),
    "hybrid_mae": float(
        focused_histgb_h1_overall[
            "mae"
        ]
    ),
    "hybrid_rmse": float(
        focused_histgb_h1_overall[
            "rmse"
        ]
    ),
    "hybrid_r2": float(
        focused_histgb_h1_overall[
            "r2"
        ]
    ),
    "rmse_improvement_vs_production_pct": float(
        focused_histgb_h1_rmse_vs_production
    ),
    "mae_improvement_vs_production_pct": float(
        focused_histgb_h1_mae_vs_production
    ),
    "rmse_improvement_vs_historical_histgb_pct": float(
        focused_histgb_h1_rmse_vs_historical
    ),
    "mae_improvement_vs_historical_histgb_pct": float(
        focused_histgb_h1_mae_vs_historical
    ),
    "learned_rmse_improvement_vs_historical_pct": float(
        focused_histgb_h1_learned_rmse_vs_historical
    ),
    "negative_raw_predictions": int(
        focused_histgb_h1_validity[
            "negative_raw_count"
        ]
    ),
    "model_size_mib": float(
        focused_histgb_h1_result[
            "model_size_mib"
        ]
    ),
}


display(
    pd.Series(
        focused_histgb_h1_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12G H1 COMPLETED — focused "
    "HistGradientBoosting has been evaluated."
)

,value
status,COMPLETED
candidate,focused_histgb_h1
training_scope,13-72h
iterations,300
training_seconds,31.001019
hybrid_mae,6.883905
hybrid_rmse,9.421639
hybrid_r2,0.066198
rmse_improvement_vs_production_pct,-0.022145
mae_improvement_vs_production_pct,-2.811722


Phase 12G H1 COMPLETED — focused HistGradientBoosting has been evaluated.


### 12G H1 Result

The controlled Histogram Gradient Boosting challenger has been evaluated.

`focused_histgb_h1` preserved the historical Phase 3 HistGB configuration and
changed only the training scope from 1–72 hours to 13–72 hours.

The candidate has been compared with both:

- the production XGBoost champion
- the historical HistGB hybrid control

Evaluation includes:

- direct 13–72 hour learned-model performance
- complete hybrid validation performance
- 13–24, 25–48, and 49–72 hour stability
- severe-PM2.5 behavior
- prediction validity
- training and inference runtime
- serialized artifact size

No additional HistGB parameters have been tested.

Whether the HistGB family receives further controlled tuning will be determined
from the H1 validation evidence before any additional variants are trained.

The historical test split remains excluded from performance evaluation.

### **12G.6.** HistGB Continuation Gate

The controlled H1 experiment produced a materially different result from the
focused-XGBoost experiment.

Focused HistGB did not beat the production champion overall, but it showed
several useful validation signals:

- hybrid RMSE was effectively tied with production
- focused training improved HistGB RMSE relative to the historical HistGB model
- all learned horizon groups improved in RMSE relative to historical HistGB
- the 49–72 hour group outperformed the production champion on RMSE
- predictions remained numerically valid with no negative outputs

The primary weakness was MAE, which remained worse than production and also
worsened relative to historical HistGB.

This mixed result justifies only a small number of hypothesis-driven variants.

Two additional HistGB configurations will therefore be evaluated:

- **H2:** change the loss to absolute error while keeping structural capacity
  unchanged
- **H3:** retain squared-error loss but moderately increase tree capacity

No broad hyperparameter search will be performed.

In [148]:
histgb_h1_learned_rmse_group_vs_production = (
    histgb_learned_group_improvement_df[
        "h1_rmse_improvement_vs_production_pct"
    ]
    .astype(float)
    .to_numpy()
)

histgb_h1_learned_rmse_group_vs_historical = (
    histgb_learned_group_improvement_df[
        "h1_rmse_improvement_vs_historical_pct"
    ]
    .astype(float)
    .to_numpy()
)


histgb_h1_near_production_rmse = bool(
    abs(
        focused_histgb_h1_rmse_vs_production
    )
    < 0.5
)

histgb_h1_improves_historical_rmse = bool(
    focused_histgb_h1_rmse_vs_historical
    > 0.0
)

histgb_h1_improves_any_production_group = bool(
    (
        histgb_h1_learned_rmse_group_vs_production
        > 0.0
    ).any()
)

histgb_h1_improves_all_historical_groups = bool(
    (
        histgb_h1_learned_rmse_group_vs_historical
        > 0.0
    ).all()
)


continue_histgb_variants = bool(
    histgb_h1_near_production_rmse
    and histgb_h1_improves_historical_rmse
    and (
        histgb_h1_improves_any_production_group
        or histgb_h1_improves_all_historical_groups
    )
)

In [149]:
histgb_continuation_gate_df = pd.DataFrame(
    [
        {
            "criterion": (
                "Hybrid RMSE within 0.5% of production"
            ),
            "passed": (
                histgb_h1_near_production_rmse
            ),
        },
        {
            "criterion": (
                "Focused RMSE improves historical HistGB"
            ),
            "passed": (
                histgb_h1_improves_historical_rmse
            ),
        },
        {
            "criterion": (
                "At least one learned RMSE group beats production"
            ),
            "passed": (
                histgb_h1_improves_any_production_group
            ),
        },
        {
            "criterion": (
                "All learned RMSE groups improve historical HistGB"
            ),
            "passed": (
                histgb_h1_improves_all_historical_groups
            ),
        },
        {
            "criterion": (
                "Continue limited HistGB variants"
            ),
            "passed": (
                continue_histgb_variants
            ),
        },
    ]
)


display(
    histgb_continuation_gate_df
)


assert continue_histgb_variants


print(
    "HistGB continuation gate: CONTINUE"
)

print(
    "Only H2 and H3 will be evaluated."
)

,criterion,passed
0,Hybrid RMSE within 0.5% of production,True
1,Focused RMSE improves historical HistGB,True
2,At least one learned RMSE group beats production,True
3,All learned RMSE groups improve historical HistGB,True
4,Continue limited HistGB variants,True


HistGB continuation gate: CONTINUE
Only H2 and H3 will be evaluated.


### **12G.7.** H2 — Absolute-Error HistGB

H1 nearly matched the production champion on RMSE but produced noticeably worse
MAE.

H2 tests whether that weakness is related to the training objective.

The estimator architecture remains unchanged from H1. The only change is:

`loss="squared_error"` → `loss="absolute_error"`

Training remains restricted to horizons 13–72.

This variant directly tests whether optimizing absolute error can improve the
typical prediction error while preserving the useful horizon behavior observed
in H1.

In [150]:
FOCUSED_HISTGB_H2_PARAMETERS = {
    **FOCUSED_HISTGB_H1_PARAMETERS,
    "loss": "absolute_error",
}


changed_h2_parameters = {
    key: (
        FOCUSED_HISTGB_H1_PARAMETERS[key],
        FOCUSED_HISTGB_H2_PARAMETERS[key],
    )
    for key in FOCUSED_HISTGB_H1_PARAMETERS
    if (
        FOCUSED_HISTGB_H1_PARAMETERS[key]
        != FOCUSED_HISTGB_H2_PARAMETERS[key]
    )
}


assert changed_h2_parameters == {
    "loss": (
        "squared_error",
        "absolute_error",
    )
}


display(
    pd.Series(
        changed_h2_parameters,
        name="H1 → H2",
    ).to_frame()
)

,H1 → H2
loss,"(squared_error, absolute_error)"


In [151]:
focused_histgb_h2_model = (
    HistGradientBoostingRegressor(
        **FOCUSED_HISTGB_H2_PARAMETERS
    )
)


focused_histgb_h2_training_start = (
    perf_counter()
)


focused_histgb_h2_model.fit(
    X_train_focused,
    y_train_focused,
)


focused_histgb_h2_training_seconds = (
    perf_counter()
    - focused_histgb_h2_training_start
)


assert (
    focused_histgb_h2_model.n_features_in_
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    focused_histgb_h2_model.n_iter_
    == FOCUSED_HISTGB_H2_PARAMETERS[
        "max_iter"
    ]
)


print(
    "Focused HistGB H2 training time:",
    f"{focused_histgb_h2_training_seconds:.3f}",
    "seconds",
)

print(
    "Iterations:",
    focused_histgb_h2_model.n_iter_,
)

Focused HistGB H2 training time: 32.943 seconds
Iterations: 300


In [152]:
(
    focused_histgb_h2_raw_predictions,
    focused_histgb_h2_hybrid_predictions,
    focused_histgb_h2_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=focused_histgb_h2_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


focused_histgb_h2_model_size_bytes = (
    measure_serialized_model_size_bytes(
        focused_histgb_h2_model
    )
)


focused_histgb_h2_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "focused_histgb_h2_absolute"
        ),
        model_family=(
            "HistGradientBoostingRegressor"
        ),
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            focused_histgb_h2_raw_predictions
        ),
        operational_predictions=(
            focused_histgb_h2_hybrid_predictions
        ),
        training_seconds=(
            focused_histgb_h2_training_seconds
        ),
        prediction_seconds=(
            focused_histgb_h2_prediction_seconds
        ),
        model_size_bytes=(
            focused_histgb_h2_model_size_bytes
        ),
        parameters=(
            FOCUSED_HISTGB_H2_PARAMETERS
        ),
    )
)


benchmark_results[
    "focused_histgb_h2_absolute"
] = focused_histgb_h2_result

### **12G.8.** H3 — Moderate-Capacity HistGB

H3 keeps the squared-error objective from H1 but moderately increases model
capacity.

The purpose is to test whether the MAE weakness observed in H1 reflects
underfitting after restricting training to the 13–72 hour region.

Only two structural parameters are changed:

- `max_leaf_nodes`: 31 → 47
- `min_samples_leaf`: 30 → 20

Learning rate, number of boosting iterations, regularization, loss function,
random seed, and training scope remain unchanged.

The capacity increase is deliberately moderate to avoid turning the experiment
into an unrestricted complexity search.

In [153]:
FOCUSED_HISTGB_H3_PARAMETERS = {
    **FOCUSED_HISTGB_H1_PARAMETERS,
    "max_leaf_nodes": 47,
    "min_samples_leaf": 20,
}


changed_h3_parameters = {
    key: (
        FOCUSED_HISTGB_H1_PARAMETERS[key],
        FOCUSED_HISTGB_H3_PARAMETERS[key],
    )
    for key in FOCUSED_HISTGB_H1_PARAMETERS
    if (
        FOCUSED_HISTGB_H1_PARAMETERS[key]
        != FOCUSED_HISTGB_H3_PARAMETERS[key]
    )
}


assert changed_h3_parameters == {
    "max_leaf_nodes": (31, 47),
    "min_samples_leaf": (30, 20),
}


display(
    pd.Series(
        changed_h3_parameters,
        name="H1 → H3",
    ).to_frame()
)

,H1 → H3
max_leaf_nodes,"(31, 47)"
min_samples_leaf,"(30, 20)"


In [154]:
focused_histgb_h3_model = (
    HistGradientBoostingRegressor(
        **FOCUSED_HISTGB_H3_PARAMETERS
    )
)


focused_histgb_h3_training_start = (
    perf_counter()
)


focused_histgb_h3_model.fit(
    X_train_focused,
    y_train_focused,
)


focused_histgb_h3_training_seconds = (
    perf_counter()
    - focused_histgb_h3_training_start
)


assert (
    focused_histgb_h3_model.n_features_in_
    == len(MODEL_FEATURE_COLUMNS)
)

assert (
    focused_histgb_h3_model.n_iter_
    == FOCUSED_HISTGB_H3_PARAMETERS[
        "max_iter"
    ]
)


print(
    "Focused HistGB H3 training time:",
    f"{focused_histgb_h3_training_seconds:.3f}",
    "seconds",
)

print(
    "Iterations:",
    focused_histgb_h3_model.n_iter_,
)

Focused HistGB H3 training time: 32.293 seconds
Iterations: 300


In [155]:
(
    focused_histgb_h3_raw_predictions,
    focused_histgb_h3_hybrid_predictions,
    focused_histgb_h3_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=focused_histgb_h3_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)


focused_histgb_h3_model_size_bytes = (
    measure_serialized_model_size_bytes(
        focused_histgb_h3_model
    )
)


focused_histgb_h3_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "focused_histgb_h3_capacity"
        ),
        model_family=(
            "HistGradientBoostingRegressor"
        ),
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            focused_histgb_h3_raw_predictions
        ),
        operational_predictions=(
            focused_histgb_h3_hybrid_predictions
        ),
        training_seconds=(
            focused_histgb_h3_training_seconds
        ),
        prediction_seconds=(
            focused_histgb_h3_prediction_seconds
        ),
        model_size_bytes=(
            focused_histgb_h3_model_size_bytes
        ),
        parameters=(
            FOCUSED_HISTGB_H3_PARAMETERS
        ),
    )
)


benchmark_results[
    "focused_histgb_h3_capacity"
] = focused_histgb_h3_result

### **12G.9.** Final HistGB Family Comparison

The limited HistGB exploration is complete.

The benchmark now compares:

- production champion version 1
- historical HistGB hybrid
- focused H1 baseline
- focused H2 absolute-error variant
- focused H3 moderate-capacity variant

No further HistGB configurations will be trained after this comparison.

The objective is to identify whether any HistGB configuration provides enough
validation evidence to remain a serious candidate for the later Stage-1
leaderboard.

In [156]:
HISTGB_RESULT_KEYS = [
    "production_champion_v1",
    "historical_histgb_hybrid",
    "focused_histgb_h1",
    "focused_histgb_h2_absolute",
    "focused_histgb_h3_capacity",
]


histgb_family_leaderboard_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[key]
            )
            for key in HISTGB_RESULT_KEYS
        ]
    )
)

In [157]:
histgb_family_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = histgb_family_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


histgb_family_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = histgb_family_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


histgb_family_leaderboard_df = (
    histgb_family_leaderboard_df
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)

In [158]:
display(
    histgb_family_leaderboard_df[
        [
            "candidate",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,focused_histgb_h2_absolute,6.584561,9.306569,0.088868,7.044038,9.581097,1.199460,1.659011,0,32.942681,0.731072,1.096167
1,production_champion_v1,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,0,NaN,0.394257,0.645863
2,focused_histgb_h1,6.883905,9.421639,0.066198,7.405951,9.716116,-0.022145,-2.811722,0,31.001019,0.888559,1.096167
3,focused_histgb_h3_capacity,6.904288,9.550370,0.040506,7.430594,9.866899,-1.388784,-3.116135,0,32.292801,1.072810,1.608862
4,historical_histgb_hybrid,6.762337,9.579777,0.034588,7.258972,9.901305,-1.700974,-0.996086,0,30.336777,1.129033,1.096182


In [159]:
histgb_variant_horizon_df = pd.concat(
    [
        benchmark_results[key][
            "horizon_group_metrics"
        ]
        for key in (
            "production_champion_v1",
            "focused_histgb_h1",
            "focused_histgb_h2_absolute",
            "focused_histgb_h3_capacity",
        )
    ],
    ignore_index=True,
)


display(
    histgb_variant_horizon_df.loc[
        histgb_variant_horizon_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
12,focused_histgb_h2_absolute,13-24h,13,24,12163,6.833065,9.411421,0.143644
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
7,focused_histgb_h1,13-24h,13,24,12163,7.381548,9.730627,0.084569
17,focused_histgb_h3_capacity,13-24h,13,24,12163,7.314944,9.770060,0.077135
13,focused_histgb_h2_absolute,25-48h,25,48,23709,7.044120,9.569383,-0.028490
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
8,focused_histgb_h1,25-48h,25,48,23709,7.475623,9.754224,-0.068606
18,focused_histgb_h3_capacity,25-48h,25,48,23709,7.473077,9.869705,-0.094058
9,focused_histgb_h1,49-72h,49,72,23065,7.347203,9.669097,-0.009982
14,focused_histgb_h2_absolute,49-72h,49,72,23065,7.155207,9.681291,-0.012531


In [160]:
histgb_variant_validity_df = pd.DataFrame(
    [
        {
            "candidate": key,
            **benchmark_results[key][
                "prediction_validity"
            ],
        }
        for key in (
            "focused_histgb_h1",
            "focused_histgb_h2_absolute",
            "focused_histgb_h3_capacity",
        )
    ]
)


display(
    histgb_variant_validity_df
)


assert (
    histgb_variant_validity_df[
        "raw_nan_count"
    ]
    == 0
).all()

assert (
    histgb_variant_validity_df[
        "raw_inf_count"
    ]
    == 0
).all()

,candidate,prediction_count,raw_nan_count,raw_inf_count,negative_raw_count,clipped_prediction_count,raw_min,raw_max,operational_min,operational_max
0,focused_histgb_h1,71256,0,0,0,0,0.1,88.9,0.1,88.9
1,focused_histgb_h2_absolute,71256,0,0,0,0,0.1,88.9,0.1,88.9
2,focused_histgb_h3_capacity,71256,0,0,0,0,0.1,88.9,0.1,88.9


### **12G.10.** HistGB Family Decision

The limited Histogram Gradient Boosting exploration is complete.

Three focused configurations were evaluated:

- H1: historical squared-error configuration trained on horizons 13–72
- H2: absolute-error objective with H1 structural parameters unchanged
- H3: squared-error objective with moderately increased model capacity

H2 produced the strongest HistGB result.

Unlike H1 and H3, H2 improved both MAE and RMSE relative to the production
champion while remaining numerically valid.

Its improvement was also distributed across all learned-model horizon groups
rather than being driven by a single forecast range.

No additional HistGB configurations will be trained.

H2 will remain in the benchmark as the representative HistGB challenger for
later Stage-1 and robustness comparisons.

In [162]:
BEST_HISTGB_CANDIDATE = (
    "focused_histgb_h2_absolute"
)

best_histgb_result = benchmark_results[
    BEST_HISTGB_CANDIDATE
]

best_histgb_overall = best_histgb_result[
    "overall_metrics"
]

best_histgb_learned = best_histgb_result[
    "learned_horizon_metrics"
]

In [163]:
best_histgb_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["rmse"]
        ),
        candidate_value=(
            best_histgb_overall["rmse"]
        ),
    )
)


best_histgb_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["mae"]
        ),
        candidate_value=(
            best_histgb_overall["mae"]
        ),
    )
)


best_histgb_learned_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_learned["rmse"]
        ),
        candidate_value=(
            best_histgb_learned["rmse"]
        ),
    )
)


best_histgb_learned_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_learned["mae"]
        ),
        candidate_value=(
            best_histgb_learned["mae"]
        ),
    )
)

In [164]:
best_histgb_group_metrics = (
    best_histgb_result[
        "horizon_group_metrics"
    ]
)


production_group_metrics = (
    benchmark_results[
        "production_champion_v1"
    ][
        "horizon_group_metrics"
    ]
)


learned_group_decision_records = []


for group_name in (
    "13-24h",
    "25-48h",
    "49-72h",
):
    production_group = (
        production_group_metrics.loc[
            production_group_metrics[
                "horizon_group"
            ].eq(group_name)
        ]
        .iloc[0]
    )

    candidate_group = (
        best_histgb_group_metrics.loc[
            best_histgb_group_metrics[
                "horizon_group"
            ].eq(group_name)
        ]
        .iloc[0]
    )

    learned_group_decision_records.append(
        {
            "horizon_group": group_name,
            "rmse_improved": bool(
                candidate_group["rmse"]
                < production_group["rmse"]
            ),
            "mae_improved": bool(
                candidate_group["mae"]
                < production_group["mae"]
            ),
        }
    )


best_histgb_group_decision_df = (
    pd.DataFrame(
        learned_group_decision_records
    )
)


display(
    best_histgb_group_decision_df
)

,horizon_group,rmse_improved,mae_improved
0,13-24h,True,True
1,25-48h,True,True
2,49-72h,True,True


In [165]:
assert (
    best_histgb_group_decision_df[
        "rmse_improved"
    ]
).all()

assert (
    best_histgb_group_decision_df[
        "mae_improved"
    ]
).all()


print(
    "HistGB H2 improves both MAE and RMSE "
    "across every learned horizon group."
)

HistGB H2 improves both MAE and RMSE across every learned horizon group.


In [166]:
histgb_family_summary = {
    "status": "CLOSED",
    "variants_evaluated": [
        "focused_histgb_h1",
        "focused_histgb_h2_absolute",
        "focused_histgb_h3_capacity",
    ],
    "selected_family_representative": (
        BEST_HISTGB_CANDIDATE
    ),
    "hybrid_mae": float(
        best_histgb_overall["mae"]
    ),
    "hybrid_rmse": float(
        best_histgb_overall["rmse"]
    ),
    "hybrid_r2": float(
        best_histgb_overall["r2"]
    ),
    "rmse_improvement_vs_production_pct": float(
        best_histgb_rmse_improvement
    ),
    "mae_improvement_vs_production_pct": float(
        best_histgb_mae_improvement
    ),
    "learned_13_72_rmse": float(
        best_histgb_learned["rmse"]
    ),
    "learned_13_72_mae": float(
        best_histgb_learned["mae"]
    ),
    "learned_rmse_improvement_pct": float(
        best_histgb_learned_rmse_improvement
    ),
    "learned_mae_improvement_pct": float(
        best_histgb_learned_mae_improvement
    ),
    "all_learned_rmse_groups_improved": bool(
        best_histgb_group_decision_df[
            "rmse_improved"
        ].all()
    ),
    "all_learned_mae_groups_improved": bool(
        best_histgb_group_decision_df[
            "mae_improved"
        ].all()
    ),
    "negative_raw_predictions": int(
        best_histgb_result[
            "prediction_validity"
        ][
            "negative_raw_count"
        ]
    ),
    "model_size_mib": float(
        best_histgb_result[
            "model_size_mib"
        ]
    ),
    "family_decision": (
        "RETAIN_H2_FOR_STAGE_1"
    ),
}


display(
    pd.Series(
        histgb_family_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12G PASSED — HistGB family "
    "exploration is closed."
)

print(
    "focused_histgb_h2_absolute will "
    "advance to the Stage-1 comparison."
)

,value
status,CLOSED
variants_evaluated,"[focused_histgb_h1, focused_histgb_h2_absolute..."
selected_family_representative,focused_histgb_h2_absolute
hybrid_mae,6.584561
hybrid_rmse,9.306569
hybrid_r2,0.088868
rmse_improvement_vs_production_pct,1.19946
mae_improvement_vs_production_pct,1.659011
learned_13_72_rmse,9.581097
learned_13_72_mae,7.044038


Phase 12G PASSED — HistGB family exploration is closed.
focused_histgb_h2_absolute will advance to the Stage-1 comparison.


### 12G Result

The Histogram Gradient Boosting family produced the first challenger in this
benchmark to outperform production model version 1 on both primary validation
metrics.

The strongest configuration was `focused_histgb_h2_absolute`.

Compared with the production champion, H2 achieved approximately:

- 1.20% lower full-hybrid RMSE
- 1.66% lower full-hybrid MAE

The improvement was not concentrated in a single forecast range. H2 improved
both MAE and RMSE across all learned-model horizon groups:

- 13–24 hours
- 25–48 hours
- 49–72 hours

H2 also remained numerically valid, with no negative, NaN, infinite, or clipped
predictions.

The absolute-error objective therefore addressed the MAE weakness observed in
H1 while also improving RMSE.

The higher-capacity H3 variant did not improve performance, so additional
HistGB complexity is not justified.

No further HistGB configurations will be trained.

`focused_histgb_h2_absolute` is retained as the representative HistGB
challenger for the later Stage-1 leaderboard and robustness analysis.

Its approximately 1.20% RMSE improvement is promising but remains below the
benchmark's provisional threshold for replacing a mature production model.
No production decision is made at this stage.

The historical test split remains untouched by challenger evaluation.

## **12H.** Random Forest Challenger

Histogram Gradient Boosting produced a credible Stage-1 challenger, with the
absolute-error H2 configuration improving both validation MAE and RMSE relative
to production.

The benchmark now evaluates a structurally different estimator family.

Random Forest uses bagging rather than sequential boosting and therefore tests
whether an ensemble of independently fitted decision trees provides useful
robustness for the existing PM2.5 feature representation.

The initial Random Forest challenger is intentionally regularized to control
training cost, serialized model size, and overfitting.

Training is restricted to horizons 13–72 because the learned estimator is not
used for production horizons 1–12.

Final evaluation continues to use the production hybrid contract:

- hours 1–12: current-value persistence
- hours 13–72: Random Forest

Only one configuration is evaluated initially.

A second Random Forest configuration will be considered only if the first
candidate produces a meaningful validation signal.

The historical test split remains excluded from performance evaluation.

In [167]:
from sklearn.ensemble import (
    RandomForestRegressor,
)

In [168]:
RANDOM_FOREST_R1_PARAMETERS = {
    "n_estimators": 250,
    "max_depth": 18,
    "min_samples_split": 10,
    "min_samples_leaf": 5,
    "max_features": 0.7,
    "bootstrap": True,
    "n_jobs": -1,
    "random_state": 42,
}

In [169]:
random_forest_r1_model = (
    RandomForestRegressor(
        **RANDOM_FOREST_R1_PARAMETERS
    )
)


random_forest_r1_model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.7
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"ma

In [170]:
random_forest_r1_training_start = (
    perf_counter()
)


random_forest_r1_model.fit(
    X_train_focused,
    y_train_focused,
)


random_forest_r1_training_seconds = (
    perf_counter()
    - random_forest_r1_training_start
)


assert (
    random_forest_r1_model.n_features_in_
    == len(MODEL_FEATURE_COLUMNS)
)


print(
    "Random Forest R1 training time:",
    f"{random_forest_r1_training_seconds:.3f}",
    "seconds",
)

print(
    "Trees fitted:",
    len(
        random_forest_r1_model.estimators_
    ),
)

Random Forest R1 training time: 713.104 seconds
Trees fitted: 250


### **12H.1.** Learned-Region Comparison

Random Forest R1 is first evaluated directly on the 13–72 hour validation
region.

The learned-estimator comparison includes:

- production XGBoost
- retained HistGB H2 challenger
- Random Forest R1

This isolates model-family performance before the common persistence component
is applied.

In [171]:
random_forest_r1_learned_prediction_start = (
    perf_counter()
)


random_forest_r1_learned_predictions = (
    random_forest_r1_model.predict(
        X_validation_focused
    )
)


random_forest_r1_learned_prediction_seconds = (
    perf_counter()
    - random_forest_r1_learned_prediction_start
)


assert np.isfinite(
    random_forest_r1_learned_predictions
).all()


random_forest_r1_learned_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            random_forest_r1_learned_predictions
        ),
    )
)

In [172]:
random_forest_learned_comparison_df = (
    pd.DataFrame(
        [
            {
                "candidate": (
                    "production_xgboost_shallower"
                ),
                **production_xgb_focused_metrics,
            },
            {
                "candidate": (
                    "focused_histgb_h2_absolute"
                ),
                **best_histgb_learned,
            },
            {
                "candidate": (
                    "random_forest_r1"
                ),
                **random_forest_r1_learned_metrics,
            },
        ]
    )
    .sort_values(
        "rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    random_forest_learned_comparison_df
)

,candidate,mae,rmse,r2
0,focused_histgb_h2_absolute,7.044038,9.581097,0.018325
1,production_xgboost_shallower,7.178338,9.713671,-0.009030
2,random_forest_r1,8.174561,10.744326,-0.234513


### **12H.2.** Full Hybrid Validation Evaluation

Random Forest R1 is now evaluated under the complete production routing
contract.

The resulting strategy is compared with:

- production champion version 1
- retained HistGB H2 challenger

This determines whether Random Forest deserves further model-family
exploration.

In [173]:
(
    random_forest_r1_raw_predictions,
    random_forest_r1_hybrid_predictions,
    random_forest_r1_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=random_forest_r1_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [174]:
random_forest_r1_model_size_bytes = (
    measure_serialized_model_size_bytes(
        random_forest_r1_model
    )
)


print(
    "Random Forest R1 artifact size:",
    f"{random_forest_r1_model_size_bytes / (1024 ** 2):.3f}",
    "MiB",
)

Random Forest R1 artifact size: 299.362 MiB


In [175]:
random_forest_r1_result = (
    evaluate_candidate_predictions(
        candidate_name="random_forest_r1",
        model_family=(
            "RandomForestRegressor"
        ),
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            random_forest_r1_raw_predictions
        ),
        operational_predictions=(
            random_forest_r1_hybrid_predictions
        ),
        training_seconds=(
            random_forest_r1_training_seconds
        ),
        prediction_seconds=(
            random_forest_r1_prediction_seconds
        ),
        model_size_bytes=(
            random_forest_r1_model_size_bytes
        ),
        parameters=(
            RANDOM_FOREST_R1_PARAMETERS
        ),
    )
)


benchmark_results[
    "random_forest_r1"
] = random_forest_r1_result

In [176]:
random_forest_r1_leaderboard_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "focused_histgb_h2_absolute"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "random_forest_r1"
                ]
            ),
        ]
    )
)

In [177]:
random_forest_r1_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = random_forest_r1_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


random_forest_r1_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = random_forest_r1_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


random_forest_r1_leaderboard_df = (
    random_forest_r1_leaderboard_df
    .sort_values(
        "hybrid_rmse"
    )
    .reset_index(
        drop=True
    )
)


display(
    random_forest_r1_leaderboard_df
)

,candidate,model_family,training_scope,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,learned_13_72_r2,negative_raw_predictions,clipped_predictions,training_seconds,prediction_seconds,model_size_mib,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct
0,focused_histgb_h2_absolute,HistGradientBoostingRegressor,13-72h,6.584561,9.306569,0.088868,7.044038,9.581097,0.018325,0,0,32.942681,0.731072,1.096167,1.199460,1.659011
1,production_champion_v1,XGBRegressor,historical production artifact,6.695642,9.419553,0.066611,7.178338,9.713671,-0.009030,0,0,NaN,0.394257,0.645863,0.000000,0.000000
2,random_forest_r1,RandomForestRegressor,13-72h,7.519635,10.303778,-0.116851,8.174561,10.744326,-0.234513,0,0,713.104323,1.257591,299.362412,-9.387129,-12.306399


In [178]:
random_forest_horizon_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "production_champion_v1"
            ]["horizon_group_metrics"],
            benchmark_results[
                "focused_histgb_h2_absolute"
            ]["horizon_group_metrics"],
            benchmark_results[
                "random_forest_r1"
            ]["horizon_group_metrics"],
        ],
        ignore_index=True,
    )
)


display(
    random_forest_horizon_comparison_df.loc[
        random_forest_horizon_comparison_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
7,focused_histgb_h2_absolute,13-24h,13,24,12163,6.833065,9.411421,0.143644
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
12,random_forest_r1,13-24h,13,24,12163,8.111692,10.685547,-0.103919
8,focused_histgb_h2_absolute,25-48h,25,48,23709,7.044120,9.569383,-0.028490
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
13,random_forest_r1,25-48h,25,48,23709,8.247586,10.752046,-0.298418
9,focused_histgb_h2_absolute,49-72h,49,72,23065,7.155207,9.681291,-0.012531
4,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042
14,random_forest_r1,49-72h,49,72,23065,8.132650,10.767275,-0.252430


In [179]:
random_forest_r1_validity = (
    random_forest_r1_result[
        "prediction_validity"
    ]
)


display(
    pd.Series(
        random_forest_r1_validity,
        name="value",
    ).to_frame()
)


assert (
    random_forest_r1_validity[
        "raw_nan_count"
    ]
    == 0
)

assert (
    random_forest_r1_validity[
        "raw_inf_count"
    ]
    == 0
)

,value
prediction_count,71256.0
raw_nan_count,0.0
raw_inf_count,0.0
negative_raw_count,0.0
clipped_prediction_count,0.0
raw_min,0.1
raw_max,88.9
operational_min,0.1
operational_max,88.9


In [180]:
random_forest_r1_overall = (
    random_forest_r1_result[
        "overall_metrics"
    ]
)


random_forest_r1_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["rmse"]
        ),
        candidate_value=(
            random_forest_r1_overall[
                "rmse"
            ]
        ),
    )
)


random_forest_r1_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_overall["mae"]
        ),
        candidate_value=(
            random_forest_r1_overall[
                "mae"
            ]
        ),
    )
)


random_forest_r1_summary = {
    "status": "COMPLETED",
    "candidate": "random_forest_r1",
    "trees": len(
        random_forest_r1_model.estimators_
    ),
    "training_seconds": float(
        random_forest_r1_training_seconds
    ),
    "hybrid_mae": float(
        random_forest_r1_overall[
            "mae"
        ]
    ),
    "hybrid_rmse": float(
        random_forest_r1_overall[
            "rmse"
        ]
    ),
    "hybrid_r2": float(
        random_forest_r1_overall[
            "r2"
        ]
    ),
    "rmse_improvement_vs_production_pct": float(
        random_forest_r1_rmse_improvement
    ),
    "mae_improvement_vs_production_pct": float(
        random_forest_r1_mae_improvement
    ),
    "negative_raw_predictions": int(
        random_forest_r1_validity[
            "negative_raw_count"
        ]
    ),
    "model_size_mib": float(
        random_forest_r1_result[
            "model_size_mib"
        ]
    ),
}


display(
    pd.Series(
        random_forest_r1_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12H R1 COMPLETED — Random "
    "Forest has been evaluated."
)

,value
status,COMPLETED
candidate,random_forest_r1
trees,250
training_seconds,713.104323
hybrid_mae,7.519635
hybrid_rmse,10.303778
hybrid_r2,-0.116851
rmse_improvement_vs_production_pct,-9.387129
mae_improvement_vs_production_pct,-12.306399
negative_raw_predictions,0


Phase 12H R1 COMPLETED — Random Forest has been evaluated.


### **12H.5.** Random Forest Family Decision

Random Forest R1 provided no validation evidence that would justify further
experimentation with the model family.

Compared with production model version 1, R1 produced substantially worse MAE
and RMSE across the full hybrid forecast.

The regression was also consistent across every learned-model horizon group:

- 13–24 hours
- 25–48 hours
- 49–72 hours

In addition to lower predictive performance, Random Forest required
substantially more training time and produced a serialized artifact hundreds of
times larger than the boosting-based candidates.

Prediction validity remained clean, but numerical validity alone is not
sufficient to justify additional model-family exploration.

No second Random Forest configuration will be trained.

The Random Forest branch is therefore closed and will not advance to the
Stage-1 finalist set.

In [181]:
random_forest_has_primary_signal = bool(
    random_forest_r1_rmse_improvement > 0.0
    or random_forest_r1_mae_improvement > 0.0
)


random_forest_group_metrics = (
    benchmark_results[
        "random_forest_r1"
    ]["horizon_group_metrics"]
)


random_forest_beats_production_group = []


for group_name in (
    "13-24h",
    "25-48h",
    "49-72h",
):
    production_group = (
        benchmark_results[
            "production_champion_v1"
        ]["horizon_group_metrics"]
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    random_forest_group = (
        random_forest_group_metrics
        .loc[
            lambda dataframe: (
                dataframe[
                    "horizon_group"
                ].eq(group_name)
            )
        ]
        .iloc[0]
    )

    random_forest_beats_production_group.append(
        bool(
            random_forest_group["rmse"]
            < production_group["rmse"]
        )
    )


random_forest_has_horizon_signal = bool(
    any(
        random_forest_beats_production_group
    )
)


continue_random_forest = bool(
    random_forest_has_primary_signal
    or random_forest_has_horizon_signal
)

In [182]:
random_forest_gate_df = pd.DataFrame(
    [
        {
            "criterion": (
                "Positive overall MAE/RMSE signal"
            ),
            "passed": (
                random_forest_has_primary_signal
            ),
        },
        {
            "criterion": (
                "At least one learned horizon "
                "group beats production RMSE"
            ),
            "passed": (
                random_forest_has_horizon_signal
            ),
        },
        {
            "criterion": (
                "Continue Random Forest variants"
            ),
            "passed": (
                continue_random_forest
            ),
        },
    ]
)


display(
    random_forest_gate_df
)

,criterion,passed
0,Positive overall MAE/RMSE signal,False
1,At least one learned horizon group beats produ...,False
2,Continue Random Forest variants,False


In [183]:
random_forest_family_summary = {
    "status": "CLOSED",
    "candidate": (
        "random_forest_r1"
    ),
    "hybrid_mae": float(
        random_forest_r1_overall[
            "mae"
        ]
    ),
    "hybrid_rmse": float(
        random_forest_r1_overall[
            "rmse"
        ]
    ),
    "rmse_improvement_vs_production_pct": float(
        random_forest_r1_rmse_improvement
    ),
    "mae_improvement_vs_production_pct": float(
        random_forest_r1_mae_improvement
    ),
    "learned_horizon_groups_beating_production": int(
        sum(
            random_forest_beats_production_group
        )
    ),
    "training_seconds": float(
        random_forest_r1_training_seconds
    ),
    "model_size_mib": float(
        random_forest_r1_result[
            "model_size_mib"
        ]
    ),
    "negative_raw_predictions": int(
        random_forest_r1_validity[
            "negative_raw_count"
        ]
    ),
    "continue_to_rf_variants": (
        continue_random_forest
    ),
    "family_decision": (
        "STOP_RANDOM_FOREST"
    ),
}


display(
    pd.Series(
        random_forest_family_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12H PASSED — Random Forest "
    "family exploration is closed."
)

,value
status,CLOSED
candidate,random_forest_r1
hybrid_mae,7.519635
hybrid_rmse,10.303778
rmse_improvement_vs_production_pct,-9.387129
mae_improvement_vs_production_pct,-12.306399
learned_horizon_groups_beating_production,0
training_seconds,713.104323
model_size_mib,299.362412
negative_raw_predictions,0


Phase 12H PASSED — Random Forest family exploration is closed.


### 12H Result

The Random Forest family is closed after the R1 experiment.

Random Forest R1 underperformed production model version 1 by approximately:

- 9.39% on full-hybrid RMSE
- 12.31% on full-hybrid MAE

The model also produced worse RMSE across every learned-model horizon group.

There was therefore no accuracy signal suggesting that a second Random Forest
configuration would be justified.

Operationally, R1 was also substantially more expensive than the boosting
models. Training required approximately 713 seconds and the serialized model
was approximately 299 MiB, compared with roughly 1 MiB or less for the leading
boosting candidates.

The model remained numerically valid with no negative, NaN, infinite, or
clipped predictions, but this does not offset its accuracy and operational
disadvantages.

No additional Random Forest variants will be trained.

Random Forest will not advance to the Stage-1 finalist set.

The historical test split remains excluded from challenger evaluation.

## **12I.** Stage-1 Validation Leaderboard

The first model-family benchmark stage is complete.

Stage 1 evaluated the existing production champion alongside controlled
challengers from three estimator families:

- XGBoost
- Histogram Gradient Boosting
- Random Forest

Only validation results are considered here.

For model families with multiple configurations, only the strongest
representative is retained in the main leaderboard. Experimental variants that
were clearly superseded or rejected remain part of the notebook history but do
not receive equal weight in the finalist comparison.

The purpose of this phase is to determine whether the current candidates provide
enough evidence to stop model-family exploration or whether one external
gradient-boosting challenger is justified.

In [184]:
STAGE_1_CANDIDATES = {
    "production_champion_v1": (
        "Production XGBoost"
    ),
    "focused_histgb_h2_absolute": (
        "HistGB H2"
    ),
    "random_forest_r1": (
        "Random Forest R1"
    ),
}

In [185]:
stage_1_leaderboard_df = pd.DataFrame(
    [
        build_leaderboard_row(
            benchmark_results[key]
        )
        | {
            "display_name": label,
        }
        for key, label
        in STAGE_1_CANDIDATES.items()
    ]
)

In [186]:
stage_1_leaderboard_df[
    "rmse_improvement_pct"
] = stage_1_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=champion_hybrid_rmse,
            candidate_value=float(value),
        )
    )
)

stage_1_leaderboard_df[
    "mae_improvement_pct"
] = stage_1_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=champion_hybrid_mae,
            candidate_value=float(value),
        )
    )
)

In [187]:
STAGE_1_STATUS = {
    "production_champion_v1": "CHAMPION",
    "focused_histgb_h2_absolute": "ADVANCE",
    "random_forest_r1": "REJECT",
}


stage_1_leaderboard_df[
    "status"
] = stage_1_leaderboard_df[
    "candidate"
].map(
    STAGE_1_STATUS
)

In [188]:
stage_1_leaderboard_df = (
    stage_1_leaderboard_df
    .sort_values("hybrid_rmse")
    .reset_index(drop=True)
)

In [189]:
display(
    stage_1_leaderboard_df[
        [
            "display_name",
            "status",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_pct",
            "mae_improvement_pct",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,display_name,status,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_pct,mae_improvement_pct,training_seconds,prediction_seconds,model_size_mib
0,HistGB H2,ADVANCE,6.584561,9.306569,0.088868,7.044038,9.581097,1.199460,1.659011,32.942681,0.731072,1.096167
1,Production XGBoost,CHAMPION,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,NaN,0.394257,0.645863
2,Random Forest R1,REJECT,7.519635,10.303778,-0.116851,8.174561,10.744326,-9.387129,-12.306399,713.104323,1.257591,299.362412


In [190]:
stage_1_horizon_df = pd.concat(
    [
        benchmark_results[
            "production_champion_v1"
        ]["horizon_group_metrics"],
        benchmark_results[
            "focused_histgb_h2_absolute"
        ]["horizon_group_metrics"],
    ],
    ignore_index=True,
)


display(
    stage_1_horizon_df.loc[
        stage_1_horizon_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
4,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042
7,focused_histgb_h2_absolute,13-24h,13,24,12163,6.833065,9.411421,0.143644
8,focused_histgb_h2_absolute,25-48h,25,48,23709,7.044120,9.569383,-0.028490
9,focused_histgb_h2_absolute,49-72h,49,72,23065,7.155207,9.681291,-0.012531


In [191]:
stage_1_severe_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=(
                "production_champion_v1"
            ),
            severe_result=(
                benchmark_results[
                    "production_champion_v1"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                "focused_histgb_h2_absolute"
            ),
            severe_result=(
                benchmark_results[
                    "focused_histgb_h2_absolute"
                ]["severe_pm25"]
            ),
        ),
    ]
)


display(stage_1_severe_df)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,production_champion_v1,55.5,181,49.371822,50.153180,-52.45170
1,focused_histgb_h2_absolute,55.5,181,44.340729,46.258939,-44.47324


In [192]:
stage_1_best_challenger = (
    benchmark_results[
        "focused_histgb_h2_absolute"
    ]
)


stage_1_best_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            benchmark_results[
                "production_champion_v1"
            ]["overall_metrics"]["rmse"]
        ),
        candidate_value=(
            stage_1_best_challenger[
                "overall_metrics"
            ]["rmse"]
        ),
    )
)


stage_1_best_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            benchmark_results[
                "production_champion_v1"
            ]["overall_metrics"]["mae"]
        ),
        candidate_value=(
            stage_1_best_challenger[
                "overall_metrics"
            ]["mae"]
        ),
    )
)

In [193]:
stage_1_summary = {
    "best_challenger": (
        "focused_histgb_h2_absolute"
    ),
    "rmse_improvement_pct": (
        stage_1_best_rmse_improvement
    ),
    "mae_improvement_pct": (
        stage_1_best_mae_improvement
    ),
    "random_forest_status": "REJECTED",
    "external_challenger_recommended": True,
}


display(
    pd.Series(
        stage_1_summary,
        name="value",
    ).to_frame()
)

,value
best_challenger,focused_histgb_h2_absolute
rmse_improvement_pct,1.19946
mae_improvement_pct,1.659011
random_forest_status,REJECTED
external_challenger_recommended,True


### 12I Result

Stage 1 produced one credible challenger.

`focused_histgb_h2_absolute` achieved the strongest validation result so far,
improving upon production model version 1 by approximately:

- 1.20% on hybrid RMSE
- 1.66% on hybrid MAE

Its improvement was also consistent across the learned 13–72 hour horizon
groups.

Random Forest was decisively rejected because it produced substantially worse
validation performance while requiring much greater training time and artifact
size.

HistGB H2 therefore advances as the current leading challenger.

However, its improvement remains modest relative to the cost and risk of
replacing an already deployed production model.

The benchmark will therefore evaluate one additional external tabular boosting
family before model-family exploration is closed.

No historical test results have been used for this decision.


## **12J.** External Challenger Gate

Stage 1 produced one credible challenger.

`focused_histgb_h2_absolute` currently leads the validation benchmark, improving
upon production model version 1 on:

- overall hybrid MAE
- overall hybrid RMSE
- all learned-model horizon groups
- severe-PM2.5 performance

However, the overall RMSE improvement is approximately 1.20%.

That result is promising, but still modest relative to the cost and risk of
replacing a mature production model.

The benchmark will therefore evaluate one additional external tabular boosting
family before model-family exploration is closed.

### External challenger selected: LightGBM

LightGBM is chosen because the current problem is already represented as a
large numerical tabular dataset with engineered temporal, PM2.5, weather, and
forecast-horizon features.

The benchmark will not introduce:

- deep-learning sequence models
- a new temporal dataset representation
- CatBoost
- broad hyperparameter optimization

LightGBM will use the existing frozen feature contract and the same 13–72 hour
learned-model training scope used by the strongest challengers.

The historical test split remains excluded from model development.

In [194]:
import lightgbm as lgb

from lightgbm import LGBMRegressor


print(
    "LightGBM version:",
    lgb.__version__,
)

LightGBM version: 4.7.0


In [195]:
LIGHTGBM_L1_PARAMETERS = {
    "objective": "regression",
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 30,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.2,
    "reg_lambda": 2.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

### **12J.2.** LightGBM L1

The first LightGBM challenger uses a moderate regularized configuration.

Training is restricted to horizons 13–72.

Unlike HistGradientBoosting, LightGBM supports explicit external validation
during training. The frozen 13–72 validation subset is therefore used for
early stopping.

The complete hybrid evaluation still uses persistence for hours 1–12.

In [196]:
lightgbm_l1_model = LGBMRegressor(
    **LIGHTGBM_L1_PARAMETERS
)


lightgbm_l1_training_start = (
    perf_counter()
)


lightgbm_l1_model.fit(
    X_train_focused,
    y_train_focused,
    eval_set=[
        (
            X_validation_focused,
            y_validation_focused,
        )
    ],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=75,
            verbose=False,
        )
    ],
)


lightgbm_l1_training_seconds = (
    perf_counter()
    - lightgbm_l1_training_start
)


print(
    "LightGBM L1 training time:",
    f"{lightgbm_l1_training_seconds:.3f}",
    "seconds",
)

print(
    "Best iteration:",
    lightgbm_l1_model.best_iteration_,
)

LightGBM L1 training time: 80.805 seconds
Best iteration: 397


In [197]:
lightgbm_l1_learned_predictions = (
    lightgbm_l1_model.predict(
        X_validation_focused,
        num_iteration=(
            lightgbm_l1_model.best_iteration_
        ),
    )
)


lightgbm_l1_learned_metrics = (
    calculate_regression_metrics(
        y_true=y_validation_focused,
        y_pred=(
            lightgbm_l1_learned_predictions
        ),
    )
)


lightgbm_learned_comparison_df = (
    pd.DataFrame(
        [
            {
                "candidate": (
                    "focused_histgb_h2_absolute"
                ),
                **best_histgb_learned,
            },
            {
                "candidate": (
                    "production_xgboost_shallower"
                ),
                **production_xgb_focused_metrics,
            },
            {
                "candidate": (
                    "lightgbm_l1"
                ),
                **lightgbm_l1_learned_metrics,
            },
        ]
    )
    .sort_values("rmse")
    .reset_index(drop=True)
)


display(
    lightgbm_learned_comparison_df
)

,candidate,mae,rmse,r2
0,lightgbm_l1,7.203784,9.577872,0.018985
1,focused_histgb_h2_absolute,7.044038,9.581097,0.018325
2,production_xgboost_shallower,7.178338,9.713671,-0.009030


In [198]:
(
    lightgbm_l1_raw_predictions,
    lightgbm_l1_hybrid_predictions,
    lightgbm_l1_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=lightgbm_l1_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [199]:
lightgbm_l1_model_size_bytes = (
    measure_serialized_model_size_bytes(
        lightgbm_l1_model
    )
)


print(
    "LightGBM L1 artifact size:",
    f"{lightgbm_l1_model_size_bytes / (1024 ** 2):.3f}",
    "MiB",
)

LightGBM L1 artifact size: 1.127 MiB


In [200]:
lightgbm_l1_result = (
    evaluate_candidate_predictions(
        candidate_name="lightgbm_l1",
        model_family="LGBMRegressor",
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            lightgbm_l1_raw_predictions
        ),
        operational_predictions=(
            lightgbm_l1_hybrid_predictions
        ),
        training_seconds=(
            lightgbm_l1_training_seconds
        ),
        prediction_seconds=(
            lightgbm_l1_prediction_seconds
        ),
        model_size_bytes=(
            lightgbm_l1_model_size_bytes
        ),
        parameters=(
            LIGHTGBM_L1_PARAMETERS
        ),
    )
)


benchmark_results[
    "lightgbm_l1"
] = lightgbm_l1_result

In [201]:
external_challenger_leaderboard_df = (
    pd.DataFrame(
        [
            build_leaderboard_row(
                benchmark_results[
                    "focused_histgb_h2_absolute"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "production_champion_v1"
                ]
            ),
            build_leaderboard_row(
                benchmark_results[
                    "lightgbm_l1"
                ]
            ),
        ]
    )
)

In [202]:
external_challenger_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = external_challenger_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_rmse
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


external_challenger_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = external_challenger_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                champion_hybrid_mae
            ),
            candidate_value=float(
                value
            ),
        )
    )
)


external_challenger_leaderboard_df = (
    external_challenger_leaderboard_df
    .sort_values("hybrid_rmse")
    .reset_index(drop=True)
)


display(
    external_challenger_leaderboard_df[
        [
            "candidate",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "negative_raw_predictions",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,negative_raw_predictions,training_seconds,prediction_seconds,model_size_mib
0,lightgbm_l1,6.716689,9.303823,0.089406,7.203784,9.577872,1.228615,-0.314334,0,80.805171,0.928181,1.126705
1,focused_histgb_h2_absolute,6.584561,9.306569,0.088868,7.044038,9.581097,1.199460,1.659011,0,32.942681,0.731072,1.096167
2,production_champion_v1,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,0,NaN,0.394257,0.645863


In [203]:
lightgbm_horizon_comparison_df = (
    pd.concat(
        [
            benchmark_results[
                "focused_histgb_h2_absolute"
            ]["horizon_group_metrics"],
            benchmark_results[
                "production_champion_v1"
            ]["horizon_group_metrics"],
            benchmark_results[
                "lightgbm_l1"
            ]["horizon_group_metrics"],
        ],
        ignore_index=True,
    )
)


display(
    lightgbm_horizon_comparison_df.loc[
        lightgbm_horizon_comparison_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
2,focused_histgb_h2_absolute,13-24h,13,24,12163,6.833065,9.411421,0.143644
12,lightgbm_l1,13-24h,13,24,12163,7.214010,9.641681,0.101229
7,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
3,focused_histgb_h2_absolute,25-48h,25,48,23709,7.044120,9.569383,-0.028490
13,lightgbm_l1,25-48h,25,48,23709,7.259254,9.595254,-0.034059
8,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
14,lightgbm_l1,49-72h,49,72,23065,7.141371,9.526087,0.019673
4,focused_histgb_h2_absolute,49-72h,49,72,23065,7.155207,9.681291,-0.012531
9,production_champion_v1,49-72h,49,72,23065,7.220054,9.769384,-0.031042


In [204]:
lightgbm_severe_comparison_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=(
                "production_champion_v1"
            ),
            severe_result=(
                benchmark_results[
                    "production_champion_v1"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                "focused_histgb_h2_absolute"
            ),
            severe_result=(
                benchmark_results[
                    "focused_histgb_h2_absolute"
                ]["severe_pm25"]
            ),
        ),
        flatten_severe_result(
            candidate_name="lightgbm_l1",
            severe_result=(
                benchmark_results[
                    "lightgbm_l1"
                ]["severe_pm25"]
            ),
        ),
    ]
)


display(
    lightgbm_severe_comparison_df
)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,production_champion_v1,55.5,181,49.371822,50.153180,-52.451700
1,focused_histgb_h2_absolute,55.5,181,44.340729,46.258939,-44.473240
2,lightgbm_l1,55.5,181,48.770710,49.995185,-52.115457


In [205]:
lightgbm_l1_overall = (
    lightgbm_l1_result[
        "overall_metrics"
    ]
)


lightgbm_l1_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_hybrid_rmse
        ),
        candidate_value=(
            lightgbm_l1_overall[
                "rmse"
            ]
        ),
    )
)


lightgbm_l1_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            champion_hybrid_mae
        ),
        candidate_value=(
            lightgbm_l1_overall[
                "mae"
            ]
        ),
    )
)


lightgbm_l1_summary = {
    "candidate": "lightgbm_l1",
    "best_iteration": int(
        lightgbm_l1_model.best_iteration_
    ),
    "hybrid_mae": float(
        lightgbm_l1_overall["mae"]
    ),
    "hybrid_rmse": float(
        lightgbm_l1_overall["rmse"]
    ),
    "hybrid_r2": float(
        lightgbm_l1_overall["r2"]
    ),
    "rmse_improvement_vs_production_pct": (
        lightgbm_l1_rmse_improvement
    ),
    "mae_improvement_vs_production_pct": (
        lightgbm_l1_mae_improvement
    ),
    "training_seconds": float(
        lightgbm_l1_training_seconds
    ),
    "model_size_mib": float(
        lightgbm_l1_result[
            "model_size_mib"
        ]
    ),
}


display(
    pd.Series(
        lightgbm_l1_summary,
        name="value",
    ).to_frame()
)

,value
candidate,lightgbm_l1
best_iteration,397
hybrid_mae,6.716689
hybrid_rmse,9.303823
hybrid_r2,0.089406
rmse_improvement_vs_production_pct,1.228615
mae_improvement_vs_production_pct,-0.314334
training_seconds,80.805171
model_size_mib,1.126705


### **12J.3.** LightGBM Continuation Decision

LightGBM L1 produced the lowest overall validation RMSE observed so far, but its
advantage over HistGB H2 is extremely small.

L1 also produced worse MAE than both HistGB H2 and the production champion.

Its performance profile is horizon-dependent:

- HistGB H2 remains stronger at 13–24 and 25–48 hours
- LightGBM L1 is strongest at 49–72 hours
- HistGB H2 remains substantially stronger on severe-PM2.5 observations

This mixed result justifies one final LightGBM experiment.

L2 changes the regression objective from squared-error regression to
absolute-error regression while retaining the remaining L1 structure.

The purpose is to determine whether the MAE weakness of L1 can be improved
without losing its RMSE and long-horizon advantages.

No additional LightGBM configurations will be evaluated after L2.

In [206]:
LIGHTGBM_L2_PARAMETERS = {
    **LIGHTGBM_L1_PARAMETERS,
    "objective": "regression_l1",
}

In [207]:
lightgbm_l2_changes = {
    "objective": (
        LIGHTGBM_L1_PARAMETERS["objective"],
        LIGHTGBM_L2_PARAMETERS["objective"],
    )
}

display(
    pd.Series(
        lightgbm_l2_changes,
        name="L1 → L2",
    ).to_frame()
)

,L1 → L2
objective,"(regression, regression_l1)"


In [208]:
lightgbm_l2_model = LGBMRegressor(
    **LIGHTGBM_L2_PARAMETERS
)


lightgbm_l2_training_start = perf_counter()


lightgbm_l2_model.fit(
    X_train_focused,
    y_train_focused,
    eval_set=[
        (
            X_validation_focused,
            y_validation_focused,
        )
    ],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=75,
            verbose=False,
        )
    ],
)


lightgbm_l2_training_seconds = (
    perf_counter()
    - lightgbm_l2_training_start
)


print(
    "LightGBM L2 training time:",
    f"{lightgbm_l2_training_seconds:.3f}",
    "seconds",
)

print(
    "Best iteration:",
    lightgbm_l2_model.best_iteration_,
)

/home/riyan/Riyan/projects/pearls-aqi-predictor/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


LightGBM L2 training time: 70.668 seconds
Best iteration: 212


In [209]:
(
    lightgbm_l2_raw_predictions,
    lightgbm_l2_hybrid_predictions,
    lightgbm_l2_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=validation_df,
    model=lightgbm_l2_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [210]:
lightgbm_l2_model_size_bytes = (
    measure_serialized_model_size_bytes(
        lightgbm_l2_model
    )
)

In [211]:
lightgbm_l2_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "lightgbm_l2_absolute"
        ),
        model_family="LGBMRegressor",
        training_scope="13-72h",
        dataframe=validation_df,
        raw_predictions=(
            lightgbm_l2_raw_predictions
        ),
        operational_predictions=(
            lightgbm_l2_hybrid_predictions
        ),
        training_seconds=(
            lightgbm_l2_training_seconds
        ),
        prediction_seconds=(
            lightgbm_l2_prediction_seconds
        ),
        model_size_bytes=(
            lightgbm_l2_model_size_bytes
        ),
        parameters=(
            LIGHTGBM_L2_PARAMETERS
        ),
    )
)


benchmark_results[
    "lightgbm_l2_absolute"
] = lightgbm_l2_result

In [212]:
FINAL_VALIDATION_CANDIDATES = [
    "production_champion_v1",
    "focused_histgb_h2_absolute",
    "lightgbm_l1",
    "lightgbm_l2_absolute",
]


external_final_leaderboard_df = pd.DataFrame(
    [
        build_leaderboard_row(
            benchmark_results[key]
        )
        for key in FINAL_VALIDATION_CANDIDATES
    ]
)

In [213]:
external_final_leaderboard_df[
    "rmse_improvement_vs_champion_pct"
] = external_final_leaderboard_df[
    "hybrid_rmse"
].map(
    lambda value: calculate_improvement_percent(
        champion_value=champion_hybrid_rmse,
        candidate_value=float(value),
    )
)


external_final_leaderboard_df[
    "mae_improvement_vs_champion_pct"
] = external_final_leaderboard_df[
    "hybrid_mae"
].map(
    lambda value: calculate_improvement_percent(
        champion_value=champion_hybrid_mae,
        candidate_value=float(value),
    )
)


external_final_leaderboard_df = (
    external_final_leaderboard_df
    .sort_values("hybrid_rmse")
    .reset_index(drop=True)
)

In [214]:
display(
    external_final_leaderboard_df[
        [
            "candidate",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "learned_13_72_mae",
            "learned_13_72_rmse",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ]
    ]
)

,candidate,hybrid_mae,hybrid_rmse,hybrid_r2,learned_13_72_mae,learned_13_72_rmse,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,training_seconds,prediction_seconds,model_size_mib
0,lightgbm_l1,6.716689,9.303823,0.089406,7.203784,9.577872,1.228615,-0.314334,80.805171,0.928181,1.126705
1,focused_histgb_h2_absolute,6.584561,9.306569,0.088868,7.044038,9.581097,1.199460,1.659011,32.942681,0.731072,1.096167
2,lightgbm_l2_absolute,6.652432,9.327069,0.084850,7.126096,9.605168,0.981822,0.645347,70.667537,0.625456,0.629396
3,production_champion_v1,6.695642,9.419553,0.066611,7.178338,9.713671,0.000000,0.000000,NaN,0.394257,0.645863


In [215]:
external_final_horizon_df = pd.concat(
    [
        benchmark_results[key][
            "horizon_group_metrics"
        ]
        for key in FINAL_VALIDATION_CANDIDATES
    ],
    ignore_index=True,
)


display(
    external_final_horizon_df.loc[
        external_final_horizon_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
7,focused_histgb_h2_absolute,13-24h,13,24,12163,6.833065,9.411421,0.143644
17,lightgbm_l2_absolute,13-24h,13,24,12163,7.007114,9.594094,0.110078
12,lightgbm_l1,13-24h,13,24,12163,7.214010,9.641681,0.101229
2,production_champion_v1,13-24h,13,24,12163,7.117727,9.692964,0.091642
8,focused_histgb_h2_absolute,25-48h,25,48,23709,7.044120,9.569383,-0.028490
18,lightgbm_l2_absolute,25-48h,25,48,23709,7.116555,9.590921,-0.033125
13,lightgbm_l1,25-48h,25,48,23709,7.259254,9.595254,-0.034059
3,production_champion_v1,25-48h,25,48,23709,7.168849,9.669828,-0.050194
14,lightgbm_l1,49-72h,49,72,23065,7.141371,9.526087,0.019673
19,lightgbm_l2_absolute,49-72h,49,72,23065,7.198646,9.625618,-0.000920


In [216]:
external_final_severe_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=key,
            severe_result=(
                benchmark_results[
                    key
                ]["severe_pm25"]
            ),
        )
        for key in FINAL_VALIDATION_CANDIDATES
    ]
)


display(
    external_final_severe_df
    .sort_values("rmse")
    .reset_index(drop=True)
)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,focused_histgb_h2_absolute,55.5,181,44.340729,46.258939,-44.473240
1,lightgbm_l2_absolute,55.5,181,47.370977,49.030728,-50.085923
2,lightgbm_l1,55.5,181,48.770710,49.995185,-52.115457
3,production_champion_v1,55.5,181,49.371822,50.153180,-52.451700


In [217]:
lightgbm_family_summary = {
    "l1_best_iteration": int(
        lightgbm_l1_model.best_iteration_
    ),
    "l2_best_iteration": int(
        lightgbm_l2_model.best_iteration_
    ),
    "l1_hybrid_mae": float(
        lightgbm_l1_result[
            "overall_metrics"
        ]["mae"]
    ),
    "l1_hybrid_rmse": float(
        lightgbm_l1_result[
            "overall_metrics"
        ]["rmse"]
    ),
    "l2_hybrid_mae": float(
        lightgbm_l2_result[
            "overall_metrics"
        ]["mae"]
    ),
    "l2_hybrid_rmse": float(
        lightgbm_l2_result[
            "overall_metrics"
        ]["rmse"]
    ),
    "family_status": "CLOSED",
}


display(
    pd.Series(
        lightgbm_family_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12J model-family exploration complete."
)

print(
    "No additional model families or "
    "LightGBM variants will be trained."
)

,value
l1_best_iteration,397
l2_best_iteration,212
l1_hybrid_mae,6.716689
l1_hybrid_rmse,9.303823
l2_hybrid_mae,6.652432
l2_hybrid_rmse,9.327069
family_status,CLOSED


Phase 12J model-family exploration complete.
No additional model families or LightGBM variants will be trained.


In [219]:
ROBUSTNESS_FINALISTS = {
    "production_champion_v1": "Production XGBoost",
    "focused_histgb_h2_absolute": "HistGB H2",
    "lightgbm_l1": "LightGBM L1",
}


model_family_exploration_summary = {
    "status": "CLOSED",
    "reference_champion": (
        "production_champion_v1"
    ),
    "challenger_finalists": [
        "focused_histgb_h2_absolute",
        "lightgbm_l1",
    ],
    "rejected_external_variant": (
        "lightgbm_l2_absolute"
    ),
    "additional_model_training": False,
}


display(
    pd.Series(
        model_family_exploration_summary,
        name="value",
    ).to_frame()
)

,value
status,CLOSED
reference_champion,production_champion_v1
challenger_finalists,"[focused_histgb_h2_absolute, lightgbm_l1]"
rejected_external_variant,lightgbm_l2_absolute
additional_model_training,False


### 12J Result

External model-family exploration is complete.

LightGBM produced a competitive challenger, but no configuration established a
decisive advantage over the retained Histogram Gradient Boosting candidate.

`lightgbm_l1` achieved the lowest overall validation RMSE observed in the
benchmark, improving upon production model version 1 by approximately 1.23%.

However, its advantage over `focused_histgb_h2_absolute` was extremely small,
while its MAE was worse than both HistGB H2 and the production champion.

The two leading challengers exhibit different strengths:

- **LightGBM L1**
  - lowest overall hybrid RMSE
  - strongest RMSE at the longest 49–72 hour horizon group

- **HistGB H2**
  - lowest overall hybrid MAE
  - nearly identical overall RMSE to LightGBM L1
  - stronger performance at 13–24 and 25–48 hours
  - substantially stronger severe-PM2.5 performance
  - faster training with no additional external model dependency beyond
    scikit-learn

The absolute-error LightGBM L2 variant improved MAE relative to L1 but was
worse than HistGB H2 on both primary metrics and therefore does not advance.

No additional LightGBM configurations, model families, or deep-learning models
will be trained in this benchmark.

The validation finalists are:

1. production XGBoost — reference champion
2. HistGB H2 — challenger finalist
3. LightGBM L1 — challenger finalist

The benchmark will now move from model exploration to robustness analysis.

The historical test split remains excluded from model selection.

## **12K.** Challenger Robustness Analysis

Model-family exploration is now closed.

Two challengers remain:

- `focused_histgb_h2_absolute`
- `lightgbm_l1`

The production XGBoost strategy remains the reference champion.

The finalists are extremely close on overall RMSE, so aggregate validation
metrics alone are not sufficient for final selection.

This phase evaluates whether each challenger improves consistently across:

- individual forecast horizons
- different chronological portions of the validation period
- the distribution of prediction errors

Previously calculated severe-PM2.5 and operational characteristics are also
considered in the final interpretation.

No models are trained in this phase.

The historical test split remains excluded.

In [220]:
robustness_frames = {
    key: benchmark_results[key][
        "evaluation_frame"
    ].copy()
    for key in ROBUSTNESS_FINALISTS
}

### **12K.1.** Per-Horizon Consistency

The finalists are compared with production independently across all 72 forecast
horizons.

Because hours 1–12 use identical persistence predictions, the meaningful
comparison focuses on horizons 13–72.

The analysis records how often each challenger improves upon production and the
largest horizon-specific regression.

In [221]:
production_per_horizon = (
    benchmark_results[
        "production_champion_v1"
    ]["per_horizon_metrics"]
    .set_index("forecast_horizon_hours")
)


per_horizon_robustness_records = []


for candidate_key in (
    "focused_histgb_h2_absolute",
    "lightgbm_l1",
):
    candidate_per_horizon = (
        benchmark_results[
            candidate_key
        ]["per_horizon_metrics"]
        .set_index(
            "forecast_horizon_hours"
        )
    )

    learned_horizons = range(
        LEARNED_HORIZON_MIN,
        LEARNED_HORIZON_MAX + 1,
    )

    rmse_improvements = []
    mae_improvements = []

    for horizon in learned_horizons:
        rmse_improvements.append(
            calculate_improvement_percent(
                champion_value=float(
                    production_per_horizon.loc[
                        horizon,
                        "rmse",
                    ]
                ),
                candidate_value=float(
                    candidate_per_horizon.loc[
                        horizon,
                        "rmse",
                    ]
                ),
            )
        )

        mae_improvements.append(
            calculate_improvement_percent(
                champion_value=float(
                    production_per_horizon.loc[
                        horizon,
                        "mae",
                    ]
                ),
                candidate_value=float(
                    candidate_per_horizon.loc[
                        horizon,
                        "mae",
                    ]
                ),
            )
        )

    per_horizon_robustness_records.append(
        {
            "candidate": (
                ROBUSTNESS_FINALISTS[
                    candidate_key
                ]
            ),
            "rmse_horizons_improved": int(
                np.sum(
                    np.array(
                        rmse_improvements
                    )
                    > 0
                )
            ),
            "mae_horizons_improved": int(
                np.sum(
                    np.array(
                        mae_improvements
                    )
                    > 0
                )
            ),
            "total_learned_horizons": 60,
            "median_rmse_improvement_pct": float(
                np.median(
                    rmse_improvements
                )
            ),
            "median_mae_improvement_pct": float(
                np.median(
                    mae_improvements
                )
            ),
            "worst_rmse_regression_pct": float(
                np.min(
                    rmse_improvements
                )
            ),
            "worst_mae_regression_pct": float(
                np.min(
                    mae_improvements
                )
            ),
        }
    )


per_horizon_robustness_df = pd.DataFrame(
    per_horizon_robustness_records
)


display(
    per_horizon_robustness_df
)

,candidate,rmse_horizons_improved,mae_horizons_improved,total_learned_horizons,median_rmse_improvement_pct,median_mae_improvement_pct,worst_rmse_regression_pct,worst_mae_regression_pct
0,HistGB H2,49,56,60,0.974154,1.398626,-0.806101,-0.144266
1,LightGBM L1,59,23,60,1.143594,-0.751866,-0.558746,-2.025263


### **12K.2.** Chronological Validation Stability

Overall validation metrics can hide performance that is concentrated in one
part of the validation period.

The validation forecast origins are therefore divided chronologically into
three approximately equal blocks.

Each finalist is compared within each block using the same complete hybrid
predictions.

This is not additional model selection data; it is a decomposition of the
existing frozen validation split.

In [222]:
validation_reference_times = (
    validation_checked_df[
        "reference_time"
    ]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)


reference_blocks = np.array_split(
    validation_reference_times,
    3,
)


reference_to_block = {
    reference_time: (
        f"block_{block_index + 1}"
    )
    for block_index, block
    in enumerate(reference_blocks)
    for reference_time in block
}

/home/riyan/Riyan/projects/pearls-aqi-predictor/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'Series.swapaxes' is deprecated and will be removed in a future version. Please use 'Series.transpose' instead.
  return bound(*args, **kwds)


In [223]:
chronological_stability_records = []


for candidate_key, display_name in (
    ROBUSTNESS_FINALISTS.items()
):
    evaluation_df = (
        robustness_frames[
            candidate_key
        ].copy()
    )

    evaluation_df[
        "validation_block"
    ] = evaluation_df[
        "reference_time"
    ].map(
        reference_to_block
    )

    for block_name, block_df in (
        evaluation_df.groupby(
            "validation_block",
            sort=True,
        )
    ):
        metrics = calculate_regression_metrics(
            y_true=block_df[
                TARGET_COLUMN
            ],
            y_pred=block_df[
                "prediction"
            ],
        )

        chronological_stability_records.append(
            {
                "candidate": display_name,
                "validation_block": (
                    block_name
                ),
                "rows": len(block_df),
                **metrics,
            }
        )


chronological_stability_df = pd.DataFrame(
    chronological_stability_records
)


display(
    chronological_stability_df
)

,candidate,validation_block,rows,mae,rmse,r2
0,Production XGBoost,block_1,22188,10.358576,13.141576,-0.306465
1,Production XGBoost,block_2,24042,5.043483,7.694003,0.009630
2,Production XGBoost,block_3,25026,5.035291,6.530454,-2.284066
3,HistGB H2,block_1,22188,10.290935,12.884099,-0.255772
4,HistGB H2,block_2,24042,5.267991,7.712786,0.004789
5,HistGB H2,block_3,25026,4.563300,6.502753,-2.256265
6,LightGBM L1,block_1,22188,10.145031,12.628327,-0.206409
7,LightGBM L1,block_2,24042,5.382684,7.921382,-0.049771
8,LightGBM L1,block_3,25026,4.958681,6.692732,-2.449308


### **12K.3.** Error Distribution

The finalists are also compared through the distribution of validation errors.

This helps distinguish a model that improves typical predictions from one whose
headline metric is driven mainly by a smaller number of large errors.

For each finalist the analysis records:

- mean signed error
- median absolute error
- 90th percentile absolute error
- 95th percentile absolute error
- 99th percentile absolute error

In [224]:
error_distribution_records = []


for candidate_key, display_name in (
    ROBUSTNESS_FINALISTS.items()
):
    evaluation_df = (
        robustness_frames[
            candidate_key
        ]
    )

    signed_error = (
        evaluation_df["prediction"]
        - evaluation_df[
            TARGET_COLUMN
        ]
    )

    absolute_error = signed_error.abs()

    error_distribution_records.append(
        {
            "candidate": display_name,
            "mean_signed_error": float(
                signed_error.mean()
            ),
            "median_absolute_error": float(
                absolute_error.median()
            ),
            "p90_absolute_error": float(
                absolute_error.quantile(
                    0.90
                )
            ),
            "p95_absolute_error": float(
                absolute_error.quantile(
                    0.95
                )
            ),
            "p99_absolute_error": float(
                absolute_error.quantile(
                    0.99
                )
            ),
        }
    )


error_distribution_df = pd.DataFrame(
    error_distribution_records
)


display(
    error_distribution_df
)

,candidate,mean_signed_error,median_absolute_error,p90_absolute_error,p95_absolute_error,p99_absolute_error
0,Production XGBoost,3.215828,4.886128,14.628294,17.762714,27.902839
1,HistGB H2,2.635024,4.587223,15.022523,17.975344,27.911140
2,LightGBM L1,2.927431,4.801871,15.190697,18.487443,28.200741


In [225]:
robustness_scorecard_df = (
    external_final_leaderboard_df.loc[
        external_final_leaderboard_df[
            "candidate"
        ].isin(
            [
                "production_champion_v1",
                "focused_histgb_h2_absolute",
                "lightgbm_l1",
            ]
        ),
        [
            "candidate",
            "hybrid_mae",
            "hybrid_rmse",
            "hybrid_r2",
            "rmse_improvement_vs_champion_pct",
            "mae_improvement_vs_champion_pct",
            "training_seconds",
            "prediction_seconds",
            "model_size_mib",
        ],
    ]
    .copy()
)

In [226]:
severe_lookup = (
    external_final_severe_df
    .set_index("candidate")
)


robustness_scorecard_df[
    "severe_mae"
] = robustness_scorecard_df[
    "candidate"
].map(
    severe_lookup["mae"]
)


robustness_scorecard_df[
    "severe_rmse"
] = robustness_scorecard_df[
    "candidate"
].map(
    severe_lookup["rmse"]
)


display(
    robustness_scorecard_df
)

,candidate,hybrid_mae,hybrid_rmse,hybrid_r2,rmse_improvement_vs_champion_pct,mae_improvement_vs_champion_pct,training_seconds,prediction_seconds,model_size_mib,severe_mae,severe_rmse
0,lightgbm_l1,6.716689,9.303823,0.089406,1.228615,-0.314334,80.805171,0.928181,1.126705,48.770710,49.995185
1,focused_histgb_h2_absolute,6.584561,9.306569,0.088868,1.199460,1.659011,32.942681,0.731072,1.096167,44.340729,46.258939
3,production_champion_v1,6.695642,9.419553,0.066611,0.000000,0.000000,NaN,0.394257,0.645863,49.371822,50.153180


### 12K Result

The robustness analysis resolves the close aggregate validation result between
HistGB H2 and LightGBM L1.

LightGBM L1 achieved the lowest overall RMSE, but its advantage over HistGB H2
was approximately 0.03%, which is too small to treat as a meaningful standalone
advantage.

The broader validation evidence favors HistGB H2.

Across the 60 learned-model horizons:

- HistGB H2 improved RMSE on 49 horizons
- HistGB H2 improved MAE on 56 horizons
- LightGBM L1 improved RMSE on 59 horizons
- LightGBM L1 improved MAE on only 23 horizons

HistGB H2 also showed:

- lower overall MAE
- lower median absolute error
- lower mean prediction bias
- substantially stronger severe-PM2.5 performance
- improvement across two of the three chronological validation blocks
- faster training and prediction
- no additional external estimator dependency

Its upper-tail error distribution remained comparable with production, with
the 99th-percentile absolute error effectively unchanged.

LightGBM L1 remains a credible validation finalist and demonstrated particularly
strong RMSE consistency and long-horizon performance, but its extremely small
aggregate RMSE advantage does not outweigh the broader robustness evidence.

`focused_histgb_h2_absolute` is therefore the preferred challenger for the
validation-selection phase.

No historical test results have been used in this decision.

## **12L.** Validation Selection Freeze

Model development and validation-based model selection are now complete.

All candidate families, configurations, and robustness diagnostics considered
for challenger selection used the frozen training and validation splits only.

Based on the complete validation evidence, the selected challenger is:

**`focused_histgb_h2_absolute`**

HistGB H2 is selected over LightGBM L1 despite LightGBM's marginally lower
overall RMSE because the difference in RMSE is negligible and HistGB provides
the stronger overall robustness profile.

The selection is supported by:

- lower overall validation MAE than production
- lower overall validation RMSE than production
- improvement across every learned horizon group
- MAE improvement across 56 of 60 learned horizons
- lower median absolute error
- lower prediction bias
- substantially stronger severe-PM2.5 performance
- acceptable chronological stability
- clean numerical prediction behavior
- lower training and inference cost than LightGBM
- no additional external estimator dependency

### Selection freeze rule

After this phase:

- no additional challenger models will be trained
- no challenger hyperparameters will be changed
- no second-place candidate will be substituted based on test performance
- no test result will be used to revise the validation selection

The selected HistGB H2 challenger will receive exactly one historical test
evaluation in Phase 12M.

If it fails to provide sufficient final evidence, the production champion will
remain unchanged.

In [227]:
SELECTED_CHALLENGER_KEY = (
    "focused_histgb_h2_absolute"
)

SELECTED_CHALLENGER_MODEL = (
    focused_histgb_h2_model
)

selected_challenger_result = (
    benchmark_results[
        SELECTED_CHALLENGER_KEY
    ]
)

In [228]:
validation_selection = {
    "status": "FROZEN",
    "selected_challenger": (
        SELECTED_CHALLENGER_KEY
    ),
    "model_family": (
        selected_challenger_result[
            "model_family"
        ]
    ),
    "training_scope": (
        selected_challenger_result[
            "training_scope"
        ]
    ),
    "validation_mae": (
        selected_challenger_result[
            "overall_metrics"
        ]["mae"]
    ),
    "validation_rmse": (
        selected_challenger_result[
            "overall_metrics"
        ]["rmse"]
    ),
    "validation_r2": (
        selected_challenger_result[
            "overall_metrics"
        ]["r2"]
    ),
    "rmse_improvement_vs_production_pct": (
        best_histgb_rmse_improvement
    ),
    "mae_improvement_vs_production_pct": (
        best_histgb_mae_improvement
    ),
    "runner_up": "lightgbm_l1",
    "runner_up_status": (
        "FINALIST_NOT_SELECTED"
    ),
    "selection_basis": (
        "validation_only"
    ),
    "test_evaluations_allowed": 1,
}


display(
    pd.Series(
        validation_selection,
        name="value",
    ).to_frame()
)

,value
status,FROZEN
selected_challenger,focused_histgb_h2_absolute
model_family,HistGradientBoostingRegressor
training_scope,13-72h
validation_mae,6.584561
validation_rmse,9.306569
validation_r2,0.088868
rmse_improvement_vs_production_pct,1.19946
mae_improvement_vs_production_pct,1.659011
runner_up,lightgbm_l1


In [229]:
selection_comparison_df = pd.DataFrame(
    [
        {
            "candidate": "HistGB H2",
            "validation_mae": (
                best_histgb_overall["mae"]
            ),
            "validation_rmse": (
                best_histgb_overall["rmse"]
            ),
            "rmse_horizons_improved": 49,
            "mae_horizons_improved": 56,
            "severe_mae": 44.340729,
            "severe_rmse": 46.258939,
            "decision": "SELECTED",
        },
        {
            "candidate": "LightGBM L1",
            "validation_mae": (
                lightgbm_l1_result[
                    "overall_metrics"
                ]["mae"]
            ),
            "validation_rmse": (
                lightgbm_l1_result[
                    "overall_metrics"
                ]["rmse"]
            ),
            "rmse_horizons_improved": 59,
            "mae_horizons_improved": 23,
            "severe_mae": 48.770710,
            "severe_rmse": 49.995185,
            "decision": (
                "FINALIST_NOT_SELECTED"
            ),
        },
    ]
)


display(selection_comparison_df)

,candidate,validation_mae,validation_rmse,rmse_horizons_improved,mae_horizons_improved,severe_mae,severe_rmse,decision
0,HistGB H2,6.584561,9.306569,49,56,44.340729,46.258939,SELECTED
1,LightGBM L1,6.716689,9.303823,59,23,48.770710,49.995185,FINALIST_NOT_SELECTED


### 12L Result

Validation-based challenger selection is frozen.

`focused_histgb_h2_absolute` is the single challenger selected for final
historical test evaluation.

LightGBM L1 remains a credible finalist, particularly for RMSE consistency, but
its approximately 0.03% aggregate RMSE advantage over HistGB H2 is not large
enough to outweigh HistGB's stronger MAE, severe-PM2.5 performance, error
distribution, chronological behavior, runtime, and deployment simplicity.

No additional model development will occur after this point.

The next phase will evaluate exactly one challenger against the production
champion on the historical test split.

The outcome of that evaluation may reject the challenger, but it will not be
used to select or tune another model.

## **12M.** One-Time Final Historical Test

Validation-based model selection is frozen.

The single selected challenger is:

**`focused_histgb_h2_absolute`**

This phase performs exactly one final historical test evaluation of that
challenger.

The comparison includes:

- production champion version 1
- selected HistGB H2 challenger
- current-value persistence
- previous-day persistence

No model is trained, tuned, or modified in this phase.

The historical test split has already been viewed previously during development
of production model version 1, so it is not globally untouched. However, it has
remained excluded from all challenger-family development, tuning, robustness
analysis, and validation-based selection in this benchmark.

The purpose of this phase is therefore to provide one non-adaptive final check
of the selected challenger.

If HistGB H2 does not provide sufficient final evidence, production version 1
will remain unchanged.

The test result will not be used to select another challenger.

In [230]:
X_test = test_df[
    MODEL_FEATURE_COLUMNS
].copy()

y_test = test_df[
    TARGET_COLUMN
].astype(float).copy()


print(
    "Historical test rows:",
    len(test_df),
)

print(
    "Test horizon range:",
    test_df[
        FORECAST_HORIZON_COLUMN
    ].min(),
    "to",
    test_df[
        FORECAST_HORIZON_COLUMN
    ].max(),
)

Historical test rows: 76813
Test horizon range: 1 to 72


In [231]:
(
    production_test_raw_predictions,
    production_test_predictions,
    production_test_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=test_df,
    model=production_model,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [232]:
production_test_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            "production_champion_v1"
        ),
        model_family="XGBRegressor",
        training_scope=(
            "historical production artifact"
        ),
        dataframe=test_df,
        raw_predictions=(
            production_test_raw_predictions
        ),
        operational_predictions=(
            production_test_predictions
        ),
        training_seconds=None,
        prediction_seconds=(
            production_test_prediction_seconds
        ),
        model_size_bytes=None,
        parameters=(
            production_metadata.get(
                "model_parameters",
                {},
            )
        ),
    )
)

In [233]:
(
    selected_test_raw_predictions,
    selected_test_predictions,
    selected_test_prediction_seconds,
) = generate_timed_hybrid_predictions(
    dataframe=test_df,
    model=SELECTED_CHALLENGER_MODEL,
    feature_columns=MODEL_FEATURE_COLUMNS,
    persistence_max_horizon=(
        PERSISTENCE_MAX_HORIZON
    ),
)

In [234]:
selected_test_result = (
    evaluate_candidate_predictions(
        candidate_name=(
            SELECTED_CHALLENGER_KEY
        ),
        model_family=(
            selected_challenger_result[
                "model_family"
            ]
        ),
        training_scope=(
            selected_challenger_result[
                "training_scope"
            ]
        ),
        dataframe=test_df,
        raw_predictions=(
            selected_test_raw_predictions
        ),
        operational_predictions=(
            selected_test_predictions
        ),
        training_seconds=None,
        prediction_seconds=(
            selected_test_prediction_seconds
        ),
        model_size_bytes=None,
        parameters=(
            selected_challenger_result[
                "parameters"
            ]
        ),
    )
)

### **12M.1.** Test Baselines

The final comparison retains the two persistence baselines used throughout the
original production model evaluation.

They provide context for whether the selected challenger improves not only upon
the current production model but also upon simple forecasting strategies.

In [235]:
current_persistence_test_predictions = (
    test_df[
        "pm25_current"
    ]
    .astype(float)
    .to_numpy()
)


previous_day_test_predictions = (
    test_df[
        "pm25_lag_24h"
    ]
    .astype(float)
    .to_numpy()
)


current_persistence_test_metrics = (
    calculate_regression_metrics(
        y_true=y_test,
        y_pred=(
            current_persistence_test_predictions
        ),
    )
)


previous_day_test_metrics = (
    calculate_regression_metrics(
        y_true=y_test,
        y_pred=(
            previous_day_test_predictions
        ),
    )
)

In [236]:
final_test_leaderboard_df = pd.DataFrame(
    [
        {
            "candidate": (
                "production_champion_v1"
            ),
            **production_test_result[
                "overall_metrics"
            ],
        },
        {
            "candidate": (
                SELECTED_CHALLENGER_KEY
            ),
            **selected_test_result[
                "overall_metrics"
            ],
        },
        {
            "candidate": (
                "current_persistence"
            ),
            **current_persistence_test_metrics,
        },
        {
            "candidate": (
                "previous_day_persistence"
            ),
            **previous_day_test_metrics,
        },
    ]
)

In [237]:
final_test_leaderboard_df = (
    final_test_leaderboard_df
    .sort_values("rmse")
    .reset_index(drop=True)
)

In [238]:
production_test_rmse = (
    production_test_result[
        "overall_metrics"
    ]["rmse"]
)

production_test_mae = (
    production_test_result[
        "overall_metrics"
    ]["mae"]
)


final_test_leaderboard_df[
    "rmse_improvement_vs_production_pct"
] = final_test_leaderboard_df[
    "rmse"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_test_rmse
            ),
            candidate_value=float(value),
        )
    )
)


final_test_leaderboard_df[
    "mae_improvement_vs_production_pct"
] = final_test_leaderboard_df[
    "mae"
].map(
    lambda value: (
        calculate_improvement_percent(
            champion_value=(
                production_test_mae
            ),
            candidate_value=float(value),
        )
    )
)

In [239]:
display(
    final_test_leaderboard_df
)

,candidate,mae,rmse,r2,rmse_improvement_vs_production_pct,mae_improvement_vs_production_pct
0,production_champion_v1,3.715507,4.975563,-0.003986,0.000000,0.000000
1,focused_histgb_h2_absolute,3.628446,4.987334,-0.008742,-0.236568,2.343173
2,current_persistence,3.835369,5.283099,-0.131933,-6.180933,-3.225996
3,previous_day_persistence,4.543849,6.117524,-0.517730,-22.951387,-22.294198


In [240]:
final_test_learned_df = pd.DataFrame(
    [
        {
            "candidate": (
                "production_champion_v1"
            ),
            **production_test_result[
                "learned_horizon_metrics"
            ],
        },
        {
            "candidate": (
                SELECTED_CHALLENGER_KEY
            ),
            **selected_test_result[
                "learned_horizon_metrics"
            ],
        },
    ]
)


display(
    final_test_learned_df
)

,candidate,mae,rmse,r2
0,production_champion_v1,3.909758,5.167296,-0.073143
1,focused_histgb_h2_absolute,3.804668,5.180975,-0.078832


In [241]:
final_test_horizon_df = pd.concat(
    [
        production_test_result[
            "horizon_group_metrics"
        ],
        selected_test_result[
            "horizon_group_metrics"
        ],
    ],
    ignore_index=True,
)


display(
    final_test_horizon_df.loc[
        final_test_horizon_df[
            "horizon_group"
        ].isin(
            [
                "13-24h",
                "25-48h",
                "49-72h",
            ]
        )
    ]
    .sort_values(
        [
            "horizon_min",
            "rmse",
        ]
    )
)

,model,horizon_group,horizon_min,horizon_max,rows,mae,rmse,r2
2,production_champion_v1,13-24h,13,24,12992,3.846434,5.019546,-0.015653
7,focused_histgb_h2_absolute,13-24h,13,24,12992,3.757901,5.094000,-0.046007
3,production_champion_v1,25-48h,25,48,25595,3.884136,5.143997,-0.063744
8,focused_histgb_h2_absolute,25-48h,25,48,25595,3.820000,5.241614,-0.104500
9,focused_histgb_h2_absolute,49-72h,49,72,25048,3.813259,5.163352,-0.069797
4,production_champion_v1,49-72h,49,72,25048,3.968784,5.265653,-0.112609


In [242]:
final_test_severe_df = pd.DataFrame(
    [
        flatten_severe_result(
            candidate_name=(
                "production_champion_v1"
            ),
            severe_result=(
                production_test_result[
                    "severe_pm25"
                ]
            ),
        ),
        flatten_severe_result(
            candidate_name=(
                SELECTED_CHALLENGER_KEY
            ),
            severe_result=(
                selected_test_result[
                    "severe_pm25"
                ]
            ),
        ),
    ]
)


display(
    final_test_severe_df
)

,candidate,threshold_ug_m3,sample_count,mae,rmse,r2
0,production_champion_v1,55.5,0,None,None,None
1,focused_histgb_h2_absolute,55.5,0,None,None,None


In [243]:
final_test_validity_df = pd.DataFrame(
    [
        {
            "candidate": (
                "production_champion_v1"
            ),
            **production_test_result[
                "prediction_validity"
            ],
        },
        {
            "candidate": (
                SELECTED_CHALLENGER_KEY
            ),
            **selected_test_result[
                "prediction_validity"
            ],
        },
    ]
)


display(
    final_test_validity_df
)

,candidate,prediction_count,raw_nan_count,raw_inf_count,negative_raw_count,clipped_prediction_count,raw_min,raw_max,operational_min,operational_max
0,production_champion_v1,76813,0,0,0,0,3.1,37.5,3.1,37.5
1,focused_histgb_h2_absolute,76813,0,0,0,0,3.1,37.5,3.1,37.5


In [244]:
selected_test_overall = (
    selected_test_result[
        "overall_metrics"
    ]
)


selected_test_rmse_improvement = (
    calculate_improvement_percent(
        champion_value=(
            production_test_result[
                "overall_metrics"
            ]["rmse"]
        ),
        candidate_value=(
            selected_test_overall[
                "rmse"
            ]
        ),
    )
)


selected_test_mae_improvement = (
    calculate_improvement_percent(
        champion_value=(
            production_test_result[
                "overall_metrics"
            ]["mae"]
        ),
        candidate_value=(
            selected_test_overall[
                "mae"
            ]
        ),
    )
)

In [245]:
validation_test_generalization_df = (
    pd.DataFrame(
        [
            {
                "metric": "RMSE",
                "validation_improvement_pct": (
                    best_histgb_rmse_improvement
                ),
                "test_improvement_pct": (
                    selected_test_rmse_improvement
                ),
            },
            {
                "metric": "MAE",
                "validation_improvement_pct": (
                    best_histgb_mae_improvement
                ),
                "test_improvement_pct": (
                    selected_test_mae_improvement
                ),
            },
        ]
    )
)


display(
    validation_test_generalization_df
)

,metric,validation_improvement_pct,test_improvement_pct
0,RMSE,1.199460,-0.236568
1,MAE,1.659011,2.343173


In [246]:
final_test_summary = {
    "status": "COMPLETED",
    "selected_challenger": (
        SELECTED_CHALLENGER_KEY
    ),
    "test_rows": len(test_df),
    "production_mae": float(
        production_test_result[
            "overall_metrics"
        ]["mae"]
    ),
    "production_rmse": float(
        production_test_result[
            "overall_metrics"
        ]["rmse"]
    ),
    "challenger_mae": float(
        selected_test_overall["mae"]
    ),
    "challenger_rmse": float(
        selected_test_overall["rmse"]
    ),
    "test_rmse_improvement_pct": float(
        selected_test_rmse_improvement
    ),
    "test_mae_improvement_pct": float(
        selected_test_mae_improvement
    ),
    "selection_remains_frozen": True,
    "additional_test_candidates_allowed": False,
}


display(
    pd.Series(
        final_test_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12M COMPLETED — the selected "
    "challenger has received its one-time "
    "historical test evaluation."
)

,value
status,COMPLETED
selected_challenger,focused_histgb_h2_absolute
test_rows,76813
production_mae,3.715507
production_rmse,4.975563
challenger_mae,3.628446
challenger_rmse,4.987334
test_rmse_improvement_pct,-0.236568
test_mae_improvement_pct,2.343173
selection_remains_frozen,True


Phase 12M COMPLETED — the selected challenger has received its one-time historical test evaluation.


### 12M Result

The selected HistGB H2 challenger has received its single historical test
evaluation.

The comparison includes:

- production champion version 1
- selected HistGB H2 challenger
- current-value persistence
- previous-day persistence

The analysis covers:

- overall MAE, RMSE, and R²
- the learned 13–72 hour region
- 13–24, 25–48, and 49–72 hour horizon groups
- severe-PM2.5 observations
- prediction validity
- validation-to-test generalization

No additional challenger has been evaluated on the historical test split.

No model parameters have been changed after validation selection was frozen.

The test result will now be interpreted in Phase 12N to determine whether the
evidence is strong enough to justify replacing production model version 1.

## **12N.** Final Model Decision

The selected HistGB H2 challenger has completed its one-time historical test
evaluation.

The final result is mixed.

Compared with production model version 1, HistGB H2 achieved:

- lower historical-test MAE
- slightly higher historical-test RMSE
- lower MAE across all learned-model horizon groups
- lower RMSE at the longest 49–72 hour horizon group
- clean numerical prediction behavior

However, the validation RMSE advantage did not generalize to the historical
test split.

The challenger improved validation RMSE by approximately 1.20%, but historical
test RMSE was approximately 0.24% worse than production.

The historical test period also contained no observations at or above the
55.5 µg/m³ severe-PM2.5 threshold, so the strong severe-event advantage observed
during validation could not be independently verified on this test period.

For a mature production system, the evidence is therefore insufficient to
justify replacing the existing champion.

### Final decision

**Production model version 1 remains the champion.**

`focused_histgb_h2_absolute` is retained as a credible benchmark challenger but
will not be promoted to production.

No second-place challenger will be evaluated on the historical test split, and
no further model tuning will be performed as part of this benchmark.

In [247]:
FINAL_MODEL_DECISION = {
    "decision": "KEEP_PRODUCTION_CHAMPION",
    "production_model": (
        "production_champion_v1"
    ),
    "selected_challenger": (
        SELECTED_CHALLENGER_KEY
    ),
    "challenger_status": (
        "NOT_PROMOTED"
    ),
    "validation_rmse_improvement_pct": (
        best_histgb_rmse_improvement
    ),
    "validation_mae_improvement_pct": (
        best_histgb_mae_improvement
    ),
    "test_rmse_improvement_pct": (
        selected_test_rmse_improvement
    ),
    "test_mae_improvement_pct": (
        selected_test_mae_improvement
    ),
    "severe_test_samples": int(
        selected_test_result[
            "severe_pm25"
        ]["sample_count"]
    ),
    "production_change_required": False,
}


display(
    pd.Series(
        FINAL_MODEL_DECISION,
        name="value",
    ).to_frame()
)

,value
decision,KEEP_PRODUCTION_CHAMPION
production_model,production_champion_v1
selected_challenger,focused_histgb_h2_absolute
challenger_status,NOT_PROMOTED
validation_rmse_improvement_pct,1.19946
validation_mae_improvement_pct,1.659011
test_rmse_improvement_pct,-0.236568
test_mae_improvement_pct,2.343173
severe_test_samples,0
production_change_required,False


In [248]:
final_decision_evidence_df = pd.DataFrame(
    [
        {
            "evidence": "Validation RMSE",
            "production": (
                champion_overall[
                    "rmse"
                ]
            ),
            "challenger": (
                best_histgb_overall[
                    "rmse"
                ]
            ),
            "result": (
                "CHALLENGER_BETTER"
            ),
        },
        {
            "evidence": "Validation MAE",
            "production": (
                champion_overall[
                    "mae"
                ]
            ),
            "challenger": (
                best_histgb_overall[
                    "mae"
                ]
            ),
            "result": (
                "CHALLENGER_BETTER"
            ),
        },
        {
            "evidence": "Historical test RMSE",
            "production": (
                production_test_result[
                    "overall_metrics"
                ]["rmse"]
            ),
            "challenger": (
                selected_test_overall[
                    "rmse"
                ]
            ),
            "result": (
                "PRODUCTION_BETTER"
            ),
        },
        {
            "evidence": "Historical test MAE",
            "production": (
                production_test_result[
                    "overall_metrics"
                ]["mae"]
            ),
            "challenger": (
                selected_test_overall[
                    "mae"
                ]
            ),
            "result": (
                "CHALLENGER_BETTER"
            ),
        },
        {
            "evidence": "Severe PM2.5 test",
            "production": None,
            "challenger": None,
            "result": (
                "NOT_EVALUABLE"
            ),
        },
    ]
)


display(
    final_decision_evidence_df
)

,evidence,production,challenger,result
0,Validation RMSE,9.419553,9.306569,CHALLENGER_BETTER
1,Validation MAE,6.695642,6.584561,CHALLENGER_BETTER
2,Historical test RMSE,4.975563,4.987334,PRODUCTION_BETTER
3,Historical test MAE,3.715507,3.628446,CHALLENGER_BETTER
4,Severe PM2.5 test,NaN,NaN,NOT_EVALUABLE


In [249]:
final_horizon_decision_records = []


for group_name in (
    "13-24h",
    "25-48h",
    "49-72h",
):
    production_row = (
        final_test_horizon_df.loc[
            (
                final_test_horizon_df[
                    "model"
                ]
                == "production_champion_v1"
            )
            & (
                final_test_horizon_df[
                    "horizon_group"
                ]
                == group_name
            )
        ]
        .iloc[0]
    )

    challenger_row = (
        final_test_horizon_df.loc[
            (
                final_test_horizon_df[
                    "model"
                ]
                == SELECTED_CHALLENGER_KEY
            )
            & (
                final_test_horizon_df[
                    "horizon_group"
                ]
                == group_name
            )
        ]
        .iloc[0]
    )

    final_horizon_decision_records.append(
        {
            "horizon_group": group_name,
            "mae_winner": (
                "challenger"
                if challenger_row["mae"]
                < production_row["mae"]
                else "production"
            ),
            "rmse_winner": (
                "challenger"
                if challenger_row["rmse"]
                < production_row["rmse"]
                else "production"
            ),
        }
    )


final_horizon_decision_df = (
    pd.DataFrame(
        final_horizon_decision_records
    )
)


display(
    final_horizon_decision_df
)

,horizon_group,mae_winner,rmse_winner
0,13-24h,challenger,production
1,25-48h,challenger,production
2,49-72h,challenger,challenger


In [250]:
final_model_summary = {
    "status": "DECIDED",
    "production_champion_retained": True,
    "challenger_promoted": False,
    "reason": (
        "Selected challenger improved MAE "
        "but did not preserve its validation "
        "RMSE advantage on historical test."
    ),
    "additional_model_training": False,
    "additional_test_evaluation": False,
    "registry_update_required": False,
    "production_pointer_change_required": False,
}


display(
    pd.Series(
        final_model_summary,
        name="value",
    ).to_frame()
)


print(
    "Phase 12N PASSED — production "
    "model version 1 remains champion."
)

,value
status,DECIDED
production_champion_retained,True
challenger_promoted,False
reason,Selected challenger improved MAE but did not p...
additional_model_training,False
additional_test_evaluation,False
registry_update_required,False
production_pointer_change_required,False


Phase 12N PASSED — production model version 1 remains champion.


### 12N Result

Production model version 1 remains the final champion.

HistGB H2 was a credible challenger and improved several meaningful aspects of
forecasting performance, particularly MAE and long-horizon behavior.

However, its validation RMSE improvement did not survive the one-time
historical test evaluation:

- validation RMSE improvement: approximately +1.20%
- historical-test RMSE improvement: approximately -0.24%

Historical-test MAE improved by approximately 2.34%, demonstrating that the
challenger learned a genuinely different and useful error profile, but the
overall evidence is mixed rather than decisively superior.

The severe-PM2.5 advantage observed during validation could not be verified
because the historical test split contained no observations at or above the
55.5 µg/m³ severe threshold.

Given the small validation advantage, slightly worse final RMSE, mixed
horizon-level evidence, and the stability of the existing deployed system,
there is insufficient evidence to justify a production model replacement.

The current production XGBoost hybrid therefore remains unchanged.

No additional challenger will be tested and model experimentation is closed.